# E44 · E43 + MSE dan ListNet · S3 Native Learnable

**Satu perlakuan: loss training menjadi `MSE + 0,1 × ListNet`.** Arsitektur dan seluruh LR tetap E43. MSE mengarahkan angka prediksi ke nilai guru; ListNet menambah pembelajaran urutan kualitas jawaban untuk soal yang sama. Ini bukan mengurutkan hasil setelah prediksi selesai.

RoBERTa → Scalar Mix empat layer + gamma → BiGRU 64 per arah + co-attention → masked mean → proyeksi 768; gabung proyeksi BGE 256 → concat 1024 → MLP → satu nilai. RoBERTa dan komponen tambahan tetap dilatih; BGE-M3 tetap frozen. Tidak ada parameter model baru.

| Pengaturan | Nilai |
|---|---|
| X / y | `soal`, `jawaban_referensi`, `response_text` / `rater` 0–100 |
| LR encoder / seluruh komponen lain | 5e-6 / 1e-4 |
| Loss | MSE + 0,1 × rerata ListNet kelompok soal yang memenuhi syarat |
| Skala target dan skor untuk kedua loss | 0–1; softmax ListNet tanpa temperature tambahan |
| Batch | Efektif 8, forward microbatch 1; sampler tetap E43 |
| Protokol | S3 saja, satu 5-fold, maksimal 25 epoch, patience 5 |
| Pemilihan | RMSE validasi skala 0–100; refit ceil(median best epoch), satu test |

**Ranking hanya menggunakan kelompok dengan ≥2 jawaban untuk teks soal yang persis sama di dalam batch efektif.** Singleton hanya menerima MSE. Tidak menyusun ulang dataset, mengganti split, atau menambah pasangan dari validation/test. Ties nilai guru menghasilkan distribusi target yang sama, bukan urutan buatan. Riwayat mencatat berapa kelompok/jawaban benar-benar menerima ranking; run dihentikan jika tidak ada kelompok ranking sepanjang suatu epoch.

Microbatch 1 tidak diperlakukan sebagai daftar ranking sendiri. Prediksi beberapa microbatch digabung dengan graph gradien yang utuh. `RANKING_RECOMPUTE=True` menghitung ulang aktivasi saat backward dengan RNG dropout yang sama untuk mengurangi memori; ini tidak menyimpan file model dan tidak mengubah batch efektif. Kebutuhan memori/waktu GPU aktual tetap bergantung runtime.

Saat evaluasi dan prediksi baru, model tetap membaca satu soal + referensi + jawaban dan menghasilkan satu nilai. Tidak membutuhkan nilai guru atau jawaban lain untuk diurutkan. Tidak ada cosine, S1/S2, loop multi-seed, kontrol ulang otomatis, atau worker tersembunyi. Seluruh Python ditulis per section di notebook.

**Dasar AES:** [Yang dkk. (2020)](https://aclanthology.org/2020.findings-emnlp.141/) menggabungkan MSE dan batch-wise ListNet dengan bobot dinamis. E44 mengadaptasinya ke fusion E43, memakai bobot tetap 0,1 dan kelompok dalam soal yang sama. Bobot 0,1 adalah pilihan eksperimen proyek, bukan optimum paper atau replikasi persis. Paper mengevaluasi QWK pada ASAP; RMSE E44 belum tentu membaik.

**Output:** lima Excel dalam satu ZIP. Excel 05 memisahkan MSE, ListNet, loss gabungan, dan cakupan ranking. Cross-entropy ListNet tidak harus mendekati nol meski prediksi tepat karena target berupa distribusi softmax. Excel 04 menambah **Prediksi Validasi OOF** untuk setiap jawaban development; Excel 05 tetap memiliki **MAE dan Toleransi CV**. Pasangan OOF terpisah dari 60 test.

Pilih **GPU T4 → Run all → unggah `fix_dataset_300.xlsx`**. Parameter loss ada di section 5, rumus dan pembentukan batch di section training. Folder/hash/checkpoint E44 terpisah. Belum ada hasil penelitian E44; acuan tetap E43 (mean CV 5,8876). Test historis yang telah dilihat berulang kali tetap bersifat eksploratif.


## 1 · Setup pustaka pada kernel Colab

Jalankan sebelum bagian import. Pustaka numerik/CUDA bawaan dipertahankan; tidak ada lingkungan Python kedua.


In [ ]:
# Jalankan pada runtime Colab baru, sebelum mengimpor pustaka numerik/model.
import sys
import subprocess
from importlib import metadata, util
from packaging.requirements import Requirement

# Pertahankan Torch/CUDA, NumPy, pandas, SciPy dan sklearn bawaan Colab.
# Mengganti pustaka numerik yang sudah dimuat kernel dapat menyebabkan konflik ABI.
CORE_REQUIREMENTS = {
    'torch': 'torch>=2.6', 'numpy': 'numpy>=1.26,<3',
    'pandas': 'pandas>=2,<3', 'scipy': 'scipy>=1.10,<2',
    'scikit-learn': 'scikit-learn>=1.2,<2',
}
for package, requirement in CORE_REQUIREMENTS.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError as error:
        raise RuntimeError('Gunakan runtime Google Colab baru; pustaka dasar belum ada: '+package) from error
    if not Requirement(requirement).specifier.contains(installed):
        raise RuntimeError(requirement+' diperlukan; runtime ini memiliki '+installed+'. Pilih runtime Colab baru.')

# Tokenizers 0.19 tidak menyediakan wheel Python 3.13; tetap satu versi per runtime.
TRANSFORMERS_REQUIREMENT = 'transformers==4.48.3' if sys.version_info >= (3, 13) else 'transformers==4.44.2'
EXTRA_REQUIREMENTS = [TRANSFORMERS_REQUIREMENT, 'openpyxl>=3.1,<4']
to_install = []
for requirement in EXTRA_REQUIREMENTS:
    req = Requirement(requirement)
    try:
        installed = metadata.version(req.name)
    except metadata.PackageNotFoundError:
        installed = None
    if installed is None or not req.specifier.contains(installed):
        if req.name in sys.modules:
            raise RuntimeError(req.name+' sudah diimpor dengan versi berbeda. Restart session, lalu Run all dari atas.')
        to_install.append(requirement)
if to_install:
    # Kendala ini mencegah pip mengganti pustaka numerik yang digunakan kernel.
    from pathlib import Path
    constraints = Path('/tmp/aes35_native_constraints.txt')
    constraints.write_text(''.join(p+'=='+metadata.version(p)+'\n' for p in CORE_REQUIREMENTS))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           '--constraint', str(constraints), *to_install])
print('Setup selesai. Semua cell berikutnya berjalan langsung di kernel notebook ini.')
print('Python:', sys.version.split()[0])


## 2 · Import dan ruang variabel notebook

Semua fungsi dan variabel berada pada kernel yang sama. `SingleScoreModel` adalah kelas model; `ROBERTA_CHECKPOINT` nanti adalah nama bobot pretrained yang dimuat kelas tersebut.


In [ ]:
import os, sys, json, math, hashlib, random, warnings, re, itertools, unicodedata, zipfile, subprocess, gc, logging, html
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.functional import normalize
from torch.utils.data import DataLoader
from torch.utils.checkpoint import checkpoint as activation_checkpoint
def configure_text_only_transformers():
    """Hindari import vision ops yang tidak diperlukan oleh RoBERTa/BGE/LoRA."""
    import os
    os.environ['USE_TORCH'] = '1'
    os.environ['USE_TF'] = '0'
    os.environ['USE_FLAX'] = '0'
    from transformers.utils import import_utils
    # Transformers 4.44/4.48 mendeteksi torchvision dari keberadaan paket,
    # meskipun binary ops-nya tidak cocok dengan torch runtime. Notebook ini
    # hanya teks: jangan muat backend gambar melalui import tidak langsung PEFT.
    # Backend teks dikonfigurasi pada kernel notebook; tidak memuat operator vision.
    if not hasattr(import_utils, '_torchvision_available'):
        raise RuntimeError('Versi Transformers berbeda dari versi notebook yang didukung.')
    import_utils._torchvision_available = False
    return {'transformers_backend': 'pytorch_text_only', 'torchvision_enabled': False}

configure_text_only_transformers()
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from scipy.stats import pearsonr
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from openpyxl import Workbook, load_workbook
from openpyxl.cell import WriteOnlyCell
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
SCENARIOS = ('S3',)  # Hanya model fusion yang dijalankan.
NOTEBOOK_CODE_SHA256='25738722fdd326f1a0fcda604f4ae0e1f2b397ccfd75cc021a5af60df986b6d2'

from IPython.display import display, HTML


<a id="nama-model"></a>
## 3 · Nama model pretrained

Cari dan ubah nama checkpoint di cell ini. Revision mempertahankan versi model/tokenizer yang sama seperti kontrol E30.


In [ ]:
# NAMA MODEL PRETRAINED — ubah di sini jika membuat eksperimen lain.
ROBERTA_CHECKPOINT = 'flax-community/indonesian-roberta-base'
BGE_CHECKPOINT = 'BAAI/bge-m3'

# Revision mengunci bobot dan tokenizer yang sama seperti E30.
ROBERTA_REVISION = 'cc5ecee7de8532880ad08d802292a6f42865bc83'
BGE_REVISION = '5617a9f61b028005a4858fdac845db406aefb181'

print('RoBERTa:', ROBERTA_CHECKPOINT, '| encoder ikut fine-tuning')
print('BGE-M3:', BGE_CHECKPOINT, '| encoder frozen; proyeksi/head tetap belajar')


<a id="kolom-xy"></a>
## 4 · Kolom input X, target y, dan file dataset

X memiliki tiga fitur teks; `rater` hanya y. Email/nomor respons hanya metadata. Mengubah nama kolom di sini benar-benar mengubah pembacaan Excel, bukan hanya label tampilan.


In [ ]:
# X terdiri atas TIGA kolom teks. Y berisi SATU skor guru.
QUESTION_COLUMN = 'soal'
REFERENCE_COLUMN = 'jawaban_referensi'
ANSWER_COLUMN = 'response_text'
TARGET_COLUMN = 'rater'
INPUT_COLUMNS = [QUESTION_COLUMN, REFERENCE_COLUMN, ANSWER_COLUMN]

# Kedua kolom ini hanya metadata/kelompok pemisahan data; tidak masuk encoder.
GROUP_COLUMN = 'Email address'
RESPONSE_NUMBER_COLUMN = 'response_number'
REQUIRED = [GROUP_COLUMN, RESPONSE_NUMBER_COLUMN, *INPUT_COLUMNS, TARGET_COLUMN]
if len(set(REQUIRED)) != len(REQUIRED):
    raise ValueError('Nama kolom input, target dan metadata harus berbeda.')

# Kosongkan untuk menggunakan pemilih file upload Colab.
DATASET_FILE = os.environ.get('AES300_DATASET', '')
SHEET_NAME = 'Sheet1'
EXPECTED_ROWS = 300
print('Kolom X:', INPUT_COLUMNS)
print('Kolom Y:', TARGET_COLUMN, '| skor guru 0–100')


<a id="hyperparameter"></a>
## 5 · Hyperparameter E44

Encoder5e-6seragam,MLP/proyeksi/BiGRU/co-attention/pooling/norm1e-4,BiGRU64perarah,attention64,dropoutMLP0.1. E44 menambah SCALAR_MIX_LAST_LAYERS=4 dan SCALAR_MIX_LR=1e-4; bobot awal seragam0.25.

Perlakuan E44: SCALAR_MIX_GAMMA_INITIAL=1.0. Gamma ikut dipelajari dengan LR SCALAR_MIX_LR=1e-4. Semua pengaturan lain mengikuti E41.


In [ ]:
# HIPERPARAMETER PENELITIAN — E30 MLP; Scalar Mix empat layer terakhir.
EPOCHS = 25
FOLDS = 5
EARLY_STOPPING_PATIENCE = 5
TRAIN_SEED = 42  # Satu konstanta pengacakan, tanpa loop multi-seed.
SPLIT_SEED = 2026
TEST_FRACTION = 0.20
MICRO_BATCH = 1
EFFECTIVE_BATCH = 8

ENCODER_LR = 5e-6             # Tetap E30; bukan LR E37 7e-6.
S3_MLP_LR = 1e-4              # MLP S3 tetap nilai asli E30.
S3_LAYERS_LR = 1e-4           # Kembali ke E30: BiGRU/co-attention/pooling/norm = 1e-4.
PROJECTION_LR = 1e-4          # Proyeksi BGE projection.* tidak ikut diturunkan.

BIGRU_HIDDEN_PER_DIRECTION = 64   # E30: dua arah menghasilkan128fitur/token.
POOLING_ATTENTION_HIDDEN = 64  # Ukuran init acuan; layer skor attention dihapus sebelum training.
DROPOUT = 0.1                   # Parameter lama untuk layer lain; tetap E30.
S3_MLP_DROPOUT = 0.1             # Kembali ke E30; bukan dropout0.2 E38.
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRADIENT_CLIP = 1.0

ROBERTA_MAX_TOKENS = 512
BGE_MAX_TOKENS = 1024
WINDOW_OVERLAP = 128
CONTEXT_MAX_TOKENS = 192
QUESTION_MAX_TOKENS = 64
DEVICE = 'cuda'
REQUIRE_GPU = True

SCALAR_MIX_LAST_LAYERS = 4      # Empat layer Transformer terakhir, tidak termasuk embeddings.
SCALAR_MIX_LR = 1e-4            # LR untuk empat logit campuran yang dipelajari.

SCALAR_MIX_GAMMA_INITIAL = 1.0  # Satu parameter baru; LR mengikuti SCALAR_MIX_LR=1e-4.

# SATU PERLAKUAN E44: fungsi loss, bukan LR atau arsitektur baru.
RANKING_WEIGHT = 0.1
RANKING_RECOMPUTE = True  # Hitung ulang aktivasi saat backward agar microbatch tetap kecil.


## 6 · Bagian yang dijalankan dan contoh jawaban baru

Jika Run all dengan nilai bawaan, notebook menjalankan training. Prediksi baru memerlukan ketiga teks, tanpa nilai guru.


In [ ]:
# KONTROL EKSEKUSI — Run all menjalankan seluruh resep default.
# Untuk membaca/audit dataset saja: ubah keempat RUN_... menjadi False.
RUN_MODELS = True
RUN_TRAINING = True
RUN_WORD_VECTORS = True
RUN_FINAL_FEATURES = True
DOWNLOAD_WITH_COLAB_BROWSER = True

# OPSIONAL: prediksi jawaban baru setelah training; isi ketiganya bersama-sama.
# Jika kosong, bagian contoh menggunakan satu jawaban test yang sudah dipisahkan.
NEW_QUESTION = ''
NEW_REFERENCE = ''
NEW_ANSWER = ''


## 7 · Lokasi lokal dan upload Excel

Kode pemilih file berada langsung di sini. File sumber disalin, tidak ditimpa. Workspace dan checkpoint E44 terpisah dari E30/E33/E34.


In [ ]:
IN_COLAB = 'google.colab' in sys.modules or Path('/content').is_dir()
default_workspace = '/content/AES_300_E44_E43_S3_MSE_ListNet_Native_Learnable_GPU' if IN_COLAB else str(Path.cwd()/'outputs/AES_300_E44_E43_S3_MSE_ListNet_Native_Learnable_GPU')
ROOT = Path(os.environ.get('AES300_WORKSPACE', default_workspace)).expanduser().resolve()
if ROOT == Path('/content/drive') or Path('/content/drive') in ROOT.parents:
    raise ValueError('Pilih penyimpanan runtime lokal, bukan Google Drive.')
ROOT.mkdir(parents=True, exist_ok=True)

def resolve_dataset_upload(workspace, configured='', search_paths=None, browser_upload=False):
    """Web Colab: pakai file yang ada, atau tampilkan Choose Files pada cell ini."""
    from pathlib import Path
    import hashlib
    import os
    import shutil
    workspace=Path(workspace).expanduser().resolve()
    destination=workspace/'data/dataset300/fix_dataset_300.xlsx'
    if configured:
        candidates=[Path(configured).expanduser().resolve()]
        if not candidates[0].is_file():
            raise FileNotFoundError('DATASET_FILE/AES300_DATASET belum ditemukan: '+str(candidates[0]))
    else:
        places=search_paths if search_paths is not None else [Path('/content'),Path.cwd(),workspace]
        candidates=[]
        for place in places:
            path=(Path(place)/'fix_dataset_300.xlsx').resolve()
            if path.is_file() and path not in candidates and path!=destination:
                candidates.append(path)
        if not candidates and destination.is_file():
            candidates=[destination]
    if not candidates and browser_upload:
        from google.colab import files
        destination.parent.mkdir(parents=True,exist_ok=True)
        print('Klik Choose Files, lalu pilih fix_dataset_300.xlsx dari komputer.',flush=True)
        previous=Path.cwd()
        try:
            os.chdir(destination.parent)
            uploaded=files.upload()
        finally:
            os.chdir(previous)
        if 'fix_dataset_300.xlsx' not in uploaded:
            raise ValueError('Pilih fix_dataset_300.xlsx. Jika dibatalkan, jalankan ulang cell setup ini.')
        candidates=[destination]
        del uploaded
    if not candidates:
        raise FileNotFoundError(
            'Dataset belum ada. Di web Colab, jalankan cell setup ini dan pilih '
            'fix_dataset_300.xlsx melalui Choose Files. Alternatif: panel Files/Berkas kiri '
            '→ Upload, lalu jalankan ulang cell ini. Jika lokasinya berbeda, isi DATASET_FILE.')
    hashes={hashlib.sha256(p.read_bytes()).hexdigest() for p in candidates}
    if len(hashes)>1:
        raise ValueError('Ada beberapa dataset berbeda: '+', '.join(map(str,candidates))+
                         '. Isi DATASET_FILE dengan path file yang ingin dipakai.')
    source=candidates[0]
    if not source.stat().st_size:
        raise ValueError('File upload masih kosong; tunggu upload selesai lalu jalankan ulang cell ini.')
    destination.parent.mkdir(parents=True,exist_ok=True)
    if source!=destination:
        shutil.copy2(source,destination)
    print('Dataset dari upload lokal:',source,'→',destination,flush=True)
    return destination

DATASET_PATH = resolve_dataset_upload(ROOT, configured=DATASET_FILE, browser_upload=IN_COLAB)
print('Dataset:', DATASET_PATH)
print('Workspace:', ROOT)


## 8 · Pemeriksaan konfigurasi dan perangkat

Nilai bawaan di bawah berasal dari variabel yang sudah terlihat di bagian3–6; nilai itu dikumpulkan ke dictionary `cfg`.


In [ ]:
# Nilai bawaan dan fungsi konfigurasi, tersedia langsung di notebook.
SCOPE = 'teacher_single_score_dataset300'

DEFAULTS = {
    'sheet': SHEET_NAME, 'expected_rows': EXPECTED_ROWS,
    # Rentang dinyatakan notebook unggahan, bukan diambil dari min/max test.
    'score_min': 0, 'score_max': 100,
    'test_fraction': TEST_FRACTION, 'folds': FOLDS, 'seed': TRAIN_SEED, 'split_seed': SPLIT_SEED, 'split_candidates': 128,
    # Maksimal 25 epoch/fold; berhenti bila RMSE validasi tidak membaik selama patience.
    'epochs': EPOCHS, 'early_stopping_patience': EARLY_STOPPING_PATIENCE, 'effective_batch': EFFECTIVE_BATCH, 'micro_batch': MICRO_BATCH,
    'encoder_lr': ENCODER_LR, 's3_encoder_lr': ENCODER_LR, 'projection_lr': PROJECTION_LR, 'head_lr': 1e-4, 'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO, 'gradient_clip': GRADIENT_CLIP, 'dropout': DROPOUT, 's3_mlp_dropout': S3_MLP_DROPOUT,
    'roberta_max_length': ROBERTA_MAX_TOKENS, 'roberta_overlap': WINDOW_OVERLAP,
    'bge_max_length': BGE_MAX_TOKENS, 'bge_overlap': WINDOW_OVERLAP, 'bge_batch': 4, 'device': DEVICE,
    'roberta': {'checkpoint': ROBERTA_CHECKPOINT,
                'revision': ROBERTA_REVISION},
    'bge': {'checkpoint': BGE_CHECKPOINT,
            'revision': BGE_REVISION},
    # Batas konteks dihitung termasuk prefix; semua token jawaban tetap dicakup.
    'context_max_tokens': CONTEXT_MAX_TOKENS, 'question_max_tokens': QUESTION_MAX_TOKENS,
    'new_question': '', 'new_reference': '',
    'input_policy':'three_texts_all_scenarios_v1', 'input_format':'single_sequence',
    's3_pooling': 'bigru_coattention_mean_pooling', 's3_sequence_hidden': BIGRU_HIDDEN_PER_DIRECTION, 'pooling_attention_hidden': POOLING_ATTENTION_HIDDEN,
    'scalar_mix_gamma_init': SCALAR_MIX_GAMMA_INITIAL, 'scalar_mix_layers': SCALAR_MIX_LAST_LAYERS, 'scalar_mix_lr': SCALAR_MIX_LR,
    's3_mlp_lr': S3_MLP_LR, 's3_layers_lr': S3_LAYERS_LR,
    'loss_recipe': 'mse_same_question_listnet_v1',
    'ranking_weight': RANKING_WEIGHT, 'ranking_recompute': RANKING_RECOMPUTE,
    's3_fusion': 'concat', 
    'vector_source': 'pretrained', 'trace_position': 0,
}


# Gabungkan parameter pilihan dengan nilai bawaan, lalu periksa batas skor, batch, dan panjang input.
def config(values):
    result = {**DEFAULTS, **values}
    if result['score_min'] >= result['score_max']:
        raise ValueError('score_min harus lebih kecil dari score_max')
    if any(not isinstance(result[k], int) for k in ('score_min', 'score_max')):
        raise ValueError('Workflow ini memakai kategori skor integer yang dinyatakan terlebih dahulu.')
    if not 0 < result['test_fraction'] < 1 or result['folds'] < 2 or result['split_candidates'] < 1:
        raise ValueError('Split/folds tidak sah')
    if result['epochs'] < 1 or not 1 <= result['micro_batch'] <= result['effective_batch']:
        raise ValueError('Epoch/batch tidak sah')
    if type(result['early_stopping_patience']) is not int or result['early_stopping_patience'] < 1:
        raise ValueError('early_stopping_patience harus bilangan bulat positif.')
    if result['vector_source'] not in ('pretrained', 'pretrained_pair', 'S3'):
        raise ValueError('vector_source harus pretrained, pretrained_pair, atau S3')
    if result['device'] not in ('auto', 'cpu', 'cuda'):
        raise ValueError('device harus auto, cpu, atau cuda')
    if not 0 <= result['warmup_ratio'] < 1:
        raise ValueError('warmup_ratio tidak sah')
    if any(not math.isfinite(result[k]) or result[k] <= 0
           for k in ('encoder_lr', 's3_encoder_lr', 'projection_lr', 'head_lr', 's3_mlp_lr', 's3_layers_lr', 'gradient_clip', 'bge_batch')):
        raise ValueError('Learning rate, gradient clip, dan batch BGE harus positif.')
    if not 0 <= result['s3_mlp_dropout'] < 1:
        raise ValueError('s3_mlp_dropout harus berada pada rentang [0,1).')
    if not 0 <= result['dropout'] < 1 or result['weight_decay'] < 0:
        raise ValueError('Dropout/weight decay tidak sah.')
    if not 2 < result['roberta_max_length'] <= 512 or not 0 <= result['roberta_overlap'] < result['roberta_max_length']-2:
        raise ValueError('Window RoBERTa melebihi batas atau overlap tidak sah.')
    if not 2 < result['bge_max_length'] <= 8192:
        raise ValueError('Batas BGE harus 3–8192 token termasuk special token.')
    if not (type(result['context_max_tokens']) is int and type(result['question_max_tokens']) is int
            and 1 <= result['question_max_tokens'] < result['context_max_tokens']
            and result['context_max_tokens']+4+result['roberta_overlap'] < result['roberta_max_length']):
        raise ValueError('Anggaran konteks harus menyisakan token jawaban lebih banyak daripada overlap.')
    if str(result.get('new_answer', '')).strip() and not all(str(result.get(k, '')).strip() for k in ('new_question', 'new_reference')):
        raise ValueError('Untuk NEW_ANSWER baru, isi juga NEW_QUESTION dan NEW_REFERENCE.')
    if not (type(result['bge_overlap']) is int and 0 <= result['bge_overlap']
            < result['bge_max_length']-result['context_max_tokens']-4):
        raise ValueError('Anggaran konteks BGE harus menyisakan kapasitas jawaban di atas overlap.')
    if result['input_policy']!='three_texts_all_scenarios_v1':
        raise ValueError('Notebook ini khusus tiga input pada semua skenario.')
    if result['input_format'] not in ('single_sequence','context_pair'):
        raise ValueError('Format input tidak sah.')
    if result['s3_pooling'] != 'bigru_coattention_mean_pooling' or type(result['s3_sequence_hidden']) is not int or result['s3_sequence_hidden'] < 1:
        raise ValueError('Jenis/hidden pooling S3 tidak cocok notebook eksperimen ini.')
    if type(result['pooling_attention_hidden']) is not int or result['pooling_attention_hidden'] < 1:
        raise ValueError('pooling_attention_hidden harus bilangan bulat positif.')
    if type(result['scalar_mix_layers']) is not int or result['scalar_mix_layers'] < 1:
        raise ValueError('scalar_mix_layers harus bilangan bulat positif.')
    if not math.isfinite(result['scalar_mix_lr']) or result['scalar_mix_lr'] <= 0:
        raise ValueError('scalar_mix_lr harus positif dan finite.')
    if isinstance(result['scalar_mix_gamma_init'], bool) or not math.isfinite(result['scalar_mix_gamma_init']):
        raise ValueError('scalar_mix_gamma_init harus finite.')
    if result['s3_fusion'] != 'concat':
        raise ValueError('Fusion/hidden gate tidak cocok dengan notebook ini.')
    if result['loss_recipe'] != 'mse_same_question_listnet_v1':
        raise ValueError('loss_recipe tidak cocok notebook E44.')
    if isinstance(result['ranking_weight'], bool) or not math.isfinite(result['ranking_weight']) or result['ranking_weight'] < 0:
        raise ValueError('ranking_weight harus finite dan tidak negatif.')
    if type(result['ranking_recompute']) is not bool:
        raise ValueError('ranking_recompute harus boolean.')
    return result


In [ ]:
# Bantuan membaca tabel: penamaan kolom, angka, dan penjelasan kata berulang.
LABELS = {'row_id':'ID Jawaban','word':'Kata','covered':'Tercakup','word_index':'Urutan Kata',
          'token_id':'ID Token','token':'Token','position':'Posisi Token','window':'Bagian Teks',
          'attention_mask':'Dibaca Model','skor_guru':'Nilai Guru','prediksi_mentah':'Prediksi Skor',
          'kategori_salinan':'Skor Dibulatkan','di_luar_rentang':'Di Luar Rentang',
          'Skenario':'Skenario','Fold':'Fold','AC2':'Gwet AC2','RMSE':'RMSE (poin)',
          'out_of_range_fraction':'Di Luar Rentang (%)','Status':'Keterangan',
          'teks_model':'Jawaban Siswa','teks_asli':'Teks Asli','score':'Nilai Guru',
          'role':'Penggunaan Data','connected_group':'Kelompok Jawaban',
          'input_format':'Format Input','response_text':'Jawaban Siswa','rater':'Nilai Guru','soal':'Soal',
          'sentence_index':'Urutan Kalimat','text':'Kalimat','fold':'Fold','scenario':'Skenario',
          'encoder':'Encoder','vector':'Panjang Vektor','head':'Lapisan Prediksi','output':'Jumlah Skor',
          'used_for':'Digunakan untuk','id':'Kode','title':'Judul',
          'rows':'Jumlah Jawaban','file':'File Hasil','stage':'Isi File','status':'Keadaan',
          'sheets_and_shapes':'Tabel yang Tersimpan','epoch':'Epoch',
          'train_mse_scaled':'Loss Latih (MSE)','target_loss':'Nilai untuk Menghitung Loss',
          'best_epoch_so_far':'Epoch Terbaik Sejauh Ini','epochs_without_improvement':'Epoch Tanpa Perbaikan',
          'selisih_mentah':'Selisih Prediksi − Guru','berubah':'Teks Dibersihkan?',
          'token_index':'Urutan Token','start':'Awal Teks','end':'Akhir Teks',
          'token_indices':'Urutan Potongan Token','token_ids':'ID Potongan Token',
          'covered_tokens':'Token yang Terbaca','total_tokens':'Jumlah Token',
          'window_coverage':'Jumlah Bagian yang Membaca Token',
          'occurrence_indices':'Posisi Kata yang Dibandingkan','same_token_ids':'ID Token Sama?',
          'initial_max_abs_difference':'Selisih Terbesar Sebelum Konteks',
          'contextual_max_abs_difference':'Selisih Terbesar Setelah Konteks',
          'contextual_allclose_1e-6':'Vektor Setelah Konteks Sama?',
          'undefined_reasons':'Alasan Metrik Tidak Tersedia','authors':'Penulis','year':'Tahun',
          'url':'Tautan','limitation':'Batas Penerapan','library':'Pustaka','version':'Versi',
          'name':'Pengaturan','value':'Nilai','field':'Keterangan','target':'Jenis Nilai',
          'Posisi Kata 1':'Posisi Kata A','Posisi Kata 2':'Posisi Kata B',
          'ID Token Sama':'ID Token Sama?','Vektor Awal Sama':'Vektor Sebelum Konteks Sama?',
          'Vektor Akhir Sama':'Vektor Setelah Konteks Sama?',
          'Apa Artinya':'Penjelasan'}

REPEATED_COLUMNS = ['Model','Kata','Posisi Kata A','Posisi Kata B','ID Token Sama?',
                    'Vektor Sebelum Konteks Sama?','Vektor Setelah Konteks Sama?',
                    'Penjelasan']

MODELS = {'S3':'Fusion RoBERTa + BGE-M3'}

ROLES = {'development':'Data latih (termasuk validasi silang)','test':'Data uji',
         'train':'Latih','validation':'Validasi'}

REPEATED_NOTES = [
    'Satu baris membandingkan kata yang sama pada dua posisi di jawaban contoh. Posisi dihitung mulai 1.',
    'Sebelum konteks = vektor dari tabel kosakata model. Setelah konteks = vektor setelah model membaca posisi dan kata di sekitarnya.',
    'Kolom Vektor ... Sama? memeriksa selisih setiap komponen, dengan batas 0,000001. Ini batas perbandingan angka komputer, bukan tolerance skor AES.',
]

METRIC_NOTES = [
    'E44 menjalankan S3: fusion RoBERTa + BGE-M3. RMSE memakai satuan poin skor: lebih kecil berarti kesalahan lebih kecil.',
    'Pearson menunjukkan hubungan perubahan skor; QWK dan Gwet AC2 menunjukkan kesepakatan skor kategori. Korelasi tinggi saja belum menjamin prediksi mendekati nilai guru.',
    'Tanda — berarti hasil tidak tersedia, bukan nol. Baca keempat metrik bersama-sama.',
]

REPEATED_HEADERS=REPEATED_COLUMNS

LABELS.update({'input_segment':'Bagian Input','question_tokens':'Token Soal Awal',
    'question_kept':'Token Soal Dibaca','reference_tokens':'Token Referensi Awal',
    'reference_kept':'Token Referensi Dibaca','question_truncated':'Soal Terpotong?',
    'reference_truncated':'Referensi Terpotong?','context_tokens_kept':'Token Konteks Dibaca',
    'content_tokens':'Token Jawaban','capacity':'Kapasitas Jawaban per Window',
    'stride':'Langkah Window','starts':'Posisi Awal Window','windows':'Jumlah Window',
    'pair_special_tokens':'Token Khusus Pasangan','answer_tokens_uncovered':'Token Jawaban Tidak Tercakup',
    'context_max_tokens':'Batas Token Konteks Semua Encoder','question_max_tokens':'Batas Token Soal Semua Encoder'})

def friendly_header(name):
    match = re.fullmatch(r'v_(\d+)', str(name))
    return 'Komponen '+str(int(match[1])+1) if match else LABELS.get(name, str(name).replace('_',' '))


def friendly_value(name, value):
    if hasattr(value,'item'):
        try:value=value.item()
        except ValueError:pass
    if name == 'row_id' and isinstance(value,str):
        match = re.fullmatch(r'row_(\d+)',value)
        return f'J{int(match[1]):03d}' if match else value
    if name == 'role':
        return ROLES.get(value,value)
    if name == 'status':
        return {'ACTUAL EXECUTION':'Dihasilkan saat kode dijalankan',
                'DESIGN / CONFIGURATION':'Rancangan dan pengaturan model'}.get(value,value)
    if name == 'attention_mask' and value in (0,1):
        return 'Ya' if value else 'Tidak'
    if name == 'out_of_range_fraction' and isinstance(value,(int,float)):
        return value*100
    if name in ('word_index','sentence_index','token_index','position','window','answer_position','reference_position') and isinstance(value,(int,float)):
        return value+1
    if isinstance(value,bool):
        return 'Ya' if value else 'Tidak'
    return value


def repeated_word_rows(records,model):
    for row in records:
        positions=row.get('occurrence_indices')
        if isinstance(positions,str):positions=json.loads(positions)
        positions=positions or []
        same_ids=row.get('same_token_ids')
        initial=row.get('initial_max_abs_difference')
        same_initial=None if initial is None else abs(initial)<=1e-6
        same_context=row.get('contextual_allclose_1e-6')
        if same_ids is False:explanation='Kata sama, tetapi potongan token berbeda. Perhatikan tokenisasinya sebelum membandingkan vektor.'
        elif same_ids is None:explanation='Informasi kesamaan ID token belum tersedia.'
        elif same_context is None:explanation='Hasil perbandingan vektor akhir belum tersedia.'
        elif same_context:explanation='ID token sama; vektor akhir juga sama dalam toleransi numerik 0,000001.'
        else:explanation='ID token sama, tetapi vektor setelah membaca konteks berbeda. Posisi dan kata di sekitarnya dapat memengaruhi vektor.'
        label={'roberta':'RoBERTa','bge':'BGE-M3'}.get(model,model)
        yield [label,row.get('word'),positions[0]+1 if positions else None,positions[1]+1 if len(positions)>1 else None,
               friendly_value('',same_ids),friendly_value('',same_initial),friendly_value('',same_context),
               explanation]

def preview_table(headers, records, title='', row_count=None, max_rows=8, max_columns=8):
    """Bentuk preview tanpa mengubah DataFrame/Excel sumber atau menghitung ulang model."""
    headers=list(headers)
    records=list(itertools.islice(records,max_rows))
    count=len(records) if row_count is None else row_count
    notes=[]
    original_columns=len(headers)
    if 'occurrence_indices' in headers:
        values=list(repeated_word_rows(records,''))
        # Nama model hanya ditampilkan bila memang diberikan oleh pemanggil.
        headers=REPEATED_HEADERS[1:]
        values=[r[1:] for r in values]
        notes=REPEATED_NOTES
        empty='Belum ada pasangan kata berulang yang bisa dibandingkan. Ini bukan error; lihat tabel vektor kata di atas.'
    elif {'file','stage','sheets_and_shapes'}.issubset(headers):
        # Dictionary ukuran tabel adalah rincian internal, bukan isi laporan belajar.
        headers=['File Hasil','Isi File','Keadaan']
        values=[[r.get('file'),r.get('stage'),friendly_value('status',r.get('status'))] for r in records]
        notes=['Daftar ini menunjukkan file yang sudah dibuat. Pilih lima laporan utama pada bagian unduhan untuk membaca hasilnya.']
        empty='Belum ada file hasil yang tersimpan.'
    else:
        selected=headers[:max_columns]
        values=[[friendly_value(key,row.get(key)) for key in selected] for row in records]
        headers=[friendly_header(key) for key in selected]
        empty='Belum ada baris hasil pada tahap ini. Tidak ada angka pengganti yang ditampilkan.'
        if 'Vektor Setelah Konteks Sama?' in headers or 'Vektor Akhir Sama' in headers:
            notes=REPEATED_NOTES
            empty='Belum ada pasangan kata berulang yang bisa dibandingkan. Ini bukan error; lihat tabel vektor kata di atas.'
        elif {'RMSE','Pearson','QWK','AC2'}.issubset(selected) or 'RMSE (poin)' in headers:
            notes=METRIC_NOTES
        elif 'prediksi_mentah' in selected or 'Prediksi S1' in headers:
            notes=['Prediksi Skor adalah nilai yang diberikan model. Nilai Guru adalah acuan jika tersedia.',
                   'Skor Dibulatkan adalah salinan untuk metrik kategori; prediksi asli tetap disimpan. Di Luar Rentang berarti prediksi melewati batas skor yang ditetapkan.']
        elif 'token_id' in selected or 'ID Token' in headers:
            notes=['Token adalah potongan teks yang dibaca model; satu kata bisa terpecah menjadi beberapa token. ID Token adalah nomor dalam kosakata model, bukan nilai siswa.',
                   'Posisi Token dan Bagian Teks dihitung mulai 1. Token khusus seperti <s> dan </s> menandai batas input.']
        components=sum(bool(re.fullmatch(r'v_\d+|Komponen \d+',str(h))) for h in selected)
        if components:
            notes=['Satu baris berisi satu vektor. Komponen 1, 2, dan seterusnya adalah angka penyusun vektor, bukan skor penilaian.',
                   'Tabel ini hanya memperlihatkan beberapa komponen pertama. Seluruh komponen tersimpan di Excel. Komponen yang sama nomornya pada RoBERTa dan BGE-M3 tidak memiliki arti yang otomatis sama.']
    return {'title':title,'columns':headers,'rows':values,'shape':[count,original_columns],
            'notes':notes,'empty_message':empty}


def cell_text(value):
    if value is None:return '—'
    if isinstance(value,bool):return 'Ya' if value else 'Tidak'
    if isinstance(value,float):
        return format(value,'.8g') if math.isfinite(value) else '—'
    if isinstance(value,dict):
        return '; '.join(f'{friendly_header(k)}: {cell_text(v)}' for k,v in value.items()) or '—'
    if isinstance(value,(list,tuple)):
        return ', '.join(cell_text(x) for x in value) or '—'
    return str(value)


def render_table(event):
    """HTML escapé dengan warna eksplisit agar terbaca pada Colab terang maupun gelap."""
    esc=lambda value:html.escape(cell_text(value))
    parts=['<section style="background:#f8fafc!important;color:#172035!important;padding:18px;border:1px solid #94a3b8;border-radius:12px;margin:12px 0">',
           '<h3 style="color:#172035!important;margin:0 0 10px">'+esc(event.get('title',''))+'</h3>']
    for note in event.get('notes',[]):parts.append('<p style="color:#334155!important;margin:6px 0">'+esc(note)+'</p>')
    if not event['rows']:
        parts.append('<p>'+esc(event.get('empty_message','Belum ada hasil.'))+'</p>')
    else:
        parts.append(f'<p style="color:#475569!important">Ditampilkan {len(event["rows"])} dari {event["shape"][0]} baris. Angka pada layar diringkas; nilai lengkap ada di Excel.</p>')
        th='background:#172035!important;color:#fff!important;padding:10px;border:1px solid #64748b;text-align:left'
        td='background:#f8fafc!important;color:#172035!important;padding:10px;border:1px solid #cbd5e1;vertical-align:top;min-width:90px;max-width:460px;white-space:normal;overflow-wrap:anywhere'
        parts.append('<div style="max-height:420px;overflow:auto"><table style="border-collapse:collapse"><thead><tr>')
        parts.extend('<th style="'+th+'">'+esc(x)+'</th>' for x in event['columns'])
        parts.append('</tr></thead><tbody>')
        for row in event['rows']:
            parts.append('<tr>'+''.join('<td style="'+td+'">'+esc(x)+'</td>' for x in row)+'</tr>')
        parts.append('</tbody></table></div>')
    parts.append('</section>')
    return ''.join(parts)

# Memeriksa nama GPU, status T4, dan VRAM; Colab berhenti sebelum training jika GPU belum aktif.
def check_gpu(require_gpu=False):
    """Periksa hardware yang benar-benar tersedia, tanpa memuat model pretrained."""
    available = torch.cuda.is_available()
    if not available:
        if require_gpu:
            raise RuntimeError('GPU belum aktif. Pilih Runtime → Change runtime type → T4 GPU, '
                               'lalu jalankan notebook dari atas. Training Colab tidak dimulai di CPU.')
        return {'name': 'CPU', 'cuda_available': False, 'is_t4': False, 'memory_gib': None}
    name = torch.cuda.get_device_name(0)
    return {'name': name, 'cuda_available': True, 'is_t4': 'T4' in name.upper(),
            'memory_gib': round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2)}

LABELS.update({'attention_weight':'Bobot Attention','pooling_weight':'Bobot Pooling'})

LABELS.update({'fusion_gate':'Gate RoBERTa','roberta_multiplier':'Pengali RoBERTa','bge_multiplier':'Pengali BGE'})

LABELS.update({'reference_content_kept':'Token isi referensi dibaca','answer_position':'Posisi Token Jawaban','answer_token':'Token Jawaban','reference_position':'Posisi Token Referensi','reference_token':'Token Referensi','alignment_weight':'Bobot Kecocokan'})


In [ ]:
def show(frame, title='', rows=8, columns=8):
    frame = frame if isinstance(frame, pd.DataFrame) else pd.DataFrame(frame)
    view = preview_table(frame.columns, frame.head(rows).to_dict('records'), title,
                         row_count=len(frame), max_rows=rows, max_columns=columns)
    display(HTML(render_table(view)))


In [ ]:
# cfg hanya mengumpulkan pilihan di atas agar dapat diteruskan ke fungsi Python.
cfg = config(dict(
    dataset=str(DATASET_PATH), output=str(ROOT/'outputs'),
    sheet=SHEET_NAME, expected_rows=EXPECTED_ROWS,
    score_min=0, score_max=100, seed=TRAIN_SEED, split_seed=SPLIT_SEED,
    test_fraction=TEST_FRACTION, epochs=EPOCHS, folds=FOLDS,
    micro_batch=MICRO_BATCH, effective_batch=EFFECTIVE_BATCH,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    encoder_lr=ENCODER_LR, s3_encoder_lr=ENCODER_LR,
    head_lr=1e-4, s3_mlp_lr=S3_MLP_LR, scalar_mix_gamma_init=SCALAR_MIX_GAMMA_INITIAL, scalar_mix_layers=SCALAR_MIX_LAST_LAYERS, scalar_mix_lr=SCALAR_MIX_LR,
    s3_layers_lr=S3_LAYERS_LR, projection_lr=PROJECTION_LR,
    dropout=DROPOUT, s3_mlp_dropout=S3_MLP_DROPOUT, weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO, gradient_clip=GRADIENT_CLIP,
    roberta={'checkpoint':ROBERTA_CHECKPOINT, 'revision':ROBERTA_REVISION},
    bge={'checkpoint':BGE_CHECKPOINT, 'revision':BGE_REVISION},
    roberta_max_length=ROBERTA_MAX_TOKENS, bge_max_length=BGE_MAX_TOKENS,
    roberta_overlap=WINDOW_OVERLAP, bge_overlap=WINDOW_OVERLAP,
    s3_pooling='bigru_coattention_mean_pooling', s3_sequence_hidden=BIGRU_HIDDEN_PER_DIRECTION, pooling_attention_hidden=POOLING_ATTENTION_HIDDEN,
    loss_recipe='mse_same_question_listnet_v1', ranking_weight=RANKING_WEIGHT,
    ranking_recompute=RANKING_RECOMPUTE,
    input_format='single_sequence', device=DEVICE,
    context_max_tokens=CONTEXT_MAX_TOKENS, question_max_tokens=QUESTION_MAX_TOKENS,
    new_answer=NEW_ANSWER, new_question=NEW_QUESTION, new_reference=NEW_REFERENCE,
    vector_source='pretrained', lesson_version='e44_native_mse_listnet_v1',
    source_columns={'question':QUESTION_COLUMN, 'reference':REFERENCE_COLUMN,
                    'answer':ANSWER_COLUMN, 'target':TARGET_COLUMN,
                    'group':GROUP_COLUMN, 'response_number':RESPONSE_NUMBER_COLUMN},
    runtime_versions={p:version(p) for p in ('torch','transformers','numpy','pandas','scikit-learn')},
))
cfg['scenarios'] = list(SCENARIOS)
torch.set_num_threads(2)
gpu = check_gpu(require_gpu=REQUIRE_GPU)
print('Perangkat:', gpu['name'])
display(pd.DataFrame([
    {'Kelompok':'Encoder RoBERTa S3', 'LR':ENCODER_LR},
    {'Kelompok':'MLP S3', 'LR':S3_MLP_LR},
    {'Kelompok':'BiGRU/co-attention/pooling/norm S3', 'LR':S3_LAYERS_LR},
    {'Kelompok':'Proyeksi BGE S3', 'LR':PROJECTION_LR},
]))

display(pd.DataFrame([{'Bagian':'MLP S3 setelah GELU','Dropout p':cfg['s3_mlp_dropout']}, {'Bagian':'Parameter dropout lama/layer lain','Dropout p':cfg['dropout']}]))

display(pd.DataFrame([{'Tahap':'Hidden BiGRU per arah','Dimensi':cfg['s3_sequence_hidden']}, {'Tahap':'Output dua arah / ringkasan co-attention','Dimensi':2*cfg['s3_sequence_hidden']}, {'Tahap':'Masked mean token valid (tanpa bottleneck)','Dimensi':2*cfg['s3_sequence_hidden']}, {'Tahap':'Proyeksi RoBERTa sebelum fusion','Dimensi':768}, {'Tahap':'Concat ke MLP','Dimensi':1024}]))

print('Scalar Mix:', cfg['scalar_mix_layers'], 'layer terakhir | LR:', cfg['scalar_mix_lr'], '| bobot awal seragam')

print('Loss: MSE +', cfg['ranking_weight'], '× ListNet | soal sama | recompute:', cfg['ranking_recompute'])


## 9 · Membaca dan memeriksa dataset

Teks dibersihkan dari karakter kontrol/Unicode; tidak melakukan stemming atau menulis ulang jawaban. Label kosong ditolak. Kelompok email/duplikat disimpan untuk menjaga pemisahan data.


In [ ]:
# Baca dan audit data tanpa mengubah file Excel asli.
# Rapikan Unicode dan karakter kontrol; ejaan, huruf besar, dan tanda baca jawaban tetap dipertahankan.
def clean(text, normalize_punctuation=False):
    text = unicodedata.normalize('NFC', text).replace('\r\n','\n').replace('\r','\n')
    text = ''.join(c for c in text if c in '\n\t' or unicodedata.category(c) != 'Cc')
    if normalize_punctuation:
        text = text.translate(str.maketrans({'“':'"','”':'"','‘':"'",'’':"'",'–':'-','—':'-'})).replace('…','...')
    return text

def digest(path):
    # Checkpoint besar diperiksa bertahap supaya tidak dimuat seluruhnya ke RAM.
    checksum = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8*1024*1024), b''):
            checksum.update(chunk)
    return checksum.hexdigest()


def dump(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name+'.pending')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)+'\n', encoding='utf-8')
    temporary.replace(path)


@dataclass
class Bundle:
    rows: list
    y: np.ndarray
    groups: np.ndarray
    source_sha256: str
    summary: dict

    @property
    def texts(self):
        return [r['text'] for r in self.rows]


# Baca jawaban dan nilai guru, periksa kelengkapan, lalu kelompokkan email/duplikat untuk mencegah kebocoran data.
def read_dataset(cfg):
    path = Path(cfg['dataset'])
    source_hash = digest(path)
    book = load_workbook(path, read_only=True, data_only=True)
    try:
        if cfg['sheet'] not in book.sheetnames:
            raise ValueError(f"Tab {cfg['sheet']} tidak tersedia: {book.sheetnames}")
        data = iter(book[cfg['sheet']].iter_rows(values_only=True))
        header = [str(x).strip() if x is not None else None for x in next(data)]
        if any(header.count(x) != 1 for x in REQUIRED):
            raise ValueError('Enam header wajib hilang atau duplikat: '+', '.join(REQUIRED))
        rows = []
        for excel_row, values in enumerate(data, 2):
            if all(v is None for v in values):
                continue  # Hanya baris format kosong, bukan label/jawaban yang hilang.
            raw = {k: values[i] for i, k in enumerate(header) if k is not None}
            if any(raw[k] is None or not str(raw[k]).strip() for k in REQUIRED):
                raise ValueError(f'Baris {excel_row}: kolom wajib kosong: '+', '.join(k for k in REQUIRED if raw[k] is None)+'; tidak diisi otomatis.')
            score = raw[TARGET_COLUMN]
            if isinstance(score, bool) or not isinstance(score, (int, float)) or not math.isfinite(score):
                raise ValueError(f'Baris {excel_row}: rater harus angka skor guru, bukan teks/ID.')
            if int(score) != score or not cfg['score_min'] <= score <= cfg['score_max']:
                raise ValueError(f'Baris {excel_row}: skor bukan kategori integer dalam rentang yang dinyatakan.')
            for key in (QUESTION_COLUMN,REFERENCE_COLUMN):
                if not isinstance(raw[key],str) or not raw[key].strip():
                    raise ValueError(f'Baris {excel_row}: {key} harus teks yang tidak kosong.')
            if not isinstance(raw[ANSWER_COLUMN], str):
                raise ValueError(f'Baris {excel_row}: jawaban bukan teks.')
            rows.append({'row_id': f'row_{excel_row-1:04d}', 'source_row': excel_row,
                         'text': clean(raw[ANSWER_COLUMN]), 'raw_text': raw[ANSWER_COLUMN],
                         'email_group': unicodedata.normalize('NFC', str(raw[GROUP_COLUMN])).strip().casefold(),
                         'question': clean(raw[QUESTION_COLUMN]), 'reference': clean(raw[REFERENCE_COLUMN]),
                         'response_number': str(raw[RESPONSE_NUMBER_COLUMN]), 'score': int(score)})
    finally:
        book.close()
    if cfg.get('expected_rows') is not None and len(rows) != cfg['expected_rows']:
        raise ValueError(f"Diharapkan {cfg['expected_rows']} baris, ditemukan {len(rows)}; tidak dipotong otomatis.")
    if not rows:
        raise ValueError('Dataset kosong')
    parent = list(range(len(rows)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(a, b):
        parent[find(b)] = find(a)

    email_first, duplicate_sets = {}, {}
    for i, row in enumerate(rows):
        email = row['email_group']
        if email in email_first:
            union(email_first[email], i)
        email_first[email] = i
        # Normalisasi ini HANYA audit kesamaan, bukan teks masukan encoder.
        normalized = ' '.join(row['text'].casefold().split())
        key = hashlib.sha256(normalized.encode()).hexdigest()
        duplicate_sets.setdefault(key, []).append(i)
    for indices in duplicate_sets.values():
        for i in indices[1:]:
            union(indices[0], i)
    roots = {root: i for i, root in enumerate(sorted({find(i) for i in range(len(rows))}))}
    groups = np.asarray([roots[find(i)] for i in range(len(rows))], dtype=int)
    y = np.asarray([r['score'] for r in rows], dtype=float)
    summary = {'scope': SCOPE, 'rows': len(rows), 'target_columns': [TARGET_COLUMN], 'model_outputs': 1,
               'score_range_observed': [float(y.min()), float(y.max())],
               'score_range_declared': [cfg['score_min'], cfg['score_max']],
               'scale_basis': '0–100 mengikuti notebook unggahan; deskriptor rubrik guru tidak tersedia.',
               'email_groups': len(email_first), 'connected_groups': len(roots),
               'questions': len({r['question'] for r in rows}),
               'duplicate_sets': sum(len(v) > 1 for v in duplicate_sets.values()),
               'cross_email_duplicate_sets': sum(len({rows[i]['email_group'] for i in v}) > 1 for v in duplicate_sets.values()),
               'source_sha256': source_hash, 'four_trait_labels_available': False,
               'four_rubric_report': 'Arsitektur empat output memungkinkan, tetapi empat skor tidak dapat ditentukan dari satu skor guru. Perlu anotasi manusia per dimensi; skor tidak disalin/dipecah otomatis.',
               'input_features': list(INPUT_COLUMNS),
               'grouping_note': 'Email dipakai sebagai kelompok pencegah kebocoran; tidak membuktikan identitas terverifikasi. Duplikat lintas email disatukan. Semua skenario memakai soal/referensi dan jawaban. Email hanya kelompok; nilai guru hanya target, bukan fitur encoder.'}
    return Bundle(rows, y, groups, source_hash, summary)

# Identitas hasil mengikuti data, konfigurasi, dan kode notebook ini.
def run_folder(cfg, bundle):
    code_hash = NOTEBOOK_CODE_SHA256
    semantic = {k: v for k, v in cfg.items() if k not in ('dataset', 'output', 'trace_position', 'vector_source', 'new_answer', 'focus_word', 'new_question', 'new_reference')}
    key = hashlib.sha256(json.dumps({'source': bundle.source_sha256, 'config': semantic, 'code': code_hash}, sort_keys=True).encode()).hexdigest()[:16]
    folder = Path(cfg['output']) / key
    folder.mkdir(parents=True, exist_ok=True)
    dump(folder / 'effective_config.json', cfg)
    return folder


## 10 · Pencatatan Excel

Setiap tahap mencatat sumber dan hasil aktual. Ini pencatatan, bukan training tambahan.


### `excel_value`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def excel_value(value):
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, ensure_ascii=False, default=str)
    if isinstance(value, float) and not math.isfinite(value):
        return None  # Tidak tercakup/undefined: tetap kosong, bukan nol.
    if value is pd.NA:
        return None
    return value


### `write_excel`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def write_excel(path, sheets, metadata):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    book = Workbook(write_only=True)
    used = set()
    counts = {}
    all_sheets = {'Info': pd.DataFrame([{'field': k, 'value': v} for k, v in metadata.items()]), **sheets}
    for name, frame in all_sheets.items():
        frame = frame if isinstance(frame, pd.DataFrame) else pd.DataFrame(frame)
        if len(frame.columns) > 16384:
            raise ValueError('Terlalu banyak kolom untuk Excel; tidak dipotong diam-diam.')
        headers = [str(x) for x in frame.columns]
        chunks = max(1, math.ceil(len(frame)/1048575))
        for part in range(chunks):
            base = re.sub(r'[\\/*?:\[\]]', '_', name)
            suffix = f'_{part+1}' if chunks > 1 else ''
            title = base[:31-len(suffix)]+suffix
            if title in used:
                raise ValueError('Nama sheet bertabrakan setelah dipendekkan: '+title)
            used.add(title)
            sheet = book.create_sheet(title)
            sheet.freeze_panes = 'B2'
            heading = []
            for text in headers:
                cell = WriteOnlyCell(sheet, value=text)
                cell.font = Font(bold=True, color='FFFFFF')
                cell.fill = PatternFill('solid', fgColor='172035')
                cell.data_type = 's'
                heading.append(cell)
            sheet.append(heading)
            subset = frame.iloc[part*1048575:(part+1)*1048575]
            for row in subset.itertuples(index=False, name=None):
                values = []
                for value in row:
                    value = excel_value(value)
                    if isinstance(value, str):
                        if len(value) > 32767:
                            raise ValueError('Teks melebihi batas satu sel Excel; ekspor dihentikan agar tidak terpotong.')
                        cell = WriteOnlyCell(sheet, value=value)
                        cell.data_type = 's'  # Teks siswa '=...' tidak dijalankan sebagai formula.
                        values.append(cell)
                    else:
                        values.append(value)
                sheet.append(values)
            counts[title] = {'rows': len(subset), 'columns': len(headers)}
    temporary = path.with_name(path.stem+'.pending.xlsx')
    book.save(temporary)
    temporary.replace(path)
    return counts


### `vector_frame`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def vector_frame(values, ids, prefix='v'):
    values = np.asarray(values)
    if values.ndim != 2 or len(values) != len(ids):
        raise ValueError('Vektor perlu [N,D] dan ID berjumlah N.')
    frame = pd.DataFrame(values, columns=[f'{prefix}_{i:04d}' for i in range(values.shape[1])])
    frame.insert(0, 'row_id', list(ids))
    return frame


### `ExcelJournal`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
class ExcelJournal:
    def __init__(self, folder, metadata, verbose=False):
        self.folder = Path(folder)
        self.folder.mkdir(parents=True, exist_ok=True)
        self.metadata = metadata
        self.verbose = verbose
        file = self.folder/'index.json'
        self.records = json.loads(file.read_text()) if file.exists() else {}

    def save(self, name, sheets, stage, **extra):
        meta = {**self.metadata, 'stage': stage, 'status': 'ACTUAL EXECUTION', **extra}
        path = self.folder/(name+'.xlsx')
        counts = write_excel(path, sheets, meta)
        self.records[name] = {'file': path.name, 'stage': stage, 'status': meta['status'],
                              'sheets_and_shapes': counts, 'source': extra.get('source', '')}
        (self.folder/'index.json').write_text(json.dumps(self.records, ensure_ascii=False, indent=2)+'\n')
        write_excel(self.folder/'00_indeks_output.xlsx', {'Daftar_file': pd.DataFrame(self.records.values())},
                    {**self.metadata, 'stage': 'Daftar keluaran aktual; hanya tahap yang selesai dicatat'})
        if self.verbose:
            print(f'Rincian tersimpan: {path.name}', flush=True)
        return path


<a id="xy-aktual"></a>
## 11 · Variabel X dan y yang benar-benar dibaca

`X` menampilkan ketiga fitur teks setelah cleaning. `y` menampilkan nilai guru. Fungsi training membawa data yang sama di dalam `bundle`, bersama ID dan kelompok split: `bundle.texts` berisi kolom jawaban, `contexts_from_bundle(bundle)` berisi soal/referensi, dan `bundle.y` adalah target. Tidak ada variabel di worker lain.


In [ ]:
bundle=read_dataset(cfg)  # Periksa enam header, label, duplikat dan kelompok email.
output=run_folder(cfg,bundle)
journal=ExcelJournal(output/'excel',{'scope':SCOPE,'dataset_sha256':bundle.source_sha256,
                     'run_folder':str(output),'notebook_code_sha256':NOTEBOOK_CODE_SHA256,
                     'privacy':'PRIVAT; bukan untuk repo publik'})
journal.save('01_konfigurasi',{'Parameter':pd.DataFrame([{'name':k,'value':v} for k,v in cfg.items()]),
                             'Versi':pd.DataFrame([{'library':p,'version':version(p)} for p in ('torch','transformers','numpy','pandas','scikit-learn','openpyxl')])},
             'Konfigurasi aktual; belum ada hasil training')
df=pd.read_excel(cfg['dataset'],sheet_name=cfg['sheet'])
df=df.loc[:,~df.columns.astype(str).str.startswith('Unnamed:')]
cleaning=pd.DataFrame([{'row_id':r['row_id'],'teks_asli':r['raw_text'],'teks_model':r['text'],
                       'soal':r['question'],'jawaban_referensi':r['reference'],
                       'berubah':r['raw_text']!=r['text'],'skor_guru':r['score'],
                       'target_loss':(r['score']-cfg['score_min'])/(cfg['score_max']-cfg['score_min'])} for r in bundle.rows])
journal.save('02_dataset_dan_cleaning',{'Dataset_asli':df,'Audit':pd.DataFrame([bundle.summary]),
                                      'Sebelum_sesudah':cleaning},'Pembacaan/cleaning aktual; sumber utuh')
show(cleaning[['row_id','teks_model','skor_guru']],'Contoh jawaban dan nilai guru')
print(len(df),'jawaban tersedia. Setiap jawaban memiliki satu nilai guru; setiap model menghasilkan satu prediksi skor.')


# INILAH X DAN Y — satu baris X dan satu nilai y untuk setiap jawaban.
X = pd.DataFrame({
    QUESTION_COLUMN: [row['question'] for row in bundle.rows],
    REFERENCE_COLUMN: [row['reference'] for row in bundle.rows],
    ANSWER_COLUMN: [row['text'] for row in bundle.rows],
}, columns=INPUT_COLUMNS)
y = pd.Series(bundle.y, index=X.index, name=TARGET_COLUMN)
assert TARGET_COLUMN not in X.columns and len(X) == len(y)
print('X:', X.shape, '| y:', y.shape)
display(X.head(3))
display(y.head(3).to_frame())


## 12 · Pisahkan development/test dan lima fold

Fold hanya pada development. Email/duplikat tidak menyeberang. Pemilihan split berdasarkan jumlah baris, tanpa membaca label. Test tidak memilih LR atau epoch.


### `make_split`

Menghasilkan daftar indeks development, test dan5pasangan train/validation yang mengikuti metode E30 untuk S3.


In [ ]:
def make_split(bundle, cfg):
    indices = np.arange(len(bundle.rows))
    splitter = GroupShuffleSplit(n_splits=cfg['split_candidates'], test_size=cfg['test_fraction'], random_state=cfg['split_seed'])
    # Pilih kedekatan JUMLAH BARIS dengan target 80:20, bukan skor/hasil model.
    # GroupShuffleSplit membagi kelompok; ukuran kelompok tidak selalu sama.
    candidates = list(splitter.split(indices, groups=bundle.groups))
    development, test = min(candidates, key=lambda pair: abs(len(pair[1])/len(indices)-cfg['test_fraction']))
    if len(set(bundle.groups[development])) < cfg['folds']:
        raise ValueError('Kelompok training tidak cukup untuk CV yang diminta.')
    folds = []
    for number, (a, b) in enumerate(GroupKFold(n_splits=cfg['folds']).split(development, groups=bundle.groups[development]), 1):
        training, validation = development[a], development[b]
        assert not set(bundle.groups[training]) & set(bundle.groups[validation])
        folds.append({'fold': number, 'train': training.tolist(), 'validation': validation.tolist()})
    assert not set(bundle.groups[development]) & set(bundle.groups[test])
    manifest = {'scope': SCOPE, 'source_sha256': bundle.source_sha256,
                'development': development.tolist(), 'test': test.tolist(), 'folds': folds,
                'development_rows': len(development), 'test_rows': len(test),
                'development_groups': len(set(bundle.groups[development])),
                'test_groups': len(set(bundle.groups[test])),
                'target_test_row_fraction': cfg['test_fraction'],
                'candidate_group_fraction': cfg['test_fraction'],
                'split_candidates': cfg['split_candidates'],
                'selection_uses_labels': False,
                'actual_test_row_fraction': len(test)/len(indices),
                'policy': 'Holdout kelompok: pilih kandidat deterministik terdekat target proporsi baris berdasarkan ukuran kelompok saja, tanpa label/hasil model. CV hanya di development. Email/duplikat tidak menyeberang. Mengukur kelompok baru pada pertanyaan ini, bukan unseen-prompt.',
                'test_used_for_selection': False}
    manifest['split_sha256'] = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
    return manifest


### `contexts_from_bundle`

Mengambil pasangan soal/referensi dari data X; tidak membaca nilai guru.


In [ ]:
def contexts_from_bundle(bundle):
    """Ambil soal/referensi tanpa membaca target skor guru."""
    contexts = []
    for row in bundle.rows:
        question, reference = row.get('question'), row.get('reference')
        if not isinstance(question, str) or not question.strip():
            raise ValueError('Context pair memerlukan soal yang tidak kosong.')
        if not isinstance(reference, str) or not reference.strip():
            raise ValueError('Context pair memerlukan jawaban referensi yang tidak kosong.')
        contexts.append((clean(question), clean(reference)))
    return contexts


In [ ]:
manifest=make_split(bundle,cfg)

dump(output/'split_manifest.json',manifest)

roles={i:'development' for i in manifest['development']}

roles.update({i:'test' for i in manifest['test']})

split_rows=pd.DataFrame([{'row_id':r['row_id'],'role':roles[i],'connected_group':int(bundle.groups[i])} for i,r in enumerate(bundle.rows)])

fold_rows=pd.DataFrame([{'fold':f['fold'],'row_id':bundle.rows[i]['row_id'],'role':role} for f in manifest['folds'] for role,key in [('train','train'),('validation','validation')] for i in f[key]])

journal.save('03_split_dan_cv',{'Split_setiap_baris':split_rows,'Fold_setiap_baris':fold_rows,
                              'Ringkasan':pd.DataFrame([{k:v for k,v in manifest.items() if k not in ('folds','development','test')}])},'Split dan 5 fold S3 sama dengan metode E30')

show(split_rows.groupby('role').size().rename('rows').reset_index(),'Pembagian aktual')

trace_index=manifest['test'][cfg['trace_position']]

trace_text=cfg.get('new_answer','').strip() or bundle.rows[trace_index]['text']

print('Jawaban contoh:',friendly_value('row_id',bundle.rows[trace_index]['row_id']),'. Contoh ini dipakai untuk melihat token, vektor, dan prediksi setelah model dipilih.')

trace_context=(clean(cfg['new_question']),clean(cfg['new_reference'])) if cfg.get('new_answer','').strip() else contexts_from_bundle(bundle)[trace_index]


## 13 · Tokenisasi, window dan posisi input

Tiga teks disusun dalam single_sequence. Posisi respons dan referensi ditandai terpisah. Mean antar-window tetap differentiable untuk jalur RoBERTa yang dilatih.


### `content_windows`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def content_windows(text, tokenizer, maximum=512, overlap=128):
    capacity = maximum - tokenizer.num_special_tokens_to_add(pair=False)
    if not 0 <= overlap < capacity: raise ValueError('Overlap harus lebih kecil dari content capacity')
    # Tokenizer mengubah teks menjadi ID subword, bukan menghitung skor.
    ids = tokenizer.encode(text, add_special_tokens=False)
    starts = [0]
    while starts[-1]+capacity < len(ids): starts.append(starts[-1]+capacity-overlap)
    chunks=[]
    for start in starts:
        content=ids[start:start+capacity]
        prepared=tokenizer.prepare_for_model(content, add_special_tokens=True, truncation=False,
                                             return_attention_mask=True, return_token_type_ids=False)
        assert len(prepared['input_ids']) <= maximum
        chunks.append(dict(prepared))
    return chunks, {'content_tokens':len(ids),'capacity':capacity,'stride':capacity-overlap,
                    'starts':starts,'windows':len(chunks)}


### `batch_windows`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def batch_windows(texts, tokenizer, device='cpu', maximum=512, overlap=128):
    chunks=[]; mapping=[]; stats=[]
    for i,text in enumerate(texts):
        c,s=content_windows(text,tokenizer,maximum,overlap); chunks.extend(c); mapping.extend([i]*len(c));stats.append(s)
    batch=tokenizer.pad(chunks, padding=True, return_tensors='pt')
    return {k:v.to(device) for k,v in batch.items()}, torch.tensor(mapping,device=device), stats


### `mean_by_essay`

Merata-ratakan representasi antar-window milik jawaban yang sama; ini mean pooling antar-window, berbeda dari masked mean pooling antartoken setelah matching.


In [ ]:
def mean_by_essay(cls, mapping, n):
    sums=cls.new_zeros((n,cls.shape[-1])).index_add(0,mapping,cls)
    counts=cls.new_zeros(n).index_add(0,mapping,cls.new_ones(len(mapping)))
    if (counts==0).any(): raise ValueError('Esai tanpa window')
    return sums / counts[:,None]


### `context_token_ids`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def context_token_ids(context, tokenizer, cfg):
    """Sisakan anggaran untuk soal dan referensi sekaligus."""
    if not isinstance(context, (tuple, list)) or len(context) != 2:
        raise ValueError('Konteks harus pasangan (soal, jawaban referensi).')
    question, reference = context
    if not all(isinstance(t, str) and t.strip() for t in context):
        raise ValueError('Isi soal dan jawaban referensi sebelum memprediksi.')
    limit, qlimit = cfg['context_max_tokens'], cfg['question_max_tokens']
    if not 1 <= qlimit < limit:
        raise ValueError('Anggaran token soal harus lebih kecil daripada konteks.')
    # Prefix menjadi bagian input; nilai guru tidak pernah dimasukkan.
    qids = tokenizer.encode('Soal: '+question, add_special_tokens=False)
    rids = tokenizer.encode('\nJawaban referensi: '+reference, add_special_tokens=False)
    question_kept = qids[:qlimit]
    reference_kept = rids[:limit-len(question_kept)]
    return question_kept+reference_kept, {
        'question_tokens': len(qids), 'question_kept': len(question_kept),
        'reference_tokens': len(rids), 'reference_kept': len(reference_kept),
        'question_truncated': len(question_kept) < len(qids),
        'reference_truncated': len(reference_kept) < len(rids),
        'context_tokens_kept': len(question_kept)+len(reference_kept),
    }


### `reference_context_positions`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def reference_context_positions(context, tok, cfg, info):
    """Posisi isi referensi dari offset karakter; ID token berulang tidak ditebak."""
    prefix = '\nJawaban referensi: '
    text = prefix + context[1]
    try:
        aligned = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    except NotImplementedError:
        # RoBERTa training tetap slow seperti E26; fast hanya membaca offset.
        if not hasattr(tok, '_aes_reference_fast'):
            spec = cfg['roberta']
            tok._aes_reference_fast = AutoTokenizer.from_pretrained(
                spec['checkpoint'], revision=spec['revision'], use_fast=True)
        aligned = tok._aes_reference_fast(text, add_special_tokens=False, return_offsets_mapping=True)
    original = tok.encode(text, add_special_tokens=False)
    if list(aligned['input_ids']) != original or len(aligned['offset_mapping']) != len(original):
        raise ValueError('ID/offset tokenizer referensi tidak identik; posisi tidak ditebak.')
    positions = []
    for index, (start, end) in enumerate(aligned['offset_mapping'][:info['reference_kept']]):
        if not 0 <= start <= end <= len(text):
            raise ValueError('Offset tokenizer referensi tidak sah.')
        # Token yang mencakup teks isi diperbolehkan; token prefix/spasi saja dikecualikan.
        if end > len(prefix) and text[max(start, len(prefix)):end].strip():
            positions.append(info['question_kept'] + index)
    if not positions:
        raise ValueError('Isi referensi habis oleh truncation/prefix. Tambah anggaran konteks sebelum training.')
    return positions


### `pair_content_windows`

Menyusun token soal/referensi/jawaban, menyimpan mask posisi asli dan memastikan seluruh token jawaban tercakup.


In [ ]:
def pair_content_windows(text, context, tokenizer, maximum=512, overlap=128, cfg=None):
    """Informasi dan window sama; E22 menambah separator resmi pasangan."""
    if cfg is None or cfg.get('input_format') not in ('single_sequence', 'context_pair'):
        raise ValueError('Pilih format input E21 atau E22 sebelum tokenisasi.')
    context_ids, context_info = context_token_ids(context, tokenizer, cfg)
    reference_positions = reference_context_positions(context, tokenizer, cfg, context_info)
    context_layout = [-1]*len(context_ids)
    for position in reference_positions:context_layout[position] = -4
    ids = tokenizer.encode(text, add_special_tokens=False)
    answer_prefix = tokenizer.encode('\nJawaban siswa:\n', add_special_tokens=False)
    if not ids:
        raise ValueError('Jawaban tidak menghasilkan token.')
    paired = cfg['input_format'] == 'context_pair'
    specials = tokenizer.num_special_tokens_to_add(pair=paired)
    # Cadangkan overhead terbesar untuk KEDUA eksperimen. Starts/cakupan sama.
    reserved = max(tokenizer.num_special_tokens_to_add(pair=False),
                   tokenizer.num_special_tokens_to_add(pair=True))
    capacity = maximum-len(context_ids)-len(answer_prefix)-reserved
    if not 0 <= overlap < capacity:
        raise ValueError('Konteks dan penanda jawaban menyisakan kapasitas di bawah overlap.')
    starts = [0]
    while starts[-1]+capacity < len(ids):
        starts.append(starts[-1]+capacity-overlap)
    chunks, answer_positions, reference_window_positions, segments = [], [], [], []
    for start in starts:
        answer = ids[start:start+capacity]
        second = answer_prefix+answer
        if paired:
            prepared = tokenizer.build_inputs_with_special_tokens(context_ids, second)
            layout = tokenizer.build_inputs_with_special_tokens(context_layout,
                     [-3]*len(answer_prefix)+[-2]*len(answer))
        else:
            prepared = tokenizer.build_inputs_with_special_tokens(context_ids+second)
            layout = tokenizer.build_inputs_with_special_tokens(
                     context_layout+[-3]*len(answer_prefix)+[-2]*len(answer))
        positions = [i for i, value in enumerate(layout) if value == -2]
        if (len(prepared) != len(layout) or len(prepared) > maximum
                or len(prepared)-len(context_ids)-len(second) != specials
                or [prepared[i] for i in positions] != answer):
            raise ValueError('Template tokenizer tidak konsisten dengan format input.')
        chunks.append({'input_ids':prepared, 'attention_mask':[1]*len(prepared)})
        answer_positions.append(positions)
        reference_window_positions.append([i for i, value in enumerate(layout) if value == -4])
        segments.append(['Soal/referensi' if v == -1 else 'Jawaban siswa' if v == -2
                         else 'Penanda jawaban' if v == -3 else 'Isi referensi' if v == -4 else 'Token khusus' for v in layout])
    stats = {**context_info, 'input_format':cfg['input_format'], 'content_tokens':len(ids),
             'capacity':capacity, 'stride':capacity-overlap, 'starts':starts,
             'windows':len(chunks), 'pair_special_tokens':specials,
             'reserved_special_tokens':reserved, 'answer_prefix_tokens':len(answer_prefix),
             'answer_positions':answer_positions, 'reference_positions':reference_window_positions,
             'reference_content_kept':len(reference_positions), 'segments':segments, 'answer_tokens_uncovered':0}
    return chunks, stats


### `pair_batch_windows`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def pair_batch_windows(texts, contexts, tokenizer, device='cpu', maximum=512, overlap=128, cfg=None):
    """Susun batch pasangan tanpa menukar konteks antarjawaban."""
    if len(texts) != len(contexts) or not texts:
        raise ValueError('Jumlah konteks harus sama dengan jumlah jawaban.')
    chunks, mapping, stats = [], [], []
    for index, (text, context) in enumerate(zip(texts, contexts)):
        values, detail = pair_content_windows(text, context, tokenizer, maximum, overlap, cfg)
        chunks.extend(values)
        mapping.extend([index]*len(values))
        stats.append(detail)
    batch = tokenizer.pad(chunks, padding=True, return_tensors='pt')
    return {key: value.to(device) for key, value in batch.items()}, torch.tensor(mapping, device=device), stats


### `pair_token_frames`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def pair_token_frames(bundle, cfg, tok, role='roberta'):
    rows, coverage = [], []
    maximum, overlap = cfg[role+'_max_length'], cfg[role+'_overlap']
    for row, text, context in zip(bundle.rows, bundle.texts, contexts_from_bundle(bundle)):
        batch, _, details = pair_batch_windows([text], [context], tok, 'cpu', maximum, overlap, cfg)
        stats = details[0]
        coverage.append({'row_id': row['row_id'], 'model': role+' tiga input',
                         **{k: v for k, v in stats.items() if k not in ('answer_positions','reference_positions','segments')}})
        for window, (ids, masks) in enumerate(zip(batch['input_ids'].tolist(), batch['attention_mask'].tolist())):
            segments = list(stats['segments'][window])
            padding = ['Padding']*(len(ids)-len(segments))
            segments = padding+segments if tok.padding_side=='left' else segments+padding
            for position, (identifier, piece, mask, segment) in enumerate(zip(ids, tok.convert_ids_to_tokens(ids), masks, segments)):
                rows.append({'row_id':row['row_id'], 'window':window, 'position':position,
                             'token_id':identifier, 'token':piece, 'attention_mask':mask,
                             'input_segment':segment})
    return pd.DataFrame(rows), coverage


### `paired_token_vectors`

Inspeksi embedding lookup dan hidden state Transformer. Mode tanpa gradien, tidak memperbarui bobot.


In [ ]:
def paired_token_vectors(text, model, slow, fast, cfg, device, context):
    """Vektor kata jawaban memakai konteks persis jalur training."""
    aligned = fast(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = slow.encode(text, add_special_tokens=False)
    if list(aligned['input_ids']) != ids:
        raise ValueError('ID fast/slow jawaban berbeda; offset tidak ditebak.')
    offsets = [tuple(value) for value in aligned['offset_mapping']]
    if len(offsets) != len(ids):
        raise ValueError('Offset jawaban tidak cocok dengan jumlah token.')
    chunks, stats = pair_content_windows(text, context, slow, cfg['roberta_max_length'], cfg['roberta_overlap'], cfg)
    dim = model.config.hidden_size
    sums = np.zeros((len(ids), dim), dtype=np.float32)
    counts = np.zeros(len(ids), dtype=np.int64)
    initial = np.full_like(sums, np.nan)
    cls_vectors = []
    model.eval()
    with torch.no_grad():
        for chunk, start, positions in zip(chunks, stats['starts'], stats['answer_positions']):
            tensor = torch.tensor([chunk['input_ids']], dtype=torch.long, device=device)
            hidden = model(input_ids=tensor, attention_mask=torch.ones_like(tensor)).last_hidden_state
            stop = start+len(positions)
            sums[start:stop] += hidden[0, positions].float().cpu().numpy()
            counts[start:stop] += 1
            initial[start:stop] = model.get_input_embeddings()(tensor[:, positions])[0].float().cpu().numpy()
            cls_vectors.append(hidden[0, 0].float().cpu().numpy())
    contextual = sums/counts[:, None]
    return {'ids': ids, 'offsets': offsets, 'initial': initial, 'contextual': contextual.astype(np.float32),
            'counts': counts, 'starts': stats['starts'], 'capacity': stats['capacity'],
            'cls': np.mean(cls_vectors, axis=0)}


### `paired_pretrained_values`

Membuat fitur dari encoder pada tiga teks. BGE memakai rata-rata CLS window lalu normalisasi L2.


In [ ]:
def paired_pretrained_values(bundle, cfg, role):
    """Mean CLS antar-window; BGE dinormalisasi L2 setelah agregasi."""
    device = device_name(cfg)
    model, tok = encoder(cfg,role).to(device), tokenizer(cfg,role)
    model.eval()
    vectors, window_vectors, ids, coverage = [], [], [], []
    with torch.no_grad():
        for index, (text,context) in enumerate(zip(bundle.texts,contexts_from_bundle(bundle))):
            windows,mapping,details = pair_batch_windows([text],[context],tok,device,
                cfg[role+'_max_length'],cfg[role+'_overlap'],cfg)
            # Satu jawaban diproses bertahap agar jumlah window tidak meledakkan memori.
            cls_parts=[]
            for start in range(0,len(mapping),cfg['bge_batch']):
                part={k:v[start:start+cfg['bge_batch']] for k,v in windows.items()}
                cls_parts.append(model(**part).last_hidden_state[:,0,:].float())
            cls=torch.cat(cls_parts,dim=0)
            value=mean_by_essay(cls,mapping,1)
            if role=='bge':value=torch.nn.functional.normalize(value,dim=1)
            vectors.append(value.cpu().numpy())
            window_vectors.append(cls.cpu().numpy())
            row_id=bundle.rows[index].get('row_id',f'inference_{index}')
            ids.extend(f'{row_id}_window_{j}' for j in range(len(cls)))
            coverage.append({k:v for k,v in details[0].items() if k not in ('answer_positions','reference_positions','segments')})
            if (index+1)%25==0 or index+1==len(bundle.rows):
                print(f'{role}: tiga input {index+1}/{len(bundle.rows)} jawaban selesai.',flush=True)
    values=np.concatenate(vectors).astype(np.float32)
    if role=='bge' and any(p.requires_grad for p in model.parameters()):
        raise ValueError('Encoder BGE harus frozen.')
    del model,tok
    release()
    return values,np.concatenate(window_vectors),ids,coverage


## 14 · Memuat tokenizer dan encoder pretrained

Nama checkpoint berasal dari bagian3. `encoder()` memuat objek model; `tokenizer()` memuat pemecah teks. Fungsi loader ini tidak langsung menjalankan supervised training.


### `device_name`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def device_name(cfg):
    requested = cfg['device']
    if requested == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA diminta tetapi tidak tersedia; pilih GPU di Colab atau device=cpu.')
    return ('cuda' if torch.cuda.is_available() else 'cpu') if requested == 'auto' else requested


### `release`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def release():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


### `ExpectedPoolerNotice`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
class ExpectedPoolerNotice(logging.Filter):
    """Hanya saring pemberitahuan dua bobot pooler yang memang tidak dipakai."""
    def filter(self, record):
        import ast
        import re
        message = record.getMessage()
        if not message.startswith('Some weights of the model checkpoint at ') or 'were not used when initializing' not in message:
            return True
        match = re.search(r"\['[^\n]+?\]", message)
        if match is None:
            return True
        try:
            keys = ast.literal_eval(match.group())
        except (ValueError, SyntaxError):
            return True
        return set(keys) != {'pooler.dense.bias', 'pooler.dense.weight'}


### `encoder`

AutoModel.from_pretrained memuat bobot yang dikunci revision. RoBERTa trainable; BGE frozen. Dimensi768/1024 diperiksa.


In [ ]:
def encoder(cfg, role):
    # Ambil nama checkpoint dan revision model yang dipilih: roberta atau bge.
    spec = cfg[role]
    logger = logging.getLogger('transformers.modeling_utils')
    notice = ExpectedPoolerNotice()
    logger.addFilter(notice)
    try:
        # Muat arsitektur dan bobot pretrained. Ini baru inisialisasi encoder, belum latihan memakai nilai guru.
        result = AutoModel.from_pretrained(spec['checkpoint'], revision=spec['revision'], add_pooling_layer=False)
    finally:
        logger.removeFilter(notice)
    # Periksa panjang vektor: RoBERTa 768 angka, BGE-M3 1024 angka.
    expected = 768 if role == 'roberta' else 1024
    if result.config.hidden_size != expected:
        raise ValueError(f'{role}: dimensi checkpoint tidak cocok dengan {expected}.')
    # Parameter RoBERTa ikut diperbarui; parameter BGE tidak ikut diperbarui.
    # RoBERTa boleh belajar; bobot encoder BGE tidak diperbarui.
    result.requires_grad_(role == 'roberta')
    return result


### `tokenizer`

AutoTokenizer.from_pretrained memuat tokenizer milik checkpoint yang sama; skor guru tidak ditokenisasi.


In [ ]:
def tokenizer(cfg, role, fast=False):
    # Muat tokenizer dari checkpoint yang sama agar ID token cocok dengan kosakata model.
    result = AutoTokenizer.from_pretrained(cfg[role]['checkpoint'], revision=cfg[role]['revision'],
                                           use_fast=fast if role == 'roberta' else True)
    if role == 'roberta' and not fast and result.num_special_tokens_to_add(False) != 2:
        raise ValueError('RoBERTa slow tokenizer harus menambahkan dua special token.')
    return result


## 15 · Ekspor token dan cakupan konteks

Periksa token dari tiga input, token yang terpotong, dan posisi yang dipakai pooling.


### `export_tokens`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def export_tokens(bundle, cfg, journal):
    roberta, rinfo = pair_token_frames(bundle,cfg,tokenizer(cfg,'roberta'),'roberta')
    bge, binfo = pair_token_frames(bundle,cfg,tokenizer(cfg,'bge'),'bge')
    journal.save('04_tokenisasi', {'RoBERTa_input':roberta, 'BGE_input':bge,
                 'Panjang_dan_coverage':pd.DataFrame(rinfo+binfo)},
                 'Token aktual tiga input seluruh skenario; lihat kolom input_format pada cakupan',
                 source={'roberta':cfg['roberta'],'bge':cfg['bge']},
                 input_features=['soal','response_text','jawaban_referensi'])
    return roberta,bge


In [ ]:
if RUN_MODELS:
    token_roberta,token_bge=export_tokens(bundle,cfg,journal)
    show(token_roberta,'Token RoBERTa');show(token_bge,'Token BGE-M3')
else:
    print('Pemeriksaan data saja: tokenisasi dan model belum dijalankan.')


## 16 · Vektor RoBERTa sebelum supervised training

Bahan pembanding fitur awal/akhir. Angka ini bukan prediksi penelitian sebelum model dilatih.


### `roberta_pretrained_vectors`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def roberta_pretrained_vectors(bundle, cfg, journal):
    values,windows,window_ids,coverage=paired_pretrained_values(bundle,cfg,'roberta')
    journal.save('05_roberta_pretrained',
                 {'Vektor_jawaban_768':vector_frame(values,[r['row_id'] for r in bundle.rows]),
                  'CLS_tiap_window':vector_frame(windows,window_ids)},
                 'RoBERTa tiga input SEBELUM fine-tuning; input tiga teks yang sama dengan S3; format mengikuti konfigurasi',
                 source=cfg['roberta'],pooling='mean window CLS; soal/referensi diulang per window')
    return values


In [ ]:
if RUN_MODELS:
    roberta_before=roberta_pretrained_vectors(bundle,cfg,journal)
    print('RoBERTa: setiap jawaban diubah menjadi vektor berisi 768 angka, sebelum pelatihan dengan nilai guru.')
    show(vector_frame(roberta_before,[r['row_id'] for r in bundle.rows]),'Vektor jawaban RoBERTa sebelum pelatihan')


## 17 · Fitur BGE-M3 frozen

Fitur1024 disimpan dalam cache berdasarkan teks/konteks/resep tokenizer, tanpa target. S3 memakai fitur BGE langsung tanpa menjalankan S2; layer setelah BGE tetap dilatih.


### `bge_features`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def bge_features(bundle, cfg, output):
    """Cache berubah jika soal, referensi, jawaban, atau resep token berubah."""
    contexts=contexts_from_bundle(bundle)
    contract={'model':cfg['bge'], 'texts':bundle.texts, 'contexts':contexts,
              'maximum':cfg['bge_max_length'],'overlap':cfg['bge_overlap'],
              'context_max_tokens':cfg['context_max_tokens'],'question_max_tokens':cfg['question_max_tokens'],
              'policy':'three_inputs_mean_CLS_then_L2_v2', 'input_format':cfg['input_format']}
    key=hashlib.sha256(json.dumps(contract,sort_keys=True,ensure_ascii=False).encode()).hexdigest()[:20]
    cache=Path(cfg['output'])/'cache'/f'bge_three_inputs_{key}.npz'
    metadata=cache.with_suffix('.json')
    if cache.exists() and metadata.exists():
        proof=json.loads(metadata.read_text())
        if proof['file_sha256']!=hashlib.sha256(cache.read_bytes()).hexdigest():
            raise ValueError('Cache BGE berubah; tidak dipakai.')
        with np.load(cache,allow_pickle=False) as data:values=data['features']
        reused=True
    else:
        values,_,_,coverage=paired_pretrained_values(bundle,cfg,'bge')
        cache.parent.mkdir(parents=True,exist_ok=True)
        np.savez_compressed(cache,features=values)
        proof={'model':cfg['bge'],'shape':list(values.shape),'frozen':True,
               'pooling':'mean paired-window CLS then L2','max_length':cfg['bge_max_length'],
               'feature_source':'soal + jawaban_referensi + response_text; no teacher score', 'input_format':cfg['input_format'],
               'answer_tokens_uncovered':sum(s['answer_tokens_uncovered'] for s in coverage),
               'context_truncated_rows':sum(s['question_truncated'] or s['reference_truncated'] for s in coverage),
               'file_sha256':hashlib.sha256(cache.read_bytes()).hexdigest()}
        dump(metadata,proof)
        reused=False
    if values.shape!=(len(bundle.rows),1024) or not np.isfinite(values).all() or not np.allclose(np.linalg.norm(values,axis=1),1,atol=1e-4):
        raise ValueError('Fitur BGE shape/normalisasi tidak sah.')
    dump(Path(output)/'bge_encoding.json',{**proof,'reused':reused})
    return torch.from_numpy(values)


In [ ]:
if RUN_MODELS:
    bge=bge_features(bundle,cfg,output)
    journal.save('06_bge_frozen',{'Vektor_jawaban_1024':vector_frame(bge.numpy(),[r['row_id'] for r in bundle.rows]),
                                'Encoding':pd.DataFrame([json.loads((output/'bge_encoding.json').read_text())])},
                 'BGE dense frozen; semua koordinat',source=cfg['bge'],pooling='CLS then L2')
    print('BGE-M3: setiap jawaban diubah menjadi vektor berisi 1024 angka. Bobot encoder tetap selama pelatihan.')
    show(vector_frame(bge.numpy(),[r['row_id'] for r in bundle.rows]),'Vektor jawaban BGE-M3')


## 18 · Inspeksi vektor kata dan kalimat

Tetap menyimpan seluruh komponen vektor, posisi token, dan pemeriksaan selisih komponen kata berulang. Hasil inspeksi bukan fitur tambahan ke model atau loss.


In [ ]:
WORD = re.compile(r"\b\w+(?:[-’']\w+)*\b", re.UNICODE)


### `sentence_spans`

Inspeksi memakai vektor aktual. Pemanggilan eksekusinya berada setelah semua fungsi bagian ini didefinisikan.


In [ ]:
def sentence_spans(text):
    """Segmentasi aturan sederhana, bukan model sintaksis; singkatan bisa terpisah."""
    result = []
    for match in re.finditer(r'[^.!?\n]+(?:[.!?]+|$)|[^.!?\n]+(?=\n)', text):
        start, end = match.span()
        while start < end and text[start].isspace():
            start += 1
        if start < end:
            result.append((start, end))
    return result


### `token_vectors`

Inspeksi memakai vektor aktual. Pemanggilan eksekusinya berada setelah semua fungsi bagian ini didefinisikan.


In [ ]:
def token_vectors(text, model, slow, fast, role, cfg, device):
    if cfg.get('pair_context') is None:
        raise ValueError('Vektor tiga input memerlukan soal dan jawaban referensi.')
    # Helper alignment yang sama dipakai untuk RoBERTa dan BGE; batas mengikuti peran.
    paired_cfg={**cfg,'roberta_max_length':cfg[role+'_max_length'],
                'roberta_overlap':cfg[role+'_overlap']}
    trace=paired_token_vectors(text,model,slow,fast,paired_cfg,device,cfg['pair_context'])
    if role=='bge':trace['cls']=torch.nn.functional.normalize(torch.from_numpy(trace['cls'])[None],dim=1)[0].numpy()
    return trace


### `word_vectors`

Inspeksi memakai vektor aktual. Pemanggilan eksekusinya berada setelah semua fungsi bagian ini didefinisikan.


In [ ]:
def word_vectors(text, trace):
    rows, initial, contextual = [], [], []
    for word in WORD.finditer(text):
        pieces = [i for i, (start, end) in enumerate(trace['offsets'])
                  if end > start and start < word.end() and end > word.start()]
        covered = bool(pieces) and all(trace['counts'][i] > 0 for i in pieces)
        dim = trace['initial'].shape[1]
        before = trace['initial'][pieces].mean(axis=0) if covered else np.full(dim, np.nan, dtype=np.float32)
        after = trace['contextual'][pieces].mean(axis=0) if covered else np.full(dim, np.nan, dtype=np.float32)
        rows.append({'word_index': len(rows), 'word': word.group(), 'start': word.start(), 'end': word.end(),
                     'token_indices': pieces, 'token_ids': [trace['ids'][i] for i in pieces],
                     'covered': covered, 'initial_first5': before[:5].tolist() if covered else None,
                     'contextual_first5': after[:5].tolist() if covered else None})
        initial.append(before)
        contextual.append(after)
    dim = trace['initial'].shape[1]
    return rows, np.asarray(initial, dtype=np.float32).reshape(-1, dim), np.asarray(contextual, dtype=np.float32).reshape(-1, dim)


### `compare_repetitions`

Inspeksi memakai vektor aktual. Pemanggilan eksekusinya berada setelah semua fungsi bagian ini didefinisikan.


In [ ]:
def compare_repetitions(rows, initial, contextual, focus=''):
    available = [r for r in rows if r['covered']]
    counts = Counter(r['word'].casefold() for r in available)
    target = focus.casefold().strip() if focus.strip() else next((w for w, n in counts.most_common() if n > 1), '')
    candidates = [r for r in available if r['word'].casefold() == target]
    results = []
    # Hindari tabel kuadratik pada kata sangat sering; semua vektor tetap disimpan.
    for left, right in itertools.islice(itertools.combinations(candidates, 2), 30):
        a, b = left['word_index'], right['word_index']
        results.append({'word': target, 'occurrence_indices': [a, b],
                        'same_token_ids': left['token_ids'] == right['token_ids'],
                        'initial_max_abs_difference': float(np.max(np.abs(initial[a]-initial[b]))),
                        'contextual_max_abs_difference': float(np.max(np.abs(contextual[a]-contextual[b]))),
                        'contextual_allclose_1e-6': bool(np.allclose(contextual[a], contextual[b], rtol=0, atol=1e-6))})
    return results


### `vector_lab`

Inspeksi memakai vektor aktual. Pemanggilan eksekusinya berada setelah semua fungsi bagian ini didefinisikan.


In [ ]:
def vector_lab(text, cfg, output, context=None):
    text = clean(text)
    if not text.strip():
        raise ValueError('Jawaban untuk analisis vektor kosong.')
    output = Path(output)
    destination = output/'vectors'/cfg['vector_source']
    destination.mkdir(parents=True, exist_ok=True)
    results = {}
    for role in ('roberta', 'bge'):
        source = cfg[role]['checkpoint']+'@'+cfg[role]['revision']
        if role == 'roberta' and cfg['vector_source'] not in ('pretrained','pretrained_pair'):
            report = json.loads((output/'final_report.json').read_text())
            checkpoint = output/cfg['vector_source']/'final.pt'
            if digest(checkpoint) != report['checkpoints'][cfg['vector_source']]:
                raise ValueError('Checkpoint vektor berbeda dari model final.')
            wrapper, _ = load_model(checkpoint, cfg)
            model = wrapper.roberta
            del wrapper
            source = cfg['vector_source']+' fine-tuned; sha256='+digest(checkpoint)
        else:
            model = encoder(cfg, role)
        device = device_name(cfg)
        model.to(device).eval()
        slow, fast = tokenizer(cfg, role), tokenizer(cfg, role, fast=True)
        paired=True
        if paired and context is None:
            raise ValueError('Semua vektor tiga input memerlukan soal dan referensi.')
        trace_cfg={**cfg,'pair_context':context} if paired else cfg
        trace = token_vectors(text, model, slow, fast, role, trace_cfg, device)
        words, before, after = word_vectors(text, trace)
        sentences, sentence_vectors = [], []
        for start, end in sentence_spans(text):
            part = token_vectors(text[start:end], model, slow, fast, role, trace_cfg, device)
            sentence_vectors.append(part['cls'])
            sentences.append({'sentence_index': len(sentences), 'text': text[start:end], 'start': start, 'end': end,
                              'covered_tokens': int(np.count_nonzero(part['counts'])), 'total_tokens': len(part['ids']),
                              'first5': part['cls'][:5].tolist()})
        vectors = np.asarray(sentence_vectors, dtype=np.float32).reshape(-1, model.config.hidden_size)
        np.savez_compressed(destination/f'{role}_vectors.npz', token_initial=trace['initial'],
                            token_contextual=trace['contextual'], token_ids=np.asarray(trace['ids']),
                            token_offsets=np.asarray(trace['offsets']).reshape(-1, 2), token_coverage=trace['counts'],
                            word_initial=before, word_contextual=after, sentence_cls=vectors)
        token_rows = [{'token_index': i, 'token': slow.convert_ids_to_tokens([identifier])[0], 'token_id': identifier,
                       'start': trace['offsets'][i][0], 'end': trace['offsets'][i][1],
                       'window_coverage': int(trace['counts'][i])} for i, identifier in enumerate(trace['ids'])]
        pd.DataFrame(words).to_csv(destination/f'{role}_words.csv', index=False)
        pd.DataFrame(sentences).to_csv(destination/f'{role}_sentences.csv', index=False)
        pd.DataFrame(token_rows).to_csv(destination/f'{role}_tokens.csv', index=False)
        item = {'source': source, 'dimension': model.config.hidden_size, 'words': words, 'sentences': sentences,
                'tokens': token_rows, 'word_shape': list(after.shape), 'sentence_shape': list(vectors.shape),
                'uncovered_tokens': int(np.count_nonzero(trace['counts'] == 0)), 'window_starts': trace['starts'],
                'repeated_words': compare_repetitions(words, before, after, cfg.get('focus_word', '')),
                'word_method': 'mean subword vectors overlapping each word; contextual overlap windows averaged per token first',
                'sentence_method': 'encode each sentence independently; mean window CLS' if role == 'roberta' else 'encode each sentence independently; CLS then L2; truncate at configured maximum',
                'npz': str(destination/f'{role}_vectors.npz')}
        item['input_policy']='soal + referensi + jawaban; '+cfg['input_format']
        if paired:
            item['sentence_method']='setiap kalimat dipasangkan dengan konteks yang sama; mean window CLS'+('; L2 setelah agregasi' if role=='bge' else '')
            item['source']+='; input context pair'
        dump(destination/f'{role}_metadata.json', item)
        results[role] = item
        label = 'RoBERTa' if role == 'roberta' else 'BGE-M3'
        if role == 'roberta':
            label += ' context pair sebelum latihan' if cfg['vector_source']=='pretrained_pair' else ' sebelum latihan' if cfg['vector_source']=='pretrained' else f' {cfg["vector_source"]} setelah latihan'
        if role == 'roberta' or cfg['vector_source']=='pretrained':
            print(f'{label}: vektor untuk {len(words)} kata dan {len(sentences)} kalimat selesai. Setiap vektor berisi {model.config.hidden_size} angka.', flush=True)
            if item['uncovered_tokens']:
                print(f'Catatan: {item["uncovered_tokens"]} token melewati batas input dan tidak mendapat vektor.', flush=True)
        del model, slow, fast
        release()
    results['interpretation'] = 'Bandingkan kemunculan dalam encoder yang sama. RoBERTa 768 dan BGE 1024 mempunyai ruang koordinat berbeda: bandingkan vektor hanya di ruang encoder yang sama. Same token ID => lookup sama pada checkpoint yang sama; konteks/posisi dapat mengubah hidden state. Kata ejaan sama bisa mempunyai ID subword berbeda (spasi/case).'
    dump(destination/'report.json', results)
    return results


### `export_word_sentence_vectors`

Inspeksi memakai vektor aktual. Pemanggilan eksekusinya berada setelah semua fungsi bagian ini didefinisikan.


In [ ]:
def export_word_sentence_vectors(result, journal, name):
    sheets = {}
    for role in ('roberta', 'bge'):
        report = result[role]
        with np.load(report['npz'], allow_pickle=False) as arrays:
            for key, id_key, metadata in [('token_initial','token_index',report['tokens']),
                                          ('token_contextual','token_index',report['tokens']),
                                          ('word_initial','word_index',report['words']),
                                          ('word_contextual','word_index',report['words']),
                                          ('sentence_cls','sentence_index',report['sentences'])]:
                values = arrays[key]
                frame = vector_frame(values, [row[id_key] for row in metadata])
                label_key = 'token' if key.startswith('token') else 'word' if key.startswith('word') else 'text'
                frame.insert(1, label_key, [row[label_key] for row in metadata])
                if key.startswith('word'):
                    frame.insert(2, 'covered', [row['covered'] for row in metadata])
                sheets[role+'_'+key] = frame
        sheets[role+'_metadata_token'] = pd.DataFrame(report['tokens'])
        sheets[role+'_metadata_word'] = pd.DataFrame(report['words'])
        sheets[role+'_metadata_sentence'] = pd.DataFrame(report['sentences'])
        sheets[role+'_kata_berulang'] = pd.DataFrame(report['repeated_words'])
    journal.save(name, sheets, 'Vektor token/kata/kalimat LENGKAP untuk SATU jawaban',
                 source={r: result[r]['source'] for r in ('roberta','bge')},
                 methods={r:{k:result[r][k] for k in ('word_method','sentence_method','uncovered_tokens','window_starts')} for r in ('roberta','bge')},
                 interpretation=result['interpretation'],
                 missing_vectors='Sel kosong berarti NaN/tidak tercakup; lihat kolom covered/coverage, bukan vektor nol.')


In [ ]:
if RUN_WORD_VECTORS:
    before_words=vector_lab(trace_text,{**cfg,'vector_source':'pretrained'},output,context=trace_context)
    export_word_sentence_vectors(before_words,journal,'07_vektor_kata_kalimat_pretrained')
    for role in ('roberta','bge'):
        word_rows=before_words[role]['words']
        preview=pd.DataFrame([{'word_index':w['word_index'],'word':w['word'],'covered':w['covered'],
                              **{f'v_{i:04d}':w['contextual_first5'][i] if w['contextual_first5'] is not None else None for i in range(5)}} for w in word_rows])
        show(preview,('RoBERTa' if role=='roberta' else 'BGE-M3')+' — contoh vektor kata')
        pairs=list(repeated_word_rows(before_words[role]['repeated_words'],role))
        show(pd.DataFrame(pairs,columns=REPEATED_COLUMNS),
             'Perbandingan kata yang berulang',columns=9)


<a id="arsitektur"></a>
## 19 · Arsitektur S3 E44

RoBERTa → Scalar Mix empat layer + gamma → token 768. Respons → BiGRU 64 per arah → 128; referensi → Linear 128. Co-attention dua arah → matching 640→128 → masked mean token valid 128 → Linear 768 + LayerNorm. BGE frozen 1024 → proyeksi 256 + LayerNorm. Concat 1024 → MLP 256 → GELU → Dropout 0,1 → satu skor.

Co-attention tetap dipelajari. Pooling akhir memakai bobot seragam 1/N untuk token valid dan 0 untuk padding.


### `answer_token_mask`

Mask memisahkan token jawaban dari soal/referensi/prefix/padding berdasarkan posisi tokenizer.


In [ ]:
def answer_token_mask(windows, mapping, details, padding_side='right'):
    """Mask posisi respons saja; konteks/prefix/special/padding tidak dipool."""
    valid = windows['attention_mask'].bool()
    mask = torch.zeros_like(valid)
    flat = [(essay, positions) for essay, info in enumerate(details)
            for positions in info['answer_positions']]
    if len(flat) != len(valid) or padding_side not in ('left', 'right'):
        raise ValueError('Pemetaan window/posisi jawaban tidak cocok.')
    for window, (essay, positions) in enumerate(flat):
        if int(mapping[window]) != essay or not positions:
            raise ValueError('Window harus berisi token jawaban dari esai yang benar.')
        offset = valid.shape[1] - int(valid[window].sum()) if padding_side == 'left' else 0
        indices = torch.tensor(positions, dtype=torch.long, device=valid.device) + offset
        if int(indices.min()) < 0 or int(indices.max()) >= valid.shape[1]:
            raise ValueError('Posisi token jawaban keluar dari window.')
        if not bool(valid[window, indices].all()):
            raise ValueError('Token padding tidak boleh menjadi token jawaban.')
        mask[window, indices] = True
    return mask


### `check_answer_pool_inputs`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def check_answer_pool_inputs(hidden, mask):
    if (hidden.ndim != 3 or mask is None or mask.dtype != torch.bool
            or tuple(mask.shape) != tuple(hidden.shape[:2])
            or mask.device != hidden.device):
        raise ValueError('Pooling memerlukan hidden [window,token,768] dan mask jawaban boolean.')
    if not bool(mask.any(dim=1).all()):
        raise ValueError('Setiap window harus memiliki minimal satu token jawaban.')


### `AnswerBiGRUAttention` · dasar inisialisasi acuan

Kelas dasar ini menyediakan inisialisasi BiGRU/proyeksi yang sama dengan E42. Subclass E44 menghapus dua layer skor attention pooling dan mendefinisikan forward masked mean sendiri. Forward attention pada kelas dasar tidak dipanggil oleh E44.


In [ ]:
class AnswerBiGRUAttention(nn.Module):
    """E26: urutan token -> BiGRU -> attention atas semua output -> proyeksi768."""
    def __init__(self, hidden_size=64, attention_hidden_size=64):
        super().__init__()
        self.gru = nn.GRU(768, hidden_size, batch_first=True, bidirectional=True)
        self.attention_hidden = nn.Linear(2*hidden_size, attention_hidden_size)
        self.attention_score = nn.Linear(attention_hidden_size, 1, bias=False)
        self.output = nn.Linear(2*hidden_size, 768)
        self.last_weights = None
        self.last_summary = None

    def forward(self, hidden, mask):
        check_answer_pool_inputs(hidden, mask)
        sequences = [row[keep] for row, keep in zip(hidden, mask)]
        lengths = torch.tensor([len(row) for row in sequences], dtype=torch.long)
        padded = torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True)
        packed = torch.nn.utils.rnn.pack_padded_sequence(padded, lengths, batch_first=True, enforce_sorted=False)
        packed_outputs, _ = self.gru(packed)
        outputs, _ = torch.nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True)
        valid = torch.arange(outputs.shape[1], device=outputs.device)[None] < lengths.to(outputs.device)[:,None]
        scores = self.attention_score(torch.tanh(self.attention_hidden(outputs))).squeeze(-1).float()
        weights = torch.softmax(scores.masked_fill(~valid, -torch.inf), dim=1)
        summary = (weights.unsqueeze(-1)*outputs.float()).sum(1).to(outputs.dtype)
        # Pemetaan balik bobot ke posisi token asli, supaya Excel tidak menggeser konteks/padding.
        self.last_weights = torch.zeros_like(mask, dtype=torch.float32).masked_scatter(mask, weights[valid]).detach()
        self.last_summary = summary.detach()
        return self.output(summary)


### `reference_token_mask`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def reference_token_mask(windows, mapping, details, padding_side='right'):
    """Pemetaan posisi referensi memakai offset padding yang sama dengan respons."""
    converted = [{**info, 'answer_positions': info['reference_positions']} for info in details]
    return answer_token_mask(windows, mapping, converted, padding_side)


### `AnswerBiGRUCoAttentionMeanPool`

Sesudah BiGRU dan matching co-attention, semua token respons valid dirata-ratakan. Layer `attention_hidden` dan `attention_score` pooling dihapus setelah inisialisasi acuan agar bobot bersama serta keadaan pengacakan tetap sama untuk seed yang sama. Seluruh parameter yang tersisa ikut jalur prediksi; tidak ada skor pooling yang dipelajari.


In [ ]:
class AnswerBiGRUCoAttentionMeanPool(AnswerBiGRUAttention):
    """E44: BiGRU + co-attention respons–referensi, lalu masked mean token matching128."""
    def __init__(self, hidden_size=64, attention_hidden_size=64):
        super().__init__(hidden_size, attention_hidden_size)
        dim = 2 * hidden_size
        # Sesudah layer acuan: inisialisasi BiGRU/attention/output E26 tetap sama.
        self.reference_projection = nn.Linear(768, dim)
        self.matching_projection = nn.Linear(5 * dim, dim)
        self.capture_alignment = False
        self.last_cross_attention = None
        self.last_reference_weights = None
        self.last_weights = None
        self.last_response_coattention_weights = None
        # Semua bobot bersama dibuat dengan urutan RNG E42 yang sama.
        # Dua layer skor pooling acuan kemudian dihapus; tidak ada parameter attention pooling yang dilatih.
        del self.attention_hidden
        del self.attention_score

    def forward(self, hidden, mask, reference_mask):
        check_answer_pool_inputs(hidden, mask)
        check_answer_pool_inputs(hidden, reference_mask)
        if bool((mask & reference_mask).any()):
            raise ValueError('Mask respons dan referensi harus terpisah.')
        responses = [row[keep] for row, keep in zip(hidden, mask)]
        references = [row[keep] for row, keep in zip(hidden, reference_mask)]
        lengths = torch.tensor([len(row) for row in responses], dtype=torch.long)
        ref_lengths = torch.tensor([len(row) for row in references], dtype=torch.long)
        padded = torch.nn.utils.rnn.pad_sequence(responses, batch_first=True)
        packed = torch.nn.utils.rnn.pack_padded_sequence(padded, lengths, batch_first=True, enforce_sorted=False)
        packed_outputs, _ = self.gru(packed)
        response, _ = torch.nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True)
        reference = self.reference_projection(torch.nn.utils.rnn.pad_sequence(references, batch_first=True))
        valid = torch.arange(response.shape[1], device=hidden.device)[None] < lengths.to(hidden.device)[:, None]
        ref_valid = torch.arange(reference.shape[1], device=hidden.device)[None] < ref_lengths.to(hidden.device)[:, None]
        # Softmax/produk skor FP32 juga saat encoder memakai AMP. Kedua urutan punya >=1 token.
        with torch.autocast(device_type=hidden.device.type, enabled=False):
            response32, reference32 = response.float(), reference.float()
            scores = torch.bmm(response32, reference32.transpose(1, 2)) / math.sqrt(response.shape[-1])
            a_to_r = torch.softmax(scores.masked_fill(~ref_valid[:, None, :], -torch.inf), dim=-1)
            a_to_r = a_to_r.masked_fill(~valid[:, :, None], 0.)
            r_to_a = torch.softmax(scores.transpose(1, 2).masked_fill(~valid[:, None, :], -torch.inf), dim=-1)
            r_to_a = r_to_a.masked_fill(~ref_valid[:, :, None], 0.)
            aligned_reference = torch.bmm(a_to_r, reference32)
            # Jalur balik respons→referensi→respons; bukan attention atas satu vektor BGE.
            round_trip = torch.bmm(a_to_r, torch.bmm(r_to_a, response32))
            matching = torch.cat([response32, aligned_reference, response32 - aligned_reference,
                                  response32 * aligned_reference, round_trip], dim=-1)
        matched = torch.tanh(self.matching_projection(matching.to(response.dtype)))
        # Setiap token matching yang valid mendapat bobot 1/N.
        # Padding berbobot 0; mean ini merangkum token dalam satu window.
        pooling_weights = valid.float() / lengths.to(hidden.device).unsqueeze(1)
        summary = (pooling_weights.unsqueeze(-1) * matched.float()).sum(1).to(response.dtype)
        self.last_weights = torch.zeros_like(mask, dtype=torch.float32).masked_scatter(mask, pooling_weights[valid]).detach()
        # Rata-rata bobot arah referensi→respons untuk inspeksi, terpisah dari pooling.
        response_importance = r_to_a.sum(1) / ref_lengths.to(hidden.device)[:, None]
        self.last_response_coattention_weights = torch.zeros_like(mask, dtype=torch.float32).masked_scatter(
            mask, response_importance[valid]).detach()
        self.last_summary = summary.detach()
        reference_importance = a_to_r.sum(1) / lengths.to(hidden.device)[:, None]
        self.last_reference_weights = torch.zeros_like(reference_mask, dtype=torch.float32).masked_scatter(
            reference_mask, reference_importance[ref_valid]).detach()
        # Matriks hanya ditahan untuk ekspor inspeksi, bukan graph training atau seluruh dataset.
        self.last_cross_attention = a_to_r.detach() if self.capture_alignment else None
        return self.output(summary)


### `ScalarMix` dan `SingleScoreModel`

Satu encoder RoBERTa mengembalikan hidden_states. ScalarMix memilih empat keluaran Transformer terakhir, mengalikan bobot softmax yang dipelajari, lalu menjumlahkannya pada setiap posisi token. Gamma mengalikan campuran sebelum BiGRU/proyeksi referensi; tidak ada normalisasi tambahan. Tiga teks masuk encoder bersama; rater hanya menjadi target loss. Campuran768dimensi masuk BiGRU64perarah serta proyeksi referensi sebelum co-attention. CLS referensi untuk inspeksi tetap berasal dari layer terakhir.


In [ ]:
class ScalarMix(nn.Module):
    """Campurkan k layer Transformer terakhir dengan bobot global yang dipelajari."""
    def __init__(self, number_of_layers=4, gamma_initial=1.0):
        super().__init__()
        if type(number_of_layers) is not int or number_of_layers < 1:
            raise ValueError('Jumlah layer Scalar Mix harus bilangan bulat positif.')
        self.number_of_layers = number_of_layers
        # Empat logit nol memberi bobot0.25; gamma awal1 mempertahankan keluaran awal E41.
        self.logits = nn.Parameter(torch.zeros(number_of_layers))
        self.gamma = nn.Parameter(torch.full((1,), float(gamma_initial)))
        self.capture_stats = False

    def weights(self):
        return torch.softmax(self.logits, dim=0)

    def forward(self, hidden_states, attention_mask=None):
        # hidden_states[0] adalah embeddings; hanya keluaran layer Transformer dipilih.
        if hidden_states is None or len(hidden_states)-1 < self.number_of_layers:
            raise ValueError('Encoder tidak menyediakan cukup layer Transformer untuk Scalar Mix.')
        selected = hidden_states[-self.number_of_layers:]
        if any(value.shape != selected[0].shape for value in selected):
            raise ValueError('Semua tensor layer Scalar Mix harus mempunyai shape yang sama.')
        weights = self.weights().to(dtype=selected[0].dtype)
        # Penjumlahan bertahap menghindari stack tambahan [4,batch,token,768].
        mixed = torch.zeros_like(selected[0])
        for weight, value in zip(weights, selected):
            mixed = mixed + weight * value
        scaled = self.gamma.to(dtype=mixed.dtype) * mixed
        # Statistik hanya saat inspeksi fitur, tanpa graph gradien atau sinkronisasi CPU tiap train step.
        if self.capture_stats:
            with torch.no_grad():
                valid = attention_mask.bool() if attention_mask is not None else torch.ones_like(mixed[..., 0], dtype=torch.bool)
                counts = valid.sum(1).clamp_min(1)
                def average_norm(value):
                    return ((value.float().norm(dim=-1)*valid).sum(1)/counts).unsqueeze(1)
                self.norm_before = average_norm(mixed)
                self.norm_after = average_norm(scaled)
                self.norm_last_layer = average_norm(hidden_states[-1])
        return scaled

    def report(self, encoder_layers):
        weights = self.weights().detach().float().cpu().tolist()
        logits = self.logits.detach().float().cpu().tolist()
        start = encoder_layers-self.number_of_layers
        return [{'Indeks Layer RoBERTa': start+i, 'Layer ke-': start+i+1,
                 'Logit': logits[i], 'Bobot Softmax': weights[i],
                 'Bobot Awal': 1/self.number_of_layers, 'Gamma': float(self.gamma.detach().cpu().item())}
                for i in range(self.number_of_layers)]


class SingleScoreModel(nn.Module):
    """S3 saja: representasi dua encoder digabung dan dinilai oleh MLP."""
    def __init__(self, scenario, roberta=None, dropout=.1, sequence_hidden=64,
                 s3_mlp_dropout=.1, pooling_attention_hidden=64, scalar_mix_layers=4, gamma_initial=1.0):
        super().__init__()
        if scenario != 'S3' or roberta is None:
            raise ValueError('E44 memerlukan S3 dan encoder RoBERTa; tidak membuat skenario lain.')
        self.scenario, self.roberta = scenario, roberta
        roberta.requires_grad_(True)
        self.dropout = nn.Dropout(dropout)  # Atribut historis; forward S3 memakai dropout di head.
        # Urutan inisialisasi layer yang tidak berubah tetap mengikuti E30.
        self.roberta_norm = nn.LayerNorm(768)
        self.projection = nn.Sequential(nn.Linear(1024, 256), nn.LayerNorm(256))
        self.head = nn.Sequential(nn.Linear(1024, 256), nn.GELU(),
                                  nn.Dropout(s3_mlp_dropout), nn.Linear(256, 1))
        self.sequence_pool = AnswerBiGRUCoAttentionMeanPool(sequence_hidden, pooling_attention_hidden)
        if roberta.config.num_hidden_layers < scalar_mix_layers:
            raise ValueError('RoBERTa mempunyai lebih sedikit layer daripada scalar_mix_layers.')
        # Inisialisasi nol tidak mengubah RNG layer E30 yang sudah dibentuk.
        self.scalar_mix = ScalarMix(scalar_mix_layers, gamma_initial)

    def forward(self, windows=None, mapping=None, n_essays=None, bge=None,
                return_features=False, answer_mask=None, reference_mask=None):
        encoded = self.roberta(**windows, output_hidden_states=True, return_dict=True)
        self.scalar_mix.capture_stats = return_features
        hidden = self.scalar_mix(encoded.hidden_states, windows['attention_mask'])
        cls = encoded.last_hidden_state[:, 0, :]  # CLS referensi inspeksi tetap layer terakhir.
        # BiGRU64/arah menghasilkan128fitur per token; pooling diproyeksikan ke768.
        pooled = self.sequence_pool(hidden, answer_mask, reference_mask)
        essay = mean_by_essay(pooled, mapping, n_essays)
        features = {}
        if return_features:
            for name, value in [('mix_norm_before', self.scalar_mix.norm_before),
                                ('mix_norm_after', self.scalar_mix.norm_after),
                                ('mix_norm_last', self.scalar_mix.norm_last_layer)]:
                features[name] = mean_by_essay(value, mapping, n_essays)
        features['roberta_cls_reference'] = mean_by_essay(cls, mapping, n_essays)
        features['answer_pool768'] = essay
        features['bigru_summary'] = mean_by_essay(self.sequence_pool.last_summary, mapping, n_essays)
        features.update(roberta=essay, roberta_window_cls=cls)
        if bge is None or bge.shape != (n_essays, 1024):
            raise ValueError('BGE frozen memerlukan satu vektor1024 untuk setiap jawaban.')
        features['bge'] = bge
        features['roberta_layernorm'] = self.roberta_norm(essay)
        features['projection'] = self.projection(bge)
        features['concat'] = torch.cat([features['roberta_layernorm'], features['projection']], dim=1)
        prediction = self.head(features['concat'])
        if prediction.ndim != 2 or prediction.shape[1] != 1:
            raise ValueError('S3 harus menghasilkan satu nilai untuk setiap jawaban.')
        return (prediction, features) if return_features else prediction


### `Engine`

Menyiapkan tensor dari teks/konteks, menjalankan model dan mengembalikan prediksi ke satuan nilai guru. Target tidak diberikan ke forward().


In [ ]:
class Engine:
    """Jalankan tiga teks melalui model fusion S3 yang sama."""
    def __init__(self, model, texts, bge, cfg, device, roberta_tokenizer=None, contexts=None):
        if model.scenario != 'S3':
            raise ValueError('Engine E44 hanya menjalankan S3.')
        self.model, self.texts, self.bge, self.cfg, self.device = model, texts, bge, cfg, device
        self.contexts = contexts
        if contexts is None or len(contexts) != len(texts):
            raise ValueError('Setiap jawaban memerlukan satu pasangan soal dan referensi.')
        self.tokenizer = roberta_tokenizer or tokenizer(cfg, 'roberta')

    def forward(self, indices, return_features=False):
        ids = [int(i) for i in indices]
        features = self.bge[ids].to(self.device)
        windows, mapping, details = pair_batch_windows(
            [self.texts[i] for i in ids], [self.contexts[i] for i in ids],
            self.tokenizer, self.device, self.cfg['roberta_max_length'],
            self.cfg['roberta_overlap'], self.cfg)
        mask = answer_token_mask(windows, mapping, details, self.tokenizer.padding_side)
        ref_mask = reference_token_mask(windows, mapping, details, self.tokenizer.padding_side)
        self.model.sequence_pool.capture_alignment = return_features
        if return_features:
            self.pooling_batch = (windows, mask)
            self.coattention_batch = (windows, mask, ref_mask)
        return self.model(windows=windows, mapping=mapping, n_essays=len(ids), bge=features,
                          return_features=return_features, answer_mask=mask, reference_mask=ref_mask)

    def predict(self, indices):
        # Mode prediksi: nonaktifkan dropout. Pembacaan di bawah memakai no_grad.
        self.model.eval()
        values = []
        # Tidak menghitung gradien untuk tahap pembacaan ini; bukan langkah latihan.
        with torch.no_grad():
            for start in range(0, len(indices), self.cfg['micro_batch']):
                ids = indices[start:start+self.cfg['micro_batch']]
                values.append(self.forward(ids).reshape(-1).float().cpu().numpy())
        scaled = np.concatenate(values)
        # Kembalikan skala asli sebelum RMSE/Pearson maupun tampilan prediksi.
        # Kembalikan prediksi ke skala nilai guru sebelum ditampilkan atau dihitung RMSE/Pearson.
        return scaled*(self.cfg['score_max']-self.cfg['score_min'])+self.cfg['score_min']


## 20 · Inisialisasi model dan spesifikasi

Setiap fold/refit membuat model baru. Seed direset pada awal fit. Ini tidak melanjutkan bobot dari fold sebelumnya.


### `seed_everything`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


### `make_model`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def make_model(scenario, cfg):
    if scenario != 'S3':
        raise ValueError('E44 hanya membuat model S3.')
    return SingleScoreModel('S3', encoder(cfg, 'roberta'), dropout=cfg['dropout'],
                            sequence_hidden=cfg['s3_sequence_hidden'],
                            s3_mlp_dropout=cfg['s3_mlp_dropout'],
                            pooling_attention_hidden=cfg['pooling_attention_hidden'],
                            scalar_mix_layers=cfg['scalar_mix_layers'], gamma_initial=cfg['scalar_mix_gamma_init'])


In [ ]:
specifications=pd.DataFrame([{
    'scenario':'S3', 'encoder':'RoBERTa fine-tuned; BGE-M3 frozen', 'vector':1024,
    'head':f"RoBERTa → Scalar Mix4 + gamma → BiGRU{cfg['s3_sequence_hidden']}/arah → co-attention{2*cfg['s3_sequence_hidden']} → masked mean pooling → proyeksi768 + BGE proyeksi256 → concat1024 → MLP256 →1",
    'output':1}])
journal.save('08_arsitektur',{'Spesifikasi':specifications},'Arsitektur S3 E44; belum ada prediksi',status='DESIGN / CONFIGURATION')
show(specifications,'Satu model S3 dan dimensi yang dipakai')


## 21 · Metrik evaluasi

RMSE/Pearson memakai prediksi kontinu. QWK/AC2 memakai salinan kategori0–100, pembulatan tie ke bawah. Angka agreement bukan persentase jawaban tepat.


### `gwet_ac2`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def gwet_ac2(teacher, predicted, categories):
    a=np.asarray(teacher);b=np.asarray(predicted);c=np.asarray(categories,dtype=float)
    if len(a)!=len(b) or len(a)<2: raise ValueError('AC2 perlu minimal dua pasangan')
    if not np.isin(a,c).all() or not np.isin(b,c).all(): raise ValueError('Kategori AC2 tidak sah')
    q=len(c); weights=1-((c[:,None]-c[None,:])/(c.max()-c.min()))**2
    ai=np.searchsorted(c,a); bi=np.searchsorted(c,b)
    pa=weights[ai,bi].mean()
    pi=(np.bincount(ai,minlength=q)+np.bincount(bi,minlength=q))/(2*len(a))
    pe=weights.sum()*np.sum(pi*(1-pi))/(q*(q-1))
    return float((pa-pe)/(1-pe))


### `metrics`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def metrics(teacher, prediction, cfg):
    y, p = np.asarray(teacher, dtype=float), np.asarray(prediction, dtype=float)
    if y.ndim != 1 or y.shape != p.shape or len(y) == 0:
        raise ValueError('Metrik satu skor memerlukan dua vektor [N] yang sama dan tidak kosong.')
    if not np.isfinite(y).all() or not np.isfinite(p).all():
        raise ValueError('Label/prediksi tidak finite.')
    cats = np.arange(cfg['score_min'], cfg['score_max']+1)
    if not np.isin(y, cats).all():
        raise ValueError('Skor guru harus kategori integer pada rentang yang dinyatakan.')
    # Buat SALINAN skor sah untuk agreement; prediksi mentah tetap utuh.
    clipped = np.clip(p, cfg['score_min'], cfg['score_max'])
    # Tie .5 memilih kategori lebih rendah, sama dengan kontrak konversi proyek.
    ordinal = cats[np.abs(clipped[:, None]-cats[None, :]).argmin(axis=1)]
    result = {'RMSE': float(np.sqrt(np.mean((y-p)**2))), 'Pearson': None,
              'QWK': None, 'AC2': None, 'undefined_reasons': {},
              'out_of_range_fraction': float(np.mean((p < cats[0]) | (p > cats[-1])))}
    if len(y) > 1 and np.ptp(y) > 0 and np.ptp(p) > 0:
        result['Pearson'] = float(pearsonr(y, p).statistic)
    else:
        result['undefined_reasons']['Pearson'] = 'Perlu minimal dua pasangan dan kedua vektor tidak konstan.'
    if len(y) > 1:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            # QWK memakai jarak kategori kuadratik, bukan vektor embedding.
            qwk = cohen_kappa_score(y, ordinal, labels=cats, weights='quadratic')
        if np.isfinite(qwk):
            result['QWK'] = float(qwk)
        else:
            result['undefined_reasons']['QWK'] = 'Expected disagreement nol; undefined, bukan nol.'
        # AC2 memakai kategori yang sama dengan QWK dan koreksi peluang berbeda.
        result['AC2'] = gwet_ac2(y, ordinal, cats)
    else:
        result['undefined_reasons'].update(QWK='Perlu minimal dua pasangan.', AC2='Perlu minimal dua pasangan.')
    return result


## 22 · Optimizer, loss MSE dan checkpoint

LR encoder seragam5e-6;MLP/proyeksi/pooling_norm1e-4. AdamW,weight decay0.01,warmup10%,scheduler linear dan clipping1 mengikuti E30. Empat logit dan satu gamma Scalar Mix mempunyai kelompok optimizer yang sama dengan LR1e-4.


### `optimizer_groups`

Setiap parameter trainable diperiksa masuk tepat satu kelompok. RoBERTa5e-6;proyeksi BGE,MLP,pooling/norm1e-4. Empat logit dan satu gamma pada scalar_mix mempunyai kelompok scalar_mix dengan LR1e-4.


In [ ]:
def optimizer_groups(model, cfg):
    """Lima kelompok optimizer S3; semua parameter trainable tercakup tepat sekali."""
    buckets = {'roberta': [], 'projection': [], 'head': [], 'pooling_norm': [], 'scalar_mix': []}
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith('roberta.'):
            group = 'roberta'
        elif name.startswith('scalar_mix.'):
            group = 'scalar_mix'
        elif name.startswith('projection.'):
            group = 'projection'
        elif name.startswith('head.'):
            group = 'head'
        else:
            group = 'pooling_norm'
        buckets[group].append(parameter)
    rates = {'roberta': cfg['s3_encoder_lr'],
             'projection': cfg['projection_lr'],
             'head': cfg['s3_mlp_lr'],
             'pooling_norm': cfg['s3_layers_lr'], 'scalar_mix': cfg['scalar_mix_lr']}
    result = [{'name': name, 'params': params, 'lr': rates[name]} for name, params in buckets.items() if params]
    ids = [id(p) for group in result for p in group['params']]
    expected = {id(p) for p in model.parameters() if p.requires_grad}
    if len(ids) != len(set(ids)) or set(ids) != expected:
        raise ValueError('Optimizer harus mencakup setiap parameter trainable tepat sekali.')
    return result


### `save_training_progress`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def save_training_progress(path,payload):
    """Dua slot pada disk runtime lokal; tidak membuat satu file per epoch."""
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    previous=json.loads(path.read_text())['snapshots'] if path.exists() else []
    slot='A' if payload['epoch']%2 else 'B'
    target=path.with_name(f'{path.stem}_slot_{slot}.pt')
    # Tulis hanya slot yang lebih lama. Bila terputus, slot terbaru lain masih utuh.
    torch.save(payload,target)
    checksum=digest(target)
    snapshots=[{'file':target.name,'sha256':checksum}]+[r for r in previous if r['file']!=target.name][:1]
    dump(path,{'scope':SCOPE,'snapshots':snapshots})


### `load_training_progress`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def load_training_progress(path, contract):
    path = Path(path)
    if not path.exists():
        return None
    manifest = json.loads(path.read_text())
    if manifest.get('scope') != SCOPE:
        raise ValueError('Checkpoint progres bukan workflow AES ini.')
    for snapshot in manifest['snapshots']:
        source = path.parent/snapshot['file']
        if source.name != snapshot['file']:
            raise ValueError('Nama file checkpoint progres tidak sah.')
        if not source.exists() or digest(source) != snapshot['sha256']:
            print('Checkpoint terbaru tidak lengkap; memeriksa cadangan epoch sebelumnya.', flush=True)
            continue
        payload = torch.load(source, map_location='cpu', weights_only=True)
        if payload.get('contract') != contract:
            raise ValueError('Data, fold, konfigurasi atau perangkat checkpoint berbeda. Progres tidak dipakai untuk percobaan lain.')
        return payload
    raise RuntimeError('Tidak ada checkpoint progres yang utuh. Hasil lama dipertahankan; training tidak diulang diam-diam.')


### `fit`

SATU latihan fold/refit. Fungsi ini menyimpan checkpoint progres beserta RNG untuk resume lokal; parameter dipilih melalui validation, bukan test.


### Loss MSE + ListNet dan batch efektif

Fungsi di bawah membentuk distribusi nilai guru dan prediksi untuk setiap soal yang sama. Tidak ada pengurutan dataset atau nilai guru pada masukan encoder. Recompute hanya menghemat aktivasi; semua prediksi tetap terhubung ke graph gradien.


In [ ]:
def same_question_listnet(prediction, targets, questions):
    """ListNet top-one: bandingkan hanya jawaban dalam soal yang persis sama."""
    if prediction.ndim != 2 or prediction.shape[1] != 1 or targets.shape != prediction.shape:
        raise ValueError('ListNet memerlukan prediction dan targets berbentuk [B,1].')
    if len(questions) != len(prediction) or not len(questions):
        raise ValueError('Satu soal diperlukan untuk setiap prediksi dalam batch.')
    if not torch.isfinite(prediction).all() or not torch.isfinite(targets).all():
        raise ValueError('Prediksi/target ListNet harus finite.')
    groups = {}
    for position, question in enumerate(questions):
        if not isinstance(question, str) or not question.strip():
            raise ValueError('Soal untuk ranking harus teks yang tidak kosong.')
        groups.setdefault(question, []).append(position)
    terms = []
    ranked_answers = 0
    for positions in groups.values():
        if len(positions) < 2:
            continue  # Singleton tetap dilatih lewat MSE; tidak dibuat pasangan beda soal.
        indices = torch.tensor(positions, dtype=torch.long, device=prediction.device)
        scores = prediction.index_select(0, indices).reshape(-1).float()
        teacher = targets.index_select(0, indices).reshape(-1).detach().float()
        # Skala 0–1 yang sama dengan MSE; tidak ada temperature atau label tambahan.
        target_probability = torch.softmax(teacher, dim=0)
        log_probability = torch.log_softmax(scores, dim=0)
        terms.append(-(target_probability * log_probability).sum())
        ranked_answers += len(positions)
    # Nol terhubung ke graph, sehingga singleton memberi gradien ranking tepat nol.
    loss = torch.stack(terms).mean() if terms else prediction.float().sum() * 0.
    return loss, {'ranking_groups': len(terms), 'ranking_answers': ranked_answers}


def forward_microbatch_for_recompute(device_marker, indices, engine):
    """Marker perangkat memastikan PyTorch menyimpan RNG dropout pada GPU juga."""
    return engine.forward(indices)


def training_batch_loss(engine, batch, bundle, cfg, device, dtype, use_cuda):
    """Satu objective batch efektif, dengan forward microbatch dan gradien end-to-end."""
    predictions = []
    # Tensor kosong ini bukan fitur model; checkpoint memakainya untuk mengenali GPU.
    device_marker = torch.empty(0, device=device)
    for start in range(0, len(batch), cfg['micro_batch']):
        ids = batch[start:start+cfg['micro_batch']]
        with torch.autocast(device_type=device, dtype=dtype, enabled=use_cuda):
            if cfg['ranking_recompute']:
                # Menghitung ulang aktivasi saat backward untuk menghemat memori.
                # Ini bukan penyimpanan file bobot. RNG dropout dipertahankan.
                value = activation_checkpoint(forward_microbatch_for_recompute, device_marker, ids, engine,
                    use_reentrant=False, preserve_rng_state=True)
            else:
                value = engine.forward(ids)
        predictions.append(value.float())  # Tidak detach: encoder sampai MLP menerima gradien.
    prediction = torch.cat(predictions, dim=0)
    targets = torch.as_tensor((bundle.y[batch]-cfg['score_min'])/(cfg['score_max']-cfg['score_min']),
                              dtype=torch.float32, device=device).reshape(-1, 1)
    if prediction.shape != targets.shape:
        raise ValueError('Shape prediksi/target harus [B,1], tanpa broadcasting.')
    mse = torch.mean((prediction-targets)**2)
    questions = [bundle.rows[index]['question'] for index in batch]
    ranking, counts = same_question_listnet(prediction, targets, questions)
    total = mse + cfg['ranking_weight'] * ranking
    return total, mse, ranking, counts


In [ ]:
def fit(scenario, training, validation, bundle, bge, cfg, epochs, seed, checkpoint=None):
    seed_everything(seed)
    device = device_name(cfg)
    # Buat model baru untuk fold ini, lalu tempatkan di GPU atau CPU yang tersedia.
    model = make_model(scenario, cfg).to(device)
    engine = Engine(model, bundle.texts, bge, cfg, device, contexts=contexts_from_bundle(bundle))
    groups = optimizer_groups(model, cfg)
    learning_rates = {group['name']: group['lr'] for group in groups}
    # AdamW memperbarui kelompok parameter sesuai learning rate masing-masing.
    optimizer = torch.optim.AdamW(groups, weight_decay=cfg['weight_decay'])
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader([int(i) for i in training], batch_size=cfg['effective_batch'],
                        shuffle=True, generator=generator, num_workers=0)
    total_steps = len(loader)*epochs
    warmup = min(int(cfg['warmup_ratio']*total_steps), max(0, total_steps-1))
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup, total_steps)
    use_cuda = device == 'cuda'
    dtype = torch.bfloat16 if use_cuda and torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=use_cuda and dtype == torch.float16)
    selected_parameters = {}
    for name, parameter in model.named_parameters():
        group = name.split('.')[0]
        if parameter.requires_grad and group not in selected_parameters:
            selected_parameters[group] = (name, parameter.detach().reshape(-1)[:32].cpu().clone())
    best_state, best_epoch, best_rmse = None, 0, float('inf')
    patience = cfg['early_stopping_patience']
    without_improvement, early_stopped = 0, False
    history, active_steps, optimizer_steps, skipped_steps = [], 0, 0, 0
    start_epoch = 1
    semantic = {k: v for k, v in cfg.items() if k not in ('dataset', 'output', 'trace_position', 'vector_source', 'new_answer', 'focus_word', 'new_question', 'new_reference')}
    contract = {'format': 'epoch_resume_v1', 'source': bundle.source_sha256, 'scenario': scenario,
                'training': [int(i) for i in training], 'validation': None if validation is None else [int(i) for i in validation],
                'config': semantic, 'epochs': epochs, 'seed': seed,
                'torch': str(torch.__version__), 'device': device,
                'precision': str(dtype) if use_cuda else 'float32',
                'gpu': torch.cuda.get_device_name() if use_cuda else None}
    saved = load_training_progress(checkpoint, contract) if checkpoint is not None else None
    if saved is not None:
        model.load_state_dict(saved['model'])
        optimizer.load_state_dict(saved['optimizer'])
        scheduler.load_state_dict(saved['scheduler'])
        scaler.load_state_dict(saved['scaler'])
        generator.set_state(saved['loader_generator'])
        selected_parameters = saved['selected_parameters']
        best_state, best_epoch, best_rmse = saved['best_state'], saved['best_epoch'], saved['best_rmse']
        history, without_improvement = saved['history'], saved['without_improvement']
        active_steps, optimizer_steps, skipped_steps = saved['active_steps'], saved['optimizer_steps'], saved['skipped_steps']
        early_stopped = saved['early_stopped']
        start_epoch = epochs+1 if early_stopped else saved['epoch']+1
        random.setstate(saved['python_rng'])
        rng = saved['numpy_rng']
        np.random.set_state((rng[0], np.asarray(rng[1], dtype=np.uint32), rng[2], rng[3], rng[4]))
        torch.set_rng_state(saved['torch_rng'])
        if use_cuda:
            torch.cuda.set_rng_state_all(saved['cuda_rng'])
        print(f'{scenario}: progres epoch {saved["epoch"]}/{epochs} ditemukan; '
              + (f'lanjut epoch {start_epoch}.' if start_epoch <= epochs else 'latihan sudah selesai, memulihkan hasilnya.'), flush=True)
        del saved
    for epoch in range(start_epoch, epochs+1):
        model.train()
        loss_total, ranking_total, combined_total = 0., 0., 0.
        ranking_groups, ranking_answers, ranking_batches = 0, 0, 0
        for effective in loader:
            batch = effective.tolist()
            optimizer.zero_grad(set_to_none=True)
            # Kumpulkan prediksi microbatch dalam satu graph loss batch efektif.
            # Urutan batch tetap E43; kelompok soal hanya dibentuk di dalam loss.
            loss, mse, ranking, counts = training_batch_loss(
                engine, batch, bundle, cfg, device, dtype, use_cuda)
            if not torch.isfinite(loss):
                raise RuntimeError('Loss nonfinite; run dihentikan, tidak diklaim berhasil.')
            scaler.scale(loss).backward()
            loss_total += mse.detach().item()*len(batch)
            ranking_total += ranking.detach().item()*len(batch)
            combined_total += loss.detach().item()*len(batch)
            ranking_groups += counts['ranking_groups']
            ranking_answers += counts['ranking_answers']
            ranking_batches += int(counts['ranking_groups'] > 0)
            del loss, mse, ranking
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['gradient_clip'])
            lr_active = any(g['lr'] > 0 for g in optimizer.param_groups)
            previous_scale = scaler.get_scale()
            # Perbarui bobot; GradScaler bisa melewati langkah yang overflow pada AMP.
            scaler.step(optimizer)
            scaler.update()
            skipped = scaler.is_enabled() and scaler.get_scale() < previous_scale
            if skipped:
                skipped_steps += 1
            else:
                active_steps += int(lr_active)
                scheduler.step()
                optimizer_steps += 1
        if cfg['ranking_weight'] > 0 and ranking_groups == 0:
            raise RuntimeError('Tidak ada kelompok soal dengan >=2 jawaban dalam batch epoch ini; ranking tidak aktif. Periksa kolom soal dan komposisi data.')
        row = {'epoch': epoch, 'train_mse_scaled': loss_total/len(training),
               'train_listnet_loss': ranking_total/len(training), 'train_combined_loss': combined_total/len(training),
               'ranking_groups': ranking_groups, 'ranking_answers': ranking_answers,
               'ranking_batches': ranking_batches, 'ranking_coverage_percent': 100*ranking_answers/len(training),
               'optimizer_steps': optimizer_steps, 'active_lr_steps': active_steps, 'amp_skipped_steps': skipped_steps,
               'effective_batch_target': cfg['effective_batch'], 'micro_batch': cfg['micro_batch']}
        if validation is not None:
            predicted = engine.predict(validation)
            row['validation'] = metrics(bundle.y[validation], predicted, cfg)
            # Simpan epoch dengan RMSE validasi terkecil; data uji akhir tidak ikut memilih epoch.
            if row['validation']['RMSE'] < best_rmse:
                best_rmse, best_epoch = row['validation']['RMSE'], epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                without_improvement = 0
            else:
                without_improvement += 1
            row.update(best_epoch_so_far=best_epoch, epochs_without_improvement=without_improvement)
            message = (f" | RMSE validasi: {row['validation']['RMSE']:.4f} poin"
                       f" | terbaik: epoch {best_epoch}, {best_rmse:.4f} poin"
                       f" | tanpa perbaikan: {without_improvement}/{patience}")
        else:
            best_epoch = epoch
            message = ' | pelatihan final; data uji belum dibuka'
        history.append(row)
        print(f'{scenario} · epoch {epoch}/{epochs} | MSE: {row["train_mse_scaled"]:.6f}'
              f' | ListNet: {row["train_listnet_loss"]:.6f} | gabungan: {row["train_combined_loss"]:.6f}'
              f' | ranking: {ranking_answers}/{len(training)} jawaban, {ranking_groups} kelompok'+message, flush=True)
        # Hanya fold validasi yang boleh menghentikan latihan; data uji tidak dibaca.
        if validation is not None and without_improvement >= patience and epoch < epochs:
            early_stopped = True
        if checkpoint is not None:
            rng = np.random.get_state()
            # Progres menyimpan keadaan belajar, bukan bobot terbaik saja.
            save_training_progress(checkpoint, {
                'contract': contract, 'epoch': epoch, 'model': model.state_dict(),
                'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
                'loader_generator': generator.get_state(), 'python_rng': random.getstate(),
                'numpy_rng': (rng[0], rng[1].tolist(), rng[2], rng[3], rng[4]),
                'torch_rng': torch.get_rng_state(), 'cuda_rng': torch.cuda.get_rng_state_all() if use_cuda else [],
                'selected_parameters': selected_parameters, 'best_state': best_state,
                'best_epoch': best_epoch, 'best_rmse': best_rmse, 'history': history,
                'without_improvement': without_improvement, 'early_stopped': early_stopped,
                'active_steps': active_steps, 'optimizer_steps': optimizer_steps, 'skipped_steps': skipped_steps})
            print(f'{scenario}: progres epoch {epoch} tersimpan di disk lokal.', flush=True)
        if early_stopped:
            print(f'{scenario}: early stopping setelah {epoch} epoch; RMSE validasi tidak membaik selama {patience} epoch. '
                  f'Bobot epoch {best_epoch} dipulihkan.', flush=True)
            break
    if active_steps == 0:
        raise RuntimeError('Tidak ada pembaruan dengan LR aktif.')
    named = dict(model.named_parameters())
    changed = {group: not torch.equal(before, named[name].detach().reshape(-1)[:32].cpu())
               for group, (name, before) in selected_parameters.items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    # Mode pembacaan: dropout dinonaktifkan agar ekstraksi vektor dan prediksi konsisten.
    model.eval()
    info = {'scenario': scenario, 'best_epoch': best_epoch, 'epochs_completed': len(history),
            'max_epochs': epochs, 'early_stopping_patience': patience if validation is not None else None,
            'early_stopped': early_stopped, 'optimizer_learning_rates': learning_rates,
            'best_validation_rmse': best_rmse if validation is not None else None,
            'stop_reason': ('RMSE validasi tidak membaik selama batas kesabaran' if early_stopped else
                            'Batas maksimum epoch tercapai' if validation is not None else
                            'Jumlah epoch final yang dipilih dari CV selesai'),
            'history': history, 'optimizer_steps': optimizer_steps, 'active_lr_steps': active_steps, 'amp_skipped_steps': skipped_steps,
            'parameter_samples_changed': changed, 'BGE_encoder_trained': False,
            'selection': 'minimum validation RMSE in original score units' if validation is not None else 'fixed refit epochs selected from CV'}
    # Bobot diambil SESUDAH best_state dipulihkan; sesuai model yang dievaluasi.
    info['scalar_mix_weights'] = model.scalar_mix.report(model.roberta.config.num_hidden_layers)
    info['scalar_mix_layers'] = cfg['scalar_mix_layers']
    info['loss_recipe'] = cfg['loss_recipe']
    info['ranking_weight'] = cfg['ranking_weight']
    info['ranking_recompute'] = cfg['ranking_recompute']
    print('Bobot Scalar Mix pada checkpoint terpilih:', flush=True)
    print(pd.DataFrame(info['scalar_mix_weights']).to_string(index=False), flush=True)
    return model, engine, info


<a id="evaluasi"></a>
## 23 · Lima fold, refit dan final test

Setiap skenario memiliki5model CV baru dan1refit. Epoch refit=ceil median epoch terbaik5fold. Data test dievaluasi sesudah pemilihan model.


### `save_model`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def save_model(path, model, cfg, bundle, manifest, info):
    payload = {'scope': SCOPE, 'trained': True, 'scenario': model.scenario,
               'source_sha256': bundle.source_sha256, 'split_sha256': manifest['split_sha256'],
               'config': cfg, 'training': info,
               'state': {k: v.detach().cpu() for k, v in model.state_dict().items()}}
    torch.save(payload, path)


### `load_model`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def load_model(path, cfg):
    # Checkpoint hanya dari workflow ini; weights_only membatasi objek pickle.
    payload = torch.load(path, map_location='cpu', weights_only=True)
    if payload.get('scope') != SCOPE or payload.get('trained') is not True:
        raise ValueError('Checkpoint bukan model satu skor guru yang telah dilatih.')
    for key in ('loss_recipe', 'ranking_weight', 'ranking_recompute', 'roberta', 'bge', 'score_min', 'score_max', 'roberta_max_length', 'roberta_overlap', 'bge_max_length', 'dropout', 's3_mlp_dropout', 's3_encoder_lr', 'scalar_mix_gamma_init', 'scalar_mix_layers', 'scalar_mix_lr', 's3_sequence_hidden', 'pooling_attention_hidden', 's3_pooling', 's3_fusion'):
        if payload['config'].get(key) != cfg[key]:
            raise ValueError('Metadata checkpoint berbeda: '+key)
    model = make_model(payload['scenario'], cfg)
    model.load_state_dict(payload['state'], strict=True)
    # Mode pembacaan: dropout dinonaktifkan agar ekstraksi vektor dan prediksi konsisten.
    model.eval()
    return model, payload


### `train_all`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def train_all(bundle, manifest, bge, cfg, output, observer=None):
    output = Path(output)
    final_path = output/'final_report.json'
    if final_path.exists():
        report = json.loads(final_path.read_text())
        if report['source_sha256'] != bundle.source_sha256 or report['split_sha256'] != manifest['split_sha256']:
            raise ValueError('Hasil lama memakai sumber/split berbeda.')
        for scenario in SCENARIOS:
            if digest(output/scenario/'final.pt') != report['checkpoints'][scenario]:
                raise ValueError('Checkpoint hasil lama berubah.')
            if digest(output/scenario/'test_predictions.csv') != report['prediction_files'][scenario]:
                raise ValueError('Prediksi test tersimpan berubah.')
        print('Memakai hasil tersimpan yang cocok dengan data, pengaturan, dan versi kode. Pelatihan serta evaluasi data uji tidak diulang.', flush=True)
        return {**report, 'reused_completed_run': True}
    cv_rows, selected_epochs, cv_folds = [], {}, {}
    for scenario in SCENARIOS:
        destination = output/scenario
        destination.mkdir(exist_ok=True)
        completed = []
        for fold in manifest['folds']:
            file = destination/f'cv_fold_{fold["fold"]}.json'
            if file.exists():
                result = json.loads(file.read_text())
                if result['split_sha256'] != manifest['split_sha256'] or result['fold'] != fold['fold']:
                    raise ValueError('Cache fold memakai split berbeda.')
                print(f'{scenario}: memakai hasil validasi silang fold {fold["fold"]} yang sudah selesai.', flush=True)
            else:
                print(f'{scenario}: validasi silang, fold {fold["fold"]} dari {cfg["folds"]}. Data uji disimpan untuk evaluasi akhir.', flush=True)
                model, engine, info = fit(scenario, fold['train'], fold['validation'], bundle, bge, cfg,
                                          cfg['epochs'], cfg['seed']+fold['fold'],
                                          checkpoint=destination/f'progress_fold_{fold["fold"]}.json')
                prediction = engine.predict(fold['validation'])
                result = {'fold': fold['fold'], 'split_sha256': manifest['split_sha256'],
                          'training': info, 'metrics': metrics(bundle.y[fold['validation']], prediction, cfg),
                          'validation_indices': fold['validation'], 'prediction': prediction.tolist()}
                dump(file, result)
                del engine, model
                release()
            completed.append(result)
            print(f'{scenario} fold {fold["fold"]} selesai: dipilih epoch {result["training"]["best_epoch"]}; '
                  f'RMSE validasi {result["metrics"]["RMSE"]:.4f} poin.', flush=True)
            if observer is not None:
                observer('cv_fold', scenario, result)
            cv_rows.append({'Skenario': scenario, 'Fold': fold['fold'],
                            **{k: result['metrics'][k] for k in ('QWK', 'AC2', 'RMSE', 'Pearson')}})
        cv_folds[scenario] = completed
        selected_epochs[scenario] = int(math.ceil(np.median([r['training']['best_epoch'] for r in completed])))
    pd.DataFrame(cv_rows).to_csv(output/'cv_metrics.csv', index=False)
    selection = {'scope': SCOPE, 'selected_epochs': selected_epochs,
                 'method': 'ceil median best validation-RMSE epoch across development folds',
                 'all_scenarios_locked_before_final_test': True, 'split_sha256': manifest['split_sha256']}
    dump(output/'selection_locked.json', selection)
    print('Validasi silang semua skenario selesai. Berikutnya pelatihan model final pada seluruh data latih; '
          'ini tahap baru, bukan mengulang fold. Data uji tetap disimpan.', flush=True)
    # Seluruh refit selesai sebelum membaca prediksi/skor test untuk pelaporan.
    lock = output/'refit_locked.json'
    if lock.exists():
        previous = json.loads(lock.read_text())
        checksums = previous['checkpoints']
        if previous['selection'] != selection or any(digest(output/s/'final.pt') != checksums[s] for s in SCENARIOS):
            raise ValueError('Model/config final yang sudah dikunci berubah; jangan mengulang test untuk tuning.')
    else:
        checksums = {}
        for scenario in SCENARIOS:
            path = output/scenario/'final.pt'
            complete = output/scenario/'refit_complete.json'
            if complete.exists():
                cached = json.loads(complete.read_text())
                if cached['selection'] != selection or cached['source'] != bundle.source_sha256 or digest(path) != cached['sha256']:
                    raise ValueError('Checkpoint pelatihan final tersimpan berbeda dari percobaan ini.')
                checksums[scenario] = cached['sha256']
                print(f'{scenario}: pelatihan final sudah selesai; memakai checkpoint tersimpan.', flush=True)
                continue
            print(f'{scenario}: pelatihan final pada {len(manifest["development"])} jawaban, '
                  f'{selected_epochs[scenario]} epoch pilihan CV. Early stopping tidak diterapkan di tahap final.', flush=True)
            model, engine, info = fit(scenario, manifest['development'], None, bundle, bge, cfg,
                                      selected_epochs[scenario], cfg['seed'], checkpoint=output/scenario/'progress_refit.json')
            save_model(path, model, cfg, bundle, manifest, info)
            checksums[scenario] = digest(path)
            dump(complete, {'selection': selection, 'source': bundle.source_sha256, 'sha256': checksums[scenario]})
            if observer is not None:
                observer('refit', scenario, info)
            del engine, model
            release()
        dump(lock, {'scope': SCOPE, 'checkpoints': checksums,
                    'selection': selection, 'test_evaluated': False})
    rows, prediction_files = [], {}
    for scenario in SCENARIOS:
        completed_test = output/scenario/'test_result.json'
        if completed_test.exists():
            cached = json.loads(completed_test.read_text())
            if cached['checkpoint_sha256'] != checksums[scenario] or cached['split_sha256'] != manifest['split_sha256'] or digest(output/scenario/'test_predictions.csv') != cached['prediction_sha256']:
                raise ValueError('Hasil test parsial berubah; tidak dievaluasi ulang.')
            rows.append(cached['row'])
            prediction_files[scenario] = cached['prediction_sha256']
            continue
        model, payload = load_model(output/scenario/'final.pt', cfg)
        if payload['source_sha256'] != bundle.source_sha256 or payload['split_sha256'] != manifest['split_sha256']:
            raise ValueError('Checkpoint final berbeda sumber/split.')
        model.to(device_name(cfg))
        engine = Engine(model, bundle.texts, bge, cfg, device_name(cfg), contexts=contexts_from_bundle(bundle))
        prediction = engine.predict(manifest['test'])
        score = metrics(bundle.y[manifest['test']], prediction, cfg)
        rows.append({'Skenario': scenario, **{k: score[k] for k in ('QWK', 'AC2', 'RMSE', 'Pearson')},
                     'Status': 'test held-out; satu skor guru', 'undefined_reasons': score['undefined_reasons'],
                     'out_of_range_fraction': score['out_of_range_fraction']})
        frame = pd.DataFrame({'row_id': [bundle.rows[i]['row_id'] for i in manifest['test']],
                              'skor_guru': bundle.y[manifest['test']], 'prediksi_mentah': prediction,
                              'selisih': prediction-bundle.y[manifest['test']]})
        path = output/scenario/'test_predictions.csv'
        frame.to_csv(path, index=False)
        prediction_files[scenario] = digest(path)
        dump(completed_test, {'row': rows[-1], 'checkpoint_sha256': checksums[scenario],
                              'split_sha256': manifest['split_sha256'], 'prediction_sha256': digest(path)})
        if observer is not None:
            observer('test', scenario, {'metrics': rows[-1], 'predictions': frame.to_dict('records')})
        del engine, model
        release()
    pd.DataFrame([{k: row[k] for k in ('Skenario', 'QWK', 'AC2', 'RMSE', 'Pearson', 'Status')} for row in rows]).to_csv(output/'test_metrics.csv', index=False)
    report = {'scope': SCOPE, 'source_sha256': bundle.source_sha256, 'split_sha256': manifest['split_sha256'],
              'rows': rows, 'checkpoints': checksums, 'prediction_files': prediction_files,
              'selected_epochs': selected_epochs, 'test_rows': manifest['test_rows'],
              'test_evaluations_per_scenario': 1, 'reused_completed_run': False,
              'cv': cv_rows, 'four_trait_labels_available': False}
    dump(final_path, report)
    return report


## 24 · Menyimpan dan menjalankan training

Fungsi ekspor di bawah mencatat hasil tahap. Cell terakhir bagian ini benar-benar memanggil `train_all(...)`; matikan RUN_TRAINING sebelum Run all jika hanya ingin audit data.


### `ordinal_copy`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def ordinal_copy(prediction, cfg):
    categories = np.arange(cfg['score_min'], cfg['score_max']+1)
    return categories[np.abs(np.clip(prediction, categories[0], categories[-1])[:, None]-categories).argmin(axis=1)]


### `prediction_frame`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def prediction_frame(bundle, indices, prediction, cfg):
    teacher = bundle.y[indices]
    ordinal = ordinal_copy(np.asarray(prediction), cfg)
    return pd.DataFrame({'row_id': [bundle.rows[i]['row_id'] for i in indices],
                         'skor_guru': teacher, 'prediksi_mentah': prediction,
                         'kategori_QWK_AC2': ordinal, 'selisih_mentah': prediction-teacher,
                         'selisih_kuadrat': (prediction-teacher)**2,
                         'bobot_agreement_kuadratik': 1-((ordinal-teacher)/(cfg['score_max']-cfg['score_min']))**2})


### `export_training_event`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def export_training_event(kind, scenario, result, bundle, cfg, journal):
    if kind == 'cv_fold':
        training = result['training']
        history = []
        for entry in result['training']['history']:
            history.append({**{k: v for k, v in entry.items() if k != 'validation'},
                            **entry.get('validation', {})})
        predictions = prediction_frame(bundle, result['validation_indices'], np.asarray(result['prediction']), cfg)
        journal.save(f'09_{scenario}_fold_{result["fold"]}', {'Riwayat_epoch': pd.DataFrame(history),
                                                           'Ringkasan_latihan': pd.DataFrame([{
                                                               'Epoch Maksimal': training.get('max_epochs'),
                                                               'Epoch Dijalankan': training['epochs_completed'],
                                                               'Epoch Terpilih': training['best_epoch'],
                                                               'Berhenti Lebih Awal?': training.get('early_stopped', False),
                                                               'Batas Tanpa Perbaikan': training.get('early_stopping_patience'),
                                                               'Alasan Selesai': training.get('stop_reason', '')}]),
                                                           'Bobot_scalar_mix': pd.DataFrame(training['scalar_mix_weights']),
                                                           'Prediksi_validation': predictions,
                                                           'Metrik_validation': pd.DataFrame([result['metrics']])},
                     'CV development; bukan final test', scenario=scenario,
                     selected_epoch=result['training']['best_epoch'], encoder_fine_tuned=scenario != 'S2')
    elif kind == 'refit':
        journal.save(f'10_{scenario}_refit', {'Riwayat_epoch': pd.DataFrame(result['history']),
                                            'Bukti_parameter': pd.DataFrame([result['parameter_samples_changed']]),
                                            'Bobot_scalar_mix': pd.DataFrame(result['scalar_mix_weights'])},
                     'Refit seluruh development; test belum dibuka', scenario=scenario,
                     epochs=result['epochs_completed'], BGE_encoder_trained=False)
    elif kind == 'test':
        raw = pd.DataFrame(result['predictions'])
        lookup = {r['row_id']: i for i, r in enumerate(bundle.rows)}
        ids = [lookup[row_id] for row_id in raw['row_id']]
        expanded = prediction_frame(bundle, ids, raw['prediksi_mentah'].to_numpy(), cfg)
        journal.save(f'11_{scenario}_test', {'Prediksi_test': expanded,
                                           'Metrik_test': pd.DataFrame([result['metrics']])},
                     'Final test satu skor guru', scenario=scenario)


### `test_score_comparison`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def test_score_comparison(report, bundle, cfg, output):
    """Bandingkan skor pada jawaban uji yang sama; baca prediksi tersimpan, tanpa forward ulang."""
    lookup = {row['row_id']: i for i, row in enumerate(bundle.rows)}
    combined = None
    for scenario in SCENARIOS:
        path = Path(output)/scenario/'test_predictions.csv'
        if digest(path) != report['prediction_files'][scenario]:
            raise ValueError('Prediksi tersimpan berubah sebelum tabel perbandingan dibuat.')
        frame = pd.read_csv(path).set_index('row_id')
        if frame.index.has_duplicates:
            raise ValueError('ID jawaban uji duplikat.')
        if combined is None:
            combined = frame[['skor_guru']].copy()
        if set(frame.index) != set(combined.index):
            raise ValueError('Jawaban uji antar skenario tidak sama.')
        frame = frame.loc[combined.index]
        actual_teacher = bundle.y[[lookup[identifier] for identifier in frame.index]]
        if not np.array_equal(frame['skor_guru'].to_numpy(), actual_teacher):
            raise ValueError('Nilai guru pada prediksi tidak cocok dengan dataset.')
        raw = frame['prediksi_mentah'].to_numpy()
        combined[f'Prediksi {scenario}'] = raw
        # Salinan kategori: clip 0–100, lalu integer terdekat; tepat .5 ke bawah.
        combined[f'Pembulatan {scenario}'] = ordinal_copy(raw, cfg)
    return combined.reset_index()


### `export_completed_training`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def export_completed_training(report, bundle, cfg, output, journal):
    output = Path(output)
    # Juga membangun ulang Excel dari artefak cocok hash ketika run di-resume.
    for scenario in SCENARIOS:
        for path in sorted((output/scenario).glob('cv_fold_*.json')):
            export_training_event('cv_fold', scenario, json.loads(path.read_text()), bundle, cfg, journal)
        checkpoint = output/scenario/'final.pt'
        if digest(checkpoint) != report['checkpoints'][scenario]:
            raise ValueError('Checkpoint final berbeda dari yang dievaluasi.')
        payload = torch.load(checkpoint, map_location='cpu', weights_only=True)
        export_training_event('refit', scenario, payload['training'], bundle, cfg, journal)
        del payload
        predicted = pd.read_csv(output/scenario/'test_predictions.csv')
        row = next(x for x in report['rows'] if x['Skenario'] == scenario)
        export_training_event('test', scenario, {'metrics': row, 'predictions': predicted.to_dict('records')}, bundle, cfg, journal)
    cv = pd.DataFrame(report['cv'])
    summary = cv.groupby('Skenario')[['QWK','AC2','RMSE','Pearson']].agg(['mean','std','count'])
    summary.columns = [f'{metric}_{stat}' for metric,stat in summary.columns]
    journal.save('12_evaluasi_s3', {'CV_semua_fold': cv, 'CV_mean_SD_n': summary.reset_index(),
                                             'TEST_final': pd.DataFrame(report['rows']),
                                             'Guru_prediksi_pembulatan': test_score_comparison(report, bundle, cfg, output),
                                             'Epoch_refit': pd.DataFrame([report['selected_epochs']])},
                 'Empat metrik QWK/AC2/RMSE/Pearson; CV dan test terpisah',
                 test_evaluations_per_scenario=report['test_evaluations_per_scenario'])


In [ ]:
if RUN_TRAINING:
    if not RUN_MODELS:raise RuntimeError('Aktifkan RUN_MODELS sebelumtraining.')
    def on_training_stage(kind,scenario,result):
        export_training_event(kind,scenario,result,bundle,cfg,journal)
    report=train_all(bundle,manifest,bge,cfg,output,observer=on_training_stage)
    export_completed_training(report,bundle,cfg,output,journal)
    show(pd.DataFrame(report['cv']),'Hasil validasi silang (lima fold)',rows=15)
    final_table=pd.DataFrame(report['rows'])[['Skenario','RMSE','Pearson','QWK','AC2']]
    show(final_table,'Hasil S3 pada data uji',rows=3)
    print('Perbandingan skor: nilai guru asli, prediksi mentah skala 0–100, dan salinan yang dibulatkan. '
          'RMSE/Pearson memakai prediksi mentah; QWK/AC2 memakai salinan kategori. Tepat .5 dibulatkan ke bawah.')
    # Semua jawaban uji memakai nilai guru asli; tidak menjalankan prediksi ulang.
    comparison=test_score_comparison(report,bundle,cfg,output)
    show(comparison,'Nilai guru, prediksi mentah, dan pembulatan pada data uji',rows=len(comparison),columns=8)
else:
    print('Pelatihan belum dijalankan. Prediksi dan evaluasi belum tersedia.')


## 25 · Prediksi tiga teks baru

Gunakan NEW_QUESTION/NEW_REFERENCE/NEW_ANSWER dari bagian6. Jika dibiarkan kosong, notebook menampilkan contoh test yang sudah dipisahkan. Tidak ada training tambahan.


### `score_answer`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def score_answer(text, cfg, output, question, reference):
    """Prediksi baru tanpa memerlukan nilai guru jawaban yang akan diprediksi."""
    output = Path(output)
    text = clean(text)
    if not text.strip():
        raise ValueError('Jawaban prediksi kosong.')
    source = hashlib.sha256(text.encode()).hexdigest()
    # Inference tidak mempunyai target/kelompok training; jangan mengisi skor 0.
    sample = Bundle([{'text': text, 'question': clean(question), 'reference': clean(reference)}], np.empty(0), np.empty(0, dtype=int), source, {})
    features = bge_features(sample, cfg, output/'prediction_encoding')
    rows = []
    report = json.loads((output/'final_report.json').read_text())
    for scenario in SCENARIOS:
        path = output/scenario/'final.pt'
        if digest(path) != report['checkpoints'][scenario]:
            raise ValueError('Checkpoint prediksi berbeda dari final yang dievaluasi.')
        model, payload = load_model(path, cfg)
        model.to(device_name(cfg))
        engine = Engine(model, [text], features, cfg, device_name(cfg), contexts=contexts_from_bundle(sample))
        prediction = float(engine.predict([0])[0])
        cats = np.arange(cfg['score_min'], cfg['score_max']+1)
        rounded = int(cats[np.abs(cats-np.clip(prediction, cats[0], cats[-1])).argmin()])
        rows.append({'Skenario': scenario, 'prediksi_mentah': prediction, 'kategori_salinan': rounded,
                     'di_luar_rentang': not cfg['score_min'] <= prediction <= cfg['score_max'],
                     'target': 'satu skor guru; bukan empat rubrik'})
        del engine, model
        release()
    dump(output/'one_answer_prediction.json', {'rows': rows})
    return rows


In [ ]:
if RUN_TRAINING:
    prediction_rows=score_answer(trace_text,cfg,output,*trace_context)
    # Contoh dari data uji mempunyai nilai guru. Jawaban baru belum mempunyai nilai guru.
    for row in prediction_rows:
        row['skor_guru']=None if cfg.get('new_answer','').strip() else float(bundle.y[trace_index])
    if cfg.get('new_answer','').strip():
        print('Jawaban baru: nilai guru belum tersedia, sehingga kolom Nilai Guru kosong.')
    journal.save('13_prediksi_satu_jawaban',{'Input_privat':pd.DataFrame([{'text':trace_text}]),
                                           'Konteks_privat':pd.DataFrame([{'soal':trace_context[0],'jawaban_referensi':trace_context[1]}]),
                                           'S3':pd.DataFrame(prediction_rows)},'Prediksi checkpointfinal; bukan trainingulang')
    show(pd.DataFrame(prediction_rows)[['Skenario','skor_guru','prediksi_mentah','kategori_salinan','di_luar_rentang']],
         'Nilai guru dan prediksi satu jawaban (skala 0–100)',columns=5)


## 26 · Fitur model setelah training

Inspeksi checkpoint final memakai eval/no_grad.300baris fitur bukan300data test. Bobot attention dan proyeksi/MLP intermediate diekspor untuk dipelajari.


### `export_final_features`

Fungsi pendukung ini dipanggil oleh bagian eksekusi di bawah; menjalankan definisi belum menjalankan training.


In [ ]:
def export_final_features(bundle, bge, cfg, output, journal):
    output = Path(output)
    report = json.loads((output/'final_report.json').read_text())
    for scenario in SCENARIOS:
        checkpoint = output/scenario/'final.pt'
        if digest(checkpoint) != report['checkpoints'][scenario]:
            raise ValueError('Checkpoint fitur berubah.')
        model, payload = load_model(checkpoint, cfg)
        del payload
        model.to(device_name(cfg)).eval()
        engine = Engine(model, bundle.texts, bge, cfg, device_name(cfg), contexts=contexts_from_bundle(bundle))
        hooks, intermediates = [], {}
        def save_hidden(name):
            def callback(module, args, value):
                intermediates[name] = value.detach().float().cpu().numpy()
            return callback
        # Tangkap fitur aktual, termasuk gate atau attention BiGRU; tahap inspeksi eval/no_grad.
        if scenario in ('S2', 'S3'):
            for name, layer in [('MLP_linear256',model.head[0]),('MLP_GELU256',model.head[1]),('MLP_dropout_eval',model.head[2])]:
                hooks.append(layer.register_forward_hook(save_hidden(name)))
        if scenario == 'S3':
            hooks.append(model.projection[0].register_forward_hook(save_hidden('proyeksi_linear256')))
        if scenario == 'S1':
            hooks.append(model.dropout.register_forward_hook(save_hidden('dropout_eval')))
        values, window_values, window_ids = {}, [], []
        pooling_rows, reference_rows, alignment_rows, coresponse_rows = [], [], [], []
        try:
            # Tidak menghitung gradien untuk tahap pembacaan ini; bukan langkah latihan.
            with torch.no_grad():
                for index, row in enumerate(bundle.rows):
                    prediction, features = engine.forward([index], return_features=True)
                    if scenario == 'S3' and hasattr(model.sequence_pool, 'last_weights'):
                        windows, mask = engine.pooling_batch
                        weights = model.sequence_pool.last_weights.cpu()
                        for window in range(len(mask)):
                            for position in mask[window].nonzero(as_tuple=False).flatten().tolist():
                                token_id = int(windows['input_ids'][window, position])
                                pooling_rows.append({'row_id':row['row_id'],'window':window,'position':position,
                                    'token_id':token_id,'token':engine.tokenizer.convert_ids_to_tokens([token_id])[0],
                                    'pooling_weight':float(weights[window, position])})
                    if scenario == 'S3':
                        windows, answer_mask, reference_mask = engine.coattention_batch
                        pool = model.sequence_pool
                        for window in range(len(answer_mask)):
                            answer_pos = answer_mask[window].nonzero(as_tuple=False).flatten().tolist()
                            ref_pos = reference_mask[window].nonzero(as_tuple=False).flatten().tolist()
                            for position in answer_pos:
                                token_id = int(windows['input_ids'][window, position])
                                coresponse_rows.append({'row_id':row['row_id'], 'window':window, 'position':position,
                                    'token_id':token_id, 'token':engine.tokenizer.convert_ids_to_tokens([token_id])[0],
                                    'attention_weight':float(pool.last_response_coattention_weights[window, position])})
                            ref_ids = [int(windows['input_ids'][window, p]) for p in ref_pos]
                            ref_tokens = engine.tokenizer.convert_ids_to_tokens(ref_ids)
                            for position, token_id, token in zip(ref_pos, ref_ids, ref_tokens):
                                reference_rows.append({'row_id':row['row_id'], 'window':window, 'position':position,
                                    'token_id':token_id, 'token':token,
                                    'attention_weight':float(pool.last_reference_weights[window, position])})
                            # Hanya kecocokan top-1 setiap token respons; bukan seluruh matriks atau skor kebenaran.
                            matrix = pool.last_cross_attention[window, :len(answer_pos), :len(ref_pos)].cpu()
                            for answer_index, position in enumerate(answer_pos):
                                best = int(matrix[answer_index].argmax())
                                token_id = int(windows['input_ids'][window, position])
                                alignment_rows.append({'row_id':row['row_id'], 'window':window,
                                    'answer_position':position, 'answer_token':engine.tokenizer.convert_ids_to_tokens([token_id])[0],
                                    'reference_position':ref_pos[best], 'reference_token':ref_tokens[best],
                                    'alignment_weight':float(matrix[answer_index, best])})
                    for name, value in features.items():
                        array = value.float().cpu().numpy()
                        if name == 'roberta_window_cls':
                            window_values.append(array)
                            window_ids.extend(f'{row["row_id"]}_window_{j}' for j in range(len(array)))
                        else:
                            values.setdefault(name, []).append(array)
                    for name, array in intermediates.items():
                        values.setdefault(name, []).append(array)
                    values.setdefault('output_normalized', []).append(prediction.float().cpu().numpy())
        finally:
            for handle in hooks:
                handle.remove()
        ids = [r['row_id'] for r in bundle.rows]
        sheets = {name: vector_frame(np.concatenate(parts), ids) for name, parts in values.items()}
        if pooling_rows:
            sheets['Bobot_mean_jawaban'] = pd.DataFrame(pooling_rows)
        if coresponse_rows:
            sheets['Bobot_coattention_respons'] = pd.DataFrame(coresponse_rows)
        if reference_rows:
            sheets['Bobot_token_referensi'] = pd.DataFrame(reference_rows)
        if alignment_rows:
            sheets['Pencocokan_token_top1'] = pd.DataFrame(alignment_rows)
        if window_values:
            sheets['CLS_tiap_window'] = vector_frame(np.concatenate(window_values), window_ids)
        raw = np.concatenate(values['output_normalized']).reshape(-1)*(cfg['score_max']-cfg['score_min'])+cfg['score_min']
        sheets['Prediksi_semua_jawaban'] = pd.DataFrame({'row_id':ids, 'prediksi_mentah':raw})
        journal.save(f'14_{scenario}_fitur_setelah_training', sheets,
                     'Fitur aktual checkpoint final; diekstrak SESUDAH final test, mode eval',
                     source={'scenario':scenario,'checkpoint_sha256':digest(checkpoint)},
                     note='Seluruh jawaban untuk inspeksi fitur; evaluasi penelitian hanya pada TEST_final, bukan tabel semua jawaban.')
        del engine, model
        release()


In [ ]:
if RUN_TRAINING and RUN_FINAL_FEATURES:
    print('Mengambil vektor jawaban dari model S3 yang sudah dilatih...')
    export_final_features(bundle,bge,cfg,output,journal)
    print('Vektor',len(bundle.rows),'jawaban siap disajikan dalam 02_Vektor_Jawaban.xlsx.')
    if RUN_WORD_VECTORS:
        for scenario in SCENARIOS:
            after_words=vector_lab(trace_text,{**cfg,'vector_source':scenario},output,context=trace_context)
            export_word_sentence_vectors(after_words,journal,f'15_{scenario}_vektor_kata_setelah_training')


## 27 · Susun lima workbook hasil

Masing-masing fungsi laporan membaca artefak sesi yang sudah ada. Tidak melatih ulang dan tidak menimpa skor guru. Rumus/metode tetap sumber E30.


### `Table`

Pembaca/penulis laporan aktual; data individual dan vektor disimpan hanya pada runtime/ZIP pribadi.


In [ ]:
@dataclass
class Table:
    headers: list
    rows: object


### `SavedOutputs`

Pembaca/penulis laporan aktual; data individual dan vektor disimpan hanya pada runtime/ZIP pribadi.


In [ ]:
class SavedOutputs:
    def __init__(self,folder):
        folder=Path(folder)
        self.folder=folder/'excel' if (folder/'excel').is_dir() else folder
        self.files={p.name:p for p in self.folder.glob('*.xlsx') if not p.name.endswith('.pending.xlsx')}
        self.names={}

    def has(self,file,sheet=None):
        if file not in self.files:return False
        if sheet is None:return True
        if file not in self.names:
            book=load_workbook(self.files[file],read_only=True,data_only=True)
            try:self.names[file]=book.sheetnames
            finally:book.close()
        return sheet in self.names[file]

    def records(self,file,sheet):
        if not self.has(file,sheet):return
        book=load_workbook(self.files[file],read_only=True,data_only=True)
        try:
            iterator=book[sheet].iter_rows(values_only=True)
            headers=next(iterator,())
            for values in iterator:
                yield dict(zip(headers,values))
        finally:book.close()

    def headers(self,file,sheet):
        book=load_workbook(self.files[file],read_only=True,data_only=True)
        try:return list(next(book[sheet].iter_rows(values_only=True),()))
        finally:book.close()

    def copy(self,file,sheet,columns=None,filter_row=None):
        headers=columns or self.headers(file,sheet)
        def values():
            for row in self.records(file,sheet):
                if filter_row and not filter_row(row):continue
                yield [friendly_value(key,row.get(key)) for key in headers]
        return Table([friendly_header(x) for x in headers],values())


### `guide`

Pembaca/penulis laporan aktual; data individual dan vektor disimpan hanya pada runtime/ZIP pribadi.


In [ ]:
def guide(description,items):
    return Table(['Bagian','Cara Membaca'],iter([
        ['Tentang file ini',description],
        ['Asal angka','Disalin dari hasil Colab yang sudah tersimpan. Tidak ada training, prediksi atau metrik yang dihitung ulang.'],
        ['ID Jawaban','J001 berarti baris jawaban pertama. ID ini hanya penghubung antarfile, bukan fitur model.'],
        ['Angka kosong','Berarti hasil tidak tersedia atau tidak dapat dihitung; bukan angka nol.'],
        *items]))


### `write_report`

Pembaca/penulis laporan aktual; data individual dan vektor disimpan hanya pada runtime/ZIP pribadi.


In [ ]:
def write_report(path,sheets):
    book=Workbook(write_only=True)
    for title,table in sheets.items():
        if len(title)>31:raise ValueError('Nama sheet terlalu panjang: '+title)
        if len(table.headers)>16384:raise ValueError('Jumlah kolom melebihi batas Excel.')
        sheet=book.create_sheet(title);sheet.freeze_panes='B2'
        sheet.sheet_format.defaultColWidth=16
        sheet.column_dimensions['A'].width=23
        if title=='Panduan':sheet.column_dimensions['B'].width=110
        for i,heading in enumerate(table.headers,1):
            if heading in ('Jawaban Siswa','Teks Asli','Kalimat','Cara Membaca','Penjelasan','Apa Artinya','Catatan'):
                sheet.column_dimensions[get_column_letter(i)].width=70
        header=[]
        for value in table.headers:
            cell=WriteOnlyCell(sheet,value=value);cell.data_type='s'
            cell.font=Font(bold=True,color='FFFFFF');cell.fill=PatternFill('solid',fgColor='172035')
            cell.alignment=Alignment(wrap_text=True,vertical='top');header.append(cell)
        sheet.append(header)
        count=1
        for row in table.rows:
            count+=1
            if count>1048576:raise ValueError('Terlalu banyak baris; data tidak dipotong diam-diam.')
            cells=[]
            for value in row:
                if isinstance(value,(dict,list,tuple)):value=json.dumps(value,ensure_ascii=False)
                if isinstance(value,float) and not math.isfinite(value):value=None
                cell=WriteOnlyCell(sheet,value=value)
                if isinstance(value,str):
                    if len(value)>32767:raise ValueError('Teks melebihi batas satu sel Excel.')
                    cell.data_type='s'
                    cell.alignment=Alignment(wrap_text=True,vertical='top')
                elif isinstance(value,float):cell.number_format='0.0000'
                cells.append(cell)
            sheet.append(cells)
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_name(path.stem+'.pending.xlsx');book.save(temporary);temporary.replace(path)
    return path


### `data_report`

Pembaca/penulis laporan aktual; data individual dan vektor disimpan hanya pada runtime/ZIP pribadi.


In [ ]:
def data_report(source):
    sheets={'Panduan':guide('Data, pembagian data dan contoh token yang benar-benar masuk model.',[
        ['Tokenisasi','Token bukan selalu satu kata. ID Token adalah nomor token pada kosakata model.'],
        ['Bagian Teks','Jawaban panjang dapat dibaca dalam beberapa bagian. Bagian bernomor mulai 1.'],
        ['Token yang ditampilkan','Token dengan attention mask 1 ditampilkan. Padding diabaikan; special token tetap ditampilkan.']])}
    file='02_dataset_dan_cleaning.xlsx'
    if source.has(file,'Sebelum_sesudah'):
        sheets['Data dan Nilai Guru']=source.copy(file,'Sebelum_sesudah',['row_id','soal','teks_model','jawaban_referensi','skor_guru'])
    file='03_split_dan_cv.xlsx'
    if source.has(file,'Split_setiap_baris'):
        sheets['Pembagian Data']=source.copy(file,'Split_setiap_baris',['row_id','role'])
    if source.has(file,'Fold_setiap_baris'):
        sheets['Pembagian Lima Fold']=source.copy(file,'Fold_setiap_baris',['fold','row_id','role'])
        sheets['Pembagian Lima Fold'].headers[0]='Fold'
    for raw,title in [('RoBERTa_input','Token RoBERTa'),('BGE_input','Token BGE-M3')]:
        file='04_tokenisasi.xlsx'
        if source.has(file,raw):
            keys=['row_id','window','position','token','token_id','input_segment']
            sheets[title]=source.copy(file,raw,keys,lambda r:r.get('attention_mask',1)==1)
    file='04_tokenisasi.xlsx'
    if source.has(file,'RoBERTa_S3_pair_input'):
        sheets['Token RoBERTa S3 Pair']=source.copy(file,'RoBERTa_S3_pair_input',['row_id','window','position','token','token_id','input_segment'])
    if source.has(file,'Panjang_dan_coverage'):
        sheets['Cakupan Token dan Konteks']=source.copy(file,'Panjang_dan_coverage')
    if source.has('13_prediksi_satu_jawaban.xlsx','Konteks_privat'):
        sheets['Konteks Jawaban Contoh']=source.copy('13_prediksi_satu_jawaban.xlsx','Konteks_privat')
    return sheets if len(sheets)>1 else None


### `answer_vectors_report`

Pembaca/penulis laporan aktual; data individual dan vektor disimpan hanya pada runtime/ZIP pribadi.


In [ ]:
def answer_vectors_report(source):
    mappings=[('14_S3_fitur_setelah_training.xlsx','mix_norm_before','Norma Campuran Sebelum'),
              ('14_S3_fitur_setelah_training.xlsx','mix_norm_after','Norma Campuran Sesudah'),
              ('14_S3_fitur_setelah_training.xlsx','mix_norm_last','Norma Layer Terakhir'),('05_roberta_pretrained.xlsx','Vektor_jawaban_768','RoBERTa Sebelum Latihan'),
              ('06_bge_frozen.xlsx','Vektor_jawaban_1024','BGE-M3'),
              ('14_S3_fitur_setelah_training.xlsx','roberta','RoBERTa S3 Setelah Latihan'),
              ('14_S3_fitur_setelah_training.xlsx','roberta_layernorm','RoBERTa S3 Setelah LayerNorm'),
              ('14_S3_fitur_setelah_training.xlsx','concat','Fusion S3'),
              ('14_S3_fitur_setelah_training.xlsx','answer_pool768','Pooling Jawaban S3'),
              ('14_S3_fitur_setelah_training.xlsx','Bobot_mean_jawaban','Bobot Mean Pooling S3'),
              ('14_S3_fitur_setelah_training.xlsx','bigru_summary','Ringkasan CoAttn Pooling S3'),
              ('14_S3_fitur_setelah_training.xlsx','Bobot_coattention_respons','Bobot CoAttention Respons S3'),
              ('14_S3_fitur_setelah_training.xlsx','Bobot_token_referensi','Bobot Referensi S3'),
              ('14_S3_fitur_setelah_training.xlsx','Pencocokan_token_top1','Pasangan Token Top1 S3')]
    sheets={'Panduan':guide('Satu baris adalah vektor satu jawaban. Semua komponen angka disimpan.',[
        ['Komponen 1, 2, ...','Satu komponen adalah satu angka dalam vektor; tidak mempunyai arti tetap seperti isi atau tata bahasa.'],
        ['Nama kolom lama','Komponen 1 adalah v_0000 pada file lama. Hanya namanya berubah, angkanya tetap.'],
        ['RoBERTa S3','Empat layer terakhir RoBERTa dicampur dengan bobot global yang dipelajari, lalu output token BiGRU128 dicocokkan dengan isi referensi128 melalui co-attention; masked mean merangkum semua token matching128yang valid lalu Linear768. Dua tahap berbeda, tanpa ringkasan ujung E27. Keluaran pooling768 yang ditampilkan, bukan CLS.'],
        ['RoBERTa','768 angka: S3 membaca soal + referensi + jawaban dengan format eksperimen yang sama. Bandingkan sebelum/sesudah pada input tiga teks yang sama.'],
        ['BGE-M3','1024 angka per jawaban. Bobot encoder tetap, sehingga dipakai langsung oleh cabang fusion S3.'],
        ['Fusion S3','Gabungan 768 angka RoBERTa setelah LayerNorm tambahan dan 256 angka hasil proyeksi BGE. Hasilnya 1024 komponen.'],
        ['LayerNorm tambahan','Sheet RoBERTa S3 Setelah Latihan menyimpan vektor asli; sheet Setelah LayerNorm menyimpan vektor yang masuk concat. Pada hasil versi lama, sheet tambahan ini belum ada.'],
        ['Co-attention dan mean pooling','Bobot Mean Pooling S3 adalah1/Npada token respons valid SETELAH matching,padding0;tidak dipelajari. Bobot CoAttention Respons S3 adalah rata-rata referensi→respons; Bobot Referensi S3 adalah rata-rata respons→referensi. Masing-masing jumlah1 per window, bukan skor guru/penjelasan kausal. Top1 hanya ringkasan kecocokan, bukan matriks lengkap.'],
        ['Batas perbandingan','Bandingkan komponen hanya dalam ruang encoder yang sama. Vektor bukan skor siswa.']])}
    for file,raw,title in mappings:
        if source.has(file,raw):sheets[title]=source.copy(file,raw)
    return sheets if len(sheets)>1 else None


### `word_vectors_report`

Pembaca/penulis laporan aktual; data individual dan vektor disimpan hanya pada runtime/ZIP pribadi.


In [ ]:
def word_vectors_report(source):
    base='07_vektor_kata_kalimat_pretrained.xlsx'
    conditions=[(base,'roberta','RoBERTa Sebelum'),(base,'bge','BGE-M3'),
                ('15_S3_vektor_kata_setelah_training.xlsx','roberta','RoBERTa S3')]
    sheets={'Panduan':guide('Vektor kata dan kalimat untuk satu jawaban contoh yang sama.',[
        ['Kata dan kalimat','Sheet Kata berisi vektor setelah konteks. Sheet Kalimat berasal dari pembacaan kalimat secara terpisah.'],
        ['Sebelum Konteks','Lookup embedding kata sebelum penambahan konteks; hanya perlu disajikan sekali untuk tiap encoder pretrained.'],
        ['Kata Berulang','Bandingkan dua posisi kata di model yang sama. Kesamaan numerik memakai toleransi 0,000001.'],
        ['Tercakup',"Nilai 'Tidak' berarti kata melewati batas input; komponen kosong bukan vektor nol."],
        ['BGE-M3','BGE dan RoBERTa membaca soal/referensi + jawaban. Vektor kata merujuk kata jawaban, dengan konteks tetap tersedia; BGE frozen.']])}
    if source.has('13_prediksi_satu_jawaban.xlsx','Input_privat'):
        sheets['Jawaban Contoh']=source.copy('13_prediksi_satu_jawaban.xlsx','Input_privat')
        sheets['Jawaban Contoh'].headers=['Jawaban Siswa']
    comparisons=[]
    def vector_table(file,sheet,unit):
        copied=source.copy(file,sheet)
        copied.headers[0]='Urutan Kata' if unit=='word' else 'Urutan Kalimat'
        return Table(copied.headers,([r[0]+1,*r[1:]] for r in copied.rows))
    for file,role,label in conditions:
        for key,title in [('word_contextual','Kata '+label),('sentence_cls','Kalimat '+label)]:
            if source.has(file,role+'_'+key):sheets[title]=vector_table(file,role+'_'+key,'word' if key.startswith('word') else 'sentence')
        if source.has(file,role+'_kata_berulang'):
            comparisons.extend(repeated_word_rows(source.records(file,role+'_kata_berulang'),label))
    for role,label in [('roberta','RoBERTa'),('bge','BGE')]:
        if source.has(base,role+'_word_initial'):
            sheets[label+' Sebelum Konteks']=vector_table(base,role+'_word_initial','word')
    if comparisons:
        sheets['Kata Berulang']=Table(REPEATED_COLUMNS,comparisons)
    return sheets if any(name.startswith('Kata ') for name in sheets) else None


### `evaluation_report`

Pembaca/penulis laporan aktual; data individual dan vektor disimpan hanya pada runtime/ZIP pribadi.


In [ ]:
def evaluation_report(source):
    file='12_evaluasi_s3.xlsx'
    sheets={'Panduan':guide('Buka Perbandingan Model untuk hasil akhir; buka Nilai Guru dan Prediksi untuk melihat setiap jawaban uji.',[
        ['S3 saja','Fusion RoBERTa fine-tuned + BGE-M3 frozen; tidak ada prediksi kosong S1/S2.'],
        ['Data uji','Angka test untuk laporan akhir. Pemilihan model/epoch memakai validasi silang, bukan tuning dari test.'],
        ['Skor mentah','Disimpan persis dari hasil sumber. Tampilan empat desimal tidak mengubah angka yang tersimpan.'],
        ['Pembulatan','Prediksi dibatasi ke 0–100, lalu dipilih integer terdekat. Tepat .5 ke bawah. Ini salinan untuk QWK/AC2; RMSE dan Pearson tetap memakai prediksi mentah.'],
        ['Prediksi Validasi OOF','Setiap jawaban development diprediksi oleh model fold yang tidak melatih jawaban tersebut; bukan hasil test.'],
        ['Ringkasan Validasi','Rata-rata ± SD antar-fold; bukan interval kepercayaan. Nilai mentah tiap-fold ada di Hasil Tiap Fold.']])}
    if source.has(file,'TEST_final'):
        def final_rows():
            for r in source.records(file,'TEST_final'):
                reasons=r.get('undefined_reasons') or ''
                if isinstance(reasons,str) and reasons.startswith('{'):
                    reasons=json.loads(reasons)
                if isinstance(reasons,dict):reasons='; '.join(f'{k}: {v}' for k,v in reasons.items())
                outside=r.get('out_of_range_fraction')
                yield [r.get('Skenario'),(MODELS.get(r.get('Skenario')) + ' + BiGRU + Co-Attention + Masked Mean Pooling' if r.get('Skenario')=='S3' else MODELS.get(r.get('Skenario'))),r.get('RMSE'),r.get('Pearson'),r.get('QWK'),r.get('AC2'),
                       None if outside is None else outside*100,reasons]
        sheets['Perbandingan Model']=Table(['Skenario','Model','RMSE (poin)','Pearson','QWK','Gwet AC2','Di Luar Rentang (%)','Catatan'],final_rows())
    if source.has(file,'CV_semua_fold'):
        sheets['Hasil Tiap Fold']=source.copy(file,'CV_semua_fold',['Skenario','Fold','RMSE','Pearson','QWK','AC2'])
    if source.has(file,'CV_mean_SD_n'):
        def summaries():
            for row in source.records(file,'CV_mean_SD_n'):
                values=[]
                for metric in ('RMSE','Pearson','QWK','AC2'):
                    mean,sd,n=row.get(metric+'_mean'),row.get(metric+'_std'),row.get(metric+'_count')
                    values.append('Tidak dapat dihitung' if mean is None else f'{mean:.4f} ± {sd:.4f} (n={n})' if sd is not None else f'{mean:.4f} (n={n})')
                yield [row.get('Skenario'),*values]
        sheets['Ringkasan Validasi']=Table(['Skenario','RMSE (rata-rata ± SD)','Pearson (rata-rata ± SD)','QWK (rata-rata ± SD)','Gwet AC2 (rata-rata ± SD)'],summaries())
    predicted={}
    for scenario in SCENARIOS:
        f=f'11_{scenario}_test.xlsx'
        for row in source.records(f,'Prediksi_test'):
            item=predicted.setdefault(row['row_id'],{'teacher':row['skor_guru']})
            if item['teacher']!=row['skor_guru']:raise ValueError('Nilai guru antarfile test berbeda.')
            item[scenario]=row['prediksi_mentah']
            item['pembulatan_'+scenario]=row.get('kategori_QWK_AC2')
            item['selisih_'+scenario]=row.get('selisih_mentah',row.get('selisih'))
    if predicted:
        def per_answer():
            for identifier,row in predicted.items():
                yield [friendly_value('row_id',identifier),row['teacher'],*[row.get(s) for s in SCENARIOS],
                       *[row.get('pembulatan_'+s) for s in SCENARIOS],
                       *[row.get('selisih_'+s) for s in SCENARIOS]]
        sheets['Nilai Guru dan Prediksi']=Table(['ID Jawaban','Nilai Guru','Prediksi S3','Pembulatan S3','Selisih S3'],per_answer())
    if source.has('13_prediksi_satu_jawaban.xlsx','S3'):
        sheets['Prediksi Satu Jawaban']=source.copy('13_prediksi_satu_jawaban.xlsx','S3',
                           ['Skenario','skor_guru','prediksi_mentah','kategori_salinan','di_luar_rentang'])
        sheets['Jawaban Contoh']=source.copy('13_prediksi_satu_jawaban.xlsx','Input_privat')
        sheets['Jawaban Contoh'].headers=['Jawaban Siswa']
    if len(sheets)==1:return None
    sheets['Arti Metrik']=Table(['Metrik','Cara Membaca','Catatan'],[
        ['RMSE','Lebih kecil berarti kesalahan prediksi lebih kecil.','Satuannya poin skor, bukan dibatasi 0–1.'],
        ['Pearson','Hubungan linear antara skor guru dan prediksi.','Tinggi belum menjamin nilai prediksi tepat.'],
        ['QWK','Kesepakatan skor kategori dengan penalti kuadratik.','Dibaca bersama RMSE dan Pearson.'],
        ['Gwet AC2','Kesepakatan kategori dengan koreksi peluang Gwet.','Dapat berbeda jauh dari QWK; daftar kategori memengaruhi hasil.']])
    # Pasangan OOF untuk audit residual; seluruhnya dari VALIDASI development.
    questions = {r['row_id']:r.get('soal') for r in source.records('02_dataset_dan_cleaning.xlsx','Sebelum_sesudah')}
    oof_rows, seen = [], set()
    for file in sorted(source.files):
        match = re.fullmatch(r'09_S3_fold_(\d+)\.xlsx', file)
        if not match:
            continue
        for row in source.records(file, 'Prediksi_validation'):
            identifier = row['row_id']
            if identifier in seen or identifier in predicted:
                raise ValueError('ID OOF duplikat atau bercampur dengan data test.')
            seen.add(identifier)
            error = row['prediksi_mentah']-row['skor_guru']
            oof_rows.append([int(match[1]), friendly_value('row_id',identifier), questions.get(identifier),
                row['skor_guru'], row['prediksi_mentah'], row['kategori_QWK_AC2'], error, abs(error), error**2])
    if oof_rows:
        sheets['Prediksi Validasi OOF'] = Table(
            ['Fold','ID Jawaban','Soal','Nilai Guru','Prediksi S3','Pembulatan S3',
             'Selisih S3','Selisih Absolut','Selisih Kuadrat'], oof_rows)
    return sheets


### `training_report`

Pembaca/penulis laporan aktual; data individual dan vektor disimpan hanya pada runtime/ZIP pribadi.


In [ ]:
def training_report(source):
    sheets={'Panduan':guide('Riwayat latihan per skenario. Lima fold digabung ke satu sheet untuk setiap model.',[
        ['Validasi silang','Model baru dilatih pada tiap fold, dengan satu fold untuk validasi.'],
        ['Latihan final','Latihan ulang memakai seluruh development dengan jumlah epoch yang dipilih dari CV.'],
        ['Loss Training (MSE)','Target dinormalisasi saat loss dihitung. Angka ini berbeda satuan dari RMSE dalam poin skor.'],
        ['Loss Ranking dan Gabungan','ListNet cross-entropy kelompok soal sama; gabungan = MSE + bobot × ListNet. Dirata-ratakan per epoch dengan bobot ukuran batch aktual. ListNet tidak harus nol karena target berupa distribusi.'],
        ['Cakupan Ranking','Jumlah jawaban yang berada dalam kelompok soal berukuran minimal dua dalam batch. Singleton hanya memakai MSE.'],
        ['Epoch','Satu putaran seluruh data latih pada fold/tahap itu.'],
        ['Bobot Scalar Mix','Empat bobot global pada checkpoint terpilih tiap fold/refit; berjumlah1. Indeks layer mulai0; layer ke- mulai1. Bukan skor siswa atau ukuran kepentingan kausal.']])}
    summaries=[]
    for scenario,label in [('S3','Fusion (S3)')]:
        files=sorted(f for f in source.files if re.fullmatch(f'09_{scenario}_fold_\\d+\\.xlsx',f))
        final=f'10_{scenario}_refit.xlsx'
        if source.has(final):files.append(final)
        if not files:continue
        def history(files=files):
            for f in files:
                match=re.search(r'_fold_(\d+)',f)
                for row in source.records(f,'Riwayat_epoch'):
                    yield ['Validasi silang' if match else 'Latihan final',int(match[1]) if match else None,
                           row.get('epoch'),row.get('train_mse_scaled'),row.get('RMSE'),row.get('Pearson'),row.get('QWK'),row.get('AC2'),
                           row.get('train_listnet_loss'),row.get('train_combined_loss'),row.get('ranking_groups'),
                           row.get('ranking_answers'),row.get('ranking_batches'),row.get('ranking_coverage_percent')]
        sheets[label]=Table(['Tahap','Fold','Epoch','Loss Training (MSE)','RMSE Validasi (poin)','Pearson Validasi','QWK Validasi','Gwet AC2 Validasi',
            'Loss Ranking (ListNet)','Loss Gabungan','Kelompok Soal untuk Ranking',
            'Jawaban untuk Ranking','Batch dengan Ranking','Cakupan Ranking (%)'],history())
        for file in files:
            match=re.search(r'_fold_(\d+)',file)
            if match and source.has(file,'Ringkasan_latihan'):
                for row in source.records(file,'Ringkasan_latihan'):
                    summaries.append([scenario,int(match[1]),row.get('Epoch Maksimal'),row.get('Epoch Dijalankan'),
                                      row.get('Epoch Terpilih'),row.get('Berhenti Lebih Awal?'),
                                      row.get('Batas Tanpa Perbaikan'),row.get('Alasan Selesai')])
    if len(sheets)==1:return None
    if summaries:
        sheets['Ringkasan Lima Fold']=Table(['Skenario','Fold','Epoch Maksimal','Epoch Dijalankan','Epoch Terpilih',
                                              'Berhenti Lebih Awal?','Batas Tanpa Perbaikan','Alasan Selesai'],summaries)
    if source.has('08_arsitektur.xlsx','Spesifikasi'):
        sheets['Model yang Dibandingkan']=source.copy('08_arsitektur.xlsx','Spesifikasi')
    settings={'loss_recipe':'Resep Loss','ranking_weight':'Bobot ListNet','ranking_recompute':'Hitung ulang aktivasi ranking','scalar_mix_gamma_init':'Gamma awal Scalar Mix','scalar_mix_layers':'Jumlah layer Scalar Mix','scalar_mix_lr':'LR Scalar Mix','pooling_attention_hidden':'Ukuran init attention acuan (dihapus)','s3_mlp_dropout':'Dropout MLP S3','dropout':'Dropout parameter lama','seed':'Seed training tetap','s3_mlp_lr':'LR MLP S3','s3_layers_lr':'LR pooling dan norm S3','s3_fusion':'Fusion S3','s3_pooling':'Pooling jawaban S3','s3_sequence_hidden':'Hidden pooling S3','input_format':'Format input tiga teks','epochs':'Maksimum epoch per fold','folds':'Jumlah fold','effective_batch':'Batch efektif',
              'micro_batch':'Micro batch','s3_encoder_lr':'Learning rate encoder RoBERTa S3',
              'context_max_tokens':'Batas token konteks semua encoder','question_max_tokens':'Batas token soal semua encoder',
              'projection_lr':'Learning rate proyeksi BGE','early_stopping_patience':'Batas epoch tanpa perbaikan',
              'score_min':'Skor minimum','score_max':'Skor maksimum','roberta_max_length':'Batas token RoBERTa',
              'bge_max_length':'Batas token BGE-M3'}
    if source.has('01_konfigurasi.xlsx','Parameter'):
        sheets['Pengaturan Eksperimen']=Table(['Pengaturan','Nilai'],([settings[r['name']],r.get('value')] for r in source.records('01_konfigurasi.xlsx','Parameter') if r.get('name') in settings))

    mix_rows = []
    for file in sorted(source.files):
        match = re.fullmatch(r'09_S3_fold_(\d+)\.xlsx', file)
        if not match and file != '10_S3_refit.xlsx':
            continue
        if source.has(file, 'Bobot_scalar_mix'):
            for row in source.records(file, 'Bobot_scalar_mix'):
                mix_rows.append(['Validasi silang' if match else 'Latihan final',
                                 int(match[1]) if match else None, row['Indeks Layer RoBERTa'],
                                 row['Layer ke-'], row['Logit'], row['Bobot Softmax'], row['Bobot Awal'], row['Gamma']])
    if mix_rows:
        sheets['Bobot Scalar Mix'] = Table(
            ['Tahap','Fold','Indeks Layer RoBERTa','Layer ke-','Logit','Bobot Softmax','Bobot Awal','Gamma'], mix_rows)
    # Ringkasan tambahan hanya dari prediksi VALIDASI yang sudah tersimpan.
    error_rows = []
    all_errors = []
    for file in sorted(source.files):
        match = re.fullmatch(r'09_S3_fold_(\d+)\.xlsx', file)
        if not match or not source.has(file, 'Prediksi_validation'):
            continue
        errors = np.asarray([abs(float(row['prediksi_mentah'])-float(row['skor_guru']))
                             for row in source.records(file, 'Prediksi_validation')])
        if not len(errors):
            continue
        all_errors.extend(errors.tolist())
        error_rows.append([int(match[1]), len(errors), float(errors.mean()),
                           int((errors<=1).sum()), int((errors<=5).sum()), int((errors<=10).sum())])
    if all_errors:
        errors = np.asarray(all_errors)
        error_rows.append(['OOF development', len(errors), float(errors.mean()),
                           int((errors<=1).sum()), int((errors<=5).sum()), int((errors<=10).sum())])
        sheets['MAE dan Toleransi CV'] = Table(
            ['Fold','Jumlah Jawaban','MAE Validasi (poin)','Dalam ±1 poin','Dalam ±5 poin','Dalam ±10 poin'], error_rows)
    return sheets


### `build_reports`

Pembaca/penulis laporan aktual; data individual dan vektor disimpan hanya pada runtime/ZIP pribadi.


In [ ]:
def build_reports(folder,destination,verbose=True):
    """Hanya membaca Excel lama; model/checkpoint/dataset asli tidak diperlukan."""
    source=SavedOutputs(folder);destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)
    if not source.files:raise ValueError('Tidak ada Excel hasil lama di folder sumber.')
    report_types=[('01_Data_dan_Tokenisasi.xlsx',data_report),('02_Vektor_Jawaban.xlsx',answer_vectors_report),
                  ('03_Vektor_Kata_dan_Kalimat.xlsx',word_vectors_report),
                  ('04_Prediksi_dan_Evaluasi.xlsx',evaluation_report),('05_Riwayat_Pelatihan.xlsx',training_report)]
    files=[]
    scope=''
    for info in source.records('01_konfigurasi.xlsx','Info'):
        if info.get('field')=='evidence_scope':scope=str(info.get('value',''))
    for name,builder in report_types:
        sheets=builder(source)
        if sheets:
            if scope:
                sheets['Panduan'].rows=itertools.chain([['Cakupan Sumber',scope]],sheets['Panduan'].rows)
            path=write_report(destination/name,sheets);files.append(path)
            if verbose:print('Siap: '+name,flush=True)
    if not files:raise ValueError('Workbook sumber belum memuat tahap yang dikenali.')
    archive=destination/'Hasil_AES_300_E44_E43_S3_MSE_ListNet_Native_Learnable_GPU.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as bundle:
        for path in files:bundle.write(path,arcname=path.name)
    result={'archive':str(archive.resolve()),'files':[str(p.resolve()) for p in files],
            'source_files':len(source.files),'training_run':False,'metrics_recomputed':False}
    (destination/'ringkasan.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
    return result


<a id="unduh"></a>
## 28 · Buat ZIP dan unduh hasil

ZIP berisi5Excel; checkpoint model tidak otomatis dimasukkan. Simpan ZIP sebelum menghapus runtime. File hasil mengandung jawaban/nilai pribadi, bukan source publik.


In [ ]:
summary=build_reports(journal.folder,output/'hasil_ringkas',verbose=False)
dump(ROOT/'outputs/dataset300_download.json',{
    'archive':summary['archive'],'run_folder':str(output),
    'excel_count':len(summary['files']),'files':[Path(p).name for p in summary['files']]})


# Unduh paket yang baru dibuat, bukan ZIP rincian teknis atau hasil sesi lain.
import zipfile
download=json.loads((ROOT/'outputs/dataset300_download.json').read_text())
with zipfile.ZipFile(download['archive']) as package:
    names=package.namelist()
if sorted(names)!=sorted(download['files']):
    raise RuntimeError('Isi ZIP tidak cocok dengan daftar laporan. Unduhan dihentikan.')
print('Hasil siap diunduh:',Path(download['archive']).name)
print('Isi paket:',len(names),'file Excel yang sudah dirapikan:')
descriptions={
    '01_Data_dan_Tokenisasi.xlsx':'Jawaban, nilai guru, pembagian data, dan token kedua model.',
    '02_Vektor_Jawaban.xlsx':'Embedding jawaban RoBERTa, BGE-M3, dan fusion.',
    '03_Vektor_Kata_dan_Kalimat.xlsx':'Vektor kata/kalimat dan perbandingan kata yang berulang.',
    '04_Prediksi_dan_Evaluasi.xlsx':'Nilai guru, prediksi, RMSE, Pearson, QWK, dan Gwet AC2.',
    '05_Riwayat_Pelatihan.xlsx':'Seluruh fold dan epoch digabung per model.'}
for name in names:
    print('•',name)
    print('  ',descriptions.get(name,''))
if len(names)<5:
    print('Paket belum lengkap: hanya tahap yang sudah selesai yang disertakan.')
if '04_Prediksi_dan_Evaluasi.xlsx' in names:
    print('Mulai membaca hasil akhir dari 04_Prediksi_dan_Evaluasi.xlsx → Perbandingan Model.')
print('Lokasi ZIP pada runtime:',download['archive'])
print('Web Colab: panel Files/Berkas → cari ZIP tersebut → menu tiga titik → Download.')
if IN_COLAB and DOWNLOAD_WITH_COLAB_BROWSER:
    from google.colab import files
    files.download(download['archive'])


## Rujukan dan batas kesimpulan

Rujukan menjelaskan komponen; angka LR/hidden size/epoch pada E44 adalah pilihan ablasi proyek, bukan nilai optimal yang diwajibkan paper.

| Komponen | Sumber dan batas penerapan |
|---|---|
| RoBERTa | [Liu dkk.,2019](https://arxiv.org/abs/1907.11692); checkpoint Indonesia dikunci terpisah. |
| BGE-M3 | [Chen dkk.,2024](https://arxiv.org/abs/2402.03216); model embedding retrieval, digunakan sebagai fitur AES frozen. |
| GRU | [Cho dkk.,2014](https://arxiv.org/abs/1406.1078); implementasi dua arah di sini memakai `nn.GRU`. |
| BiGRU/attention untuk AES | [PaSTO-GNN,Alhothali2026](https://doi.org/10.3389/frai.2026.1842542), [Dong dkk.,2017](https://aclanthology.org/K17-1017/); proyek mengadaptasi token dan komponen berbeda. |
| Co-attention | [Zhang & Litman,2018](https://aclanthology.org/W18-0549/); paper tingkat kalimat, E44 mengadaptasi token respons–referensi. |
| LayerNorm / GELU | [Ba dkk.,2016](https://arxiv.org/abs/1607.06450), [Hendrycks & Gimpel,2016](https://arxiv.org/abs/1606.08415). |
| AdamW | [Loshchilov & Hutter](https://arxiv.org/abs/1711.05101); resep decay0,01 di sini mengikuti kontrol E30. |

**Dropout:** Srivastava dkk.(2014), [Dropout: A Simple Way to Prevent Neural Networks from Overfitting](https://jmlr.org/papers/v15/srivastava14a.html). Paper mendukung metode regularisasi dropout; dropout0.1 dipertahankan dari E30, tidak menjadi perlakuan E44.


Schuster&Paliwal(1997), [Bidirectional Recurrent Neural Networks](https://doi.org/10.1109/78.650093), menjadi dasar konteks dua arah. Hidden64perarah mengikuti resep E30; bukan optimum yang ditetapkan paper.

**Scalar Mix:** Peters dkk.(2018), [Deep Contextualized Word Representations / ELMo](https://aclanthology.org/N18-1202/), menggabungkan representasi beberapa layer dengan bobot yang dipelajari. ELMo memakai arsitektur berbeda serta skala gamma; E44 hanya mengadaptasi campuran softmax empat layer RoBERTa terakhir dengan gamma dan tanpa normalisasi tambahan. [Tenney dkk.(2019), BERT Rediscovers the Classical NLP Pipeline](https://aclanthology.org/P19-1452/), memberi dasar bahwa informasi bahasa tersebar pada kedalaman layer berbeda. Pemilihan empat layer dan penerapan pada AES adalah hipotesis E44; hasil lebih baik tidak dijamin. Bobot global tidak boleh ditafsirkan sebagai penjelasan kausal atau penilaian siswa secara terpisah.

**Gamma:** formulasi Peters dkk.(2018), [ELMo](https://aclanthology.org/N18-1202/), memakai skala gamma pada campuran layer. [AllenNLP ScalarMix](https://github.com/allenai/allennlp/blob/main/allennlp/modules/scalar_mix.py) menginisialisasi gamma1.0 dan [PretrainedTransformerEmbedder](https://github.com/allenai/allennlp/blob/main/allennlp/modules/token_embedders/pretrained_transformer_embedder.py) memakai modul tersebut untuk keluaran Transformer. Ini landasan metode dan implementasi; belum ada paper AES dengan susunan persis penelitian ini yang diverifikasi membuktikan gamma memperbaiki RMSE. LR1e-4 adalah pilihan eksperimen yang dipertahankan, bukan optimum yang ditetapkan paper.

**Pooling acuan E43:** Taghipour&Ng(2016), [A Neural Approach to Automated Essay Scoring](https://aclanthology.org/D16-1193/), dan [repositori penulis](https://github.com/nusnlp/nea) memakai MeanOverTime untuk merangkum urutan recurrent pada AES. [Dong,Zhang&Yang(2017)](https://aclanthology.org/K17-1017/) menjadi rujukan attention untuk AES. E44 mengadaptasi mean pooling pada keluaran matching co-attention,berbeda dari arsitektur paper;paper tidak membuktikan mean akan unggul pada fusion ini.

**Perlakuan E44:** [Yang, Cao, Wen, Wu & He (2020), Enhancing Automated Essay Scoring Performance via Fine-tuning Pre-trained Language Models with Combination of Regression and Ranking](https://aclanthology.org/2020.findings-emnlp.141/). Paper menggabungkan MSE dan batch-wise ListNet berbobot dinamis pada AES/ASAP, dievaluasi QWK. E44 mengadaptasinya dengan bobot tetap 0,1, skala target 0–1, dan kelompok soal sama pada fusion E43; bukan replikasi persis atau jaminan RMSE lebih baik. [PyTorch activation checkpointing](https://docs.pytorch.org/docs/stable/checkpoint.html) dipakai untuk recomputation dengan RNG dropout dipertahankan; tidak menambah model prediksi.
